<a href="https://www.kaggle.com/code/wganme/measurement-dependent?scriptVersionId=369651734" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Measurement-Dependent Clustering — the analysis notebook

This notebook is the only source of the numbers in `main.tex` and
`supplement.tex`. It reads the archive, computes every quantity, and writes
the values and table bodies into the two documents. Nothing in either
document is typed by hand.

Run it top to bottom. The first section reads the raw archive once and caches
the census in `data/census.json`; set `REBUILD_CENSUS = True` to read the
archive again.

In [1]:
# §2–§5, S1–S20 — shared imports and the value registry
import csv, hashlib, io, json, math, os, pathlib, re, sqlite3, subprocess, sys, time, zipfile
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor

import numpy as np
from scipy import stats
from scipy.spatial import ConvexHull, cKDTree

_IN_KAGGLE = pathlib.Path("/kaggle/input").exists() and pathlib.Path("/kaggle/working").exists()
if _IN_KAGGLE:
    ROOT = pathlib.Path("/kaggle/working")
    _INPUT = pathlib.Path("/kaggle/input")
    DATA = None
    for _c in [_INPUT / "measurementdependent", _INPUT / "datasets" / "wganme" / "measurementdependent"]:
        if (_c / "census.json").exists():
            DATA = _c; break
    if DATA is None:
        for _d in sorted(_INPUT.glob("*")):
            if (_d / "census.json").exists():
                DATA = _d; break
    assert DATA is not None, "Attach wganme/measurementdependent. Saw: %s" % sorted(p.name for p in _INPUT.glob("*"))
else:
    ROOT = pathlib.Path(os.environ.get("PAPER_ROOT") or ".").resolve()
    DATA = ROOT / "data"
FIGDIR = ROOT / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)
DB = DATA / "urban_observatory.db"
ARCHIVE = pathlib.Path(os.environ.get("UO_ARCHIVE") or ROOT / "archive")   # read only to rebuild the census
REBUILD_CENSUS = False
WORKERS = min(40, os.cpu_count() or 1)
BASE_SEED = 20261003

LAT0, LON0, R_EARTH = 54.97, -1.61, 6371000.0
LIMIT_M = 60000.0
YEARS = list(range(2013, 2027))

## The value registry

`put(key, value, fmt)` records one printed value. `table(name, body)` records
one table body. The last section writes both into the documents, where a value
is read as `\V{key}`. A key that the documents use and the notebook never set
stops the LaTeX build.

In [2]:
VAL = {}      # key -> {"value": raw, "text": printed string}
TABLES = {}   # marker name -> LaTeX rows
MACROS = {}   # main.tex macro name -> key


def _fmt(value, fmt):
    if fmt is None:
        return str(value)
    if callable(fmt):
        return fmt(value)
    return fmt.format(value)


def put(key, value, fmt=None, macro=None):
    """Record a printed value; `macro` also exposes it as a named macro in main.tex."""
    assert re.fullmatch(r"[A-Za-z0-9.\-]+", key), key
    if isinstance(value, (np.floating, np.integer)):
        value = value.item()
    text = _fmt(value, fmt)
    assert key not in VAL or VAL[key]["text"] == text, "value %s set twice: %s vs %s" % (key, VAL[key]["text"], text)
    VAL[key] = {"value": value, "text": text}
    if macro:
        MACROS[macro] = key
    return value


def get(key):
    return VAL[key]["value"]


def table(name, rows):
    """Record a table body. `rows` is a list of LaTeX row strings without the trailing rule."""
    assert name not in TABLES, name
    TABLES[name] = " \\\\\n\\rowrule\n".join(rows) + " \\\\"


def rng(tag):
    """One reproducible stream per named computation, independent of cell order."""
    h = int(hashlib.sha256(("%d:%s" % (BASE_SEED, tag)).encode()).hexdigest()[:16], 16)
    return np.random.default_rng(h)


def pct(x, d=1):
    return ("{:.%df}" % d).format(100.0 * x)


def signed(d):
    return "{:+.%df}" % d


def interval(lo, hi, d=4):
    return ("[{:+.%df},{:+.%df}]" % (d, d)).format(lo, hi)


def pval(p, d=3):
    """Printed p-value: '<0.001' below the last printable digit."""
    floor = 10.0 ** (-d)
    return "<%s" % ("{:.%df}" % d).format(floor) if p < floor else ("{:.%df}" % d).format(p)

## §2.1 · The census, read from the raw archive

Every archive file is read once. For each identifier and year the census keeps
the first coordinate seen, as the strings the archive prints, every distinct
coordinate, the theme directory, the first and last date and the row count.
Files are merged in sorted path order, so the first file fixes an identifier's
coordinate and theme.

In [3]:
csv.field_size_limit(1 << 30)


def read_file(path):
    """One zip: rows, and per identifier its first coordinate strings, date range and row count."""
    z = pathlib.Path(path)
    rel = z.relative_to(ARCHIVE).parts
    out = {"path": "/".join(rel), "year": int(rel[0]), "theme": rel[1], "bytes": z.stat().st_size,
           "rows": 0, "ids": {}, "months": set(), "bad": False, "nohead": False}
    try:
        with zipfile.ZipFile(z) as zf:
            for member in zf.namelist():
                if not member.endswith(".csv"):
                    continue
                with zf.open(member) as fh:
                    rdr = csv.reader(io.TextIOWrapper(fh, encoding="utf-8", errors="replace", newline=""))
                    try:
                        head = next(rdr)
                    except StopIteration:
                        continue
                    try:
                        i_id = head.index("Raw ID"); i_lat = head.index("Sensor Centroid Latitude")
                        i_lon = head.index("Sensor Centroid Longitude"); i_ts = head.index("Timestamp")
                    except ValueError:
                        out["nohead"] = True
                        continue
                    need = max(i_id, i_lat, i_lon, i_ts)
                    ids = out["ids"]
                    for row in rdr:
                        if len(row) <= need:
                            continue
                        i = row[i_id].strip()
                        if not i:
                            continue
                        out["rows"] += 1
                        ts = row[i_ts].strip()[:10]
                        rec = ids.get(i)
                        if rec is None:
                            ids[i] = [row[i_lon].strip(), row[i_lat].strip(), ts, ts, 1, member,
                                      {(row[i_lon].strip(), row[i_lat].strip())}]
                        else:
                            rec[4] += 1
                            if ts:
                                if not rec[2] or ts < rec[2]: rec[2] = ts
                                if ts > rec[3]: rec[3] = ts
                            c = (row[i_lon].strip(), row[i_lat].strip())
                            if c not in rec[6]:
                                rec[6].add(c)
                        if ts:
                            out["months"].add(ts[:7])
    except zipfile.BadZipFile:
        out["bad"] = True
    out["months"] = sorted(out["months"])
    for rec in out["ids"].values():
        rec[6] = sorted(rec[6])
    return out


def build_census(workers=WORKERS):
    files = sorted(str(p) for p in ARCHIVE.rglob("*.csv.zip"))
    t0 = time.time()
    with ProcessPoolExecutor(workers) as ex:
        res = list(ex.map(read_file, files, chunksize=1))
    census = {"files": [], "years": {}}
    for r in res:  # sorted path order: the first file seen fixes an identifier's coordinate and theme
        y = str(r["year"])
        Y = census["years"].setdefault(y, {"identifiers": {}, "files": 0, "rows": 0, "bytes": 0, "months": set()})
        Y["files"] += 1; Y["rows"] += r["rows"]; Y["bytes"] += r["bytes"]; Y["months"].update(r["months"])
        census["files"].append({k: r[k] for k in ("path", "year", "theme", "bytes", "rows", "bad", "nohead")}
                               | {"n_ids": len(r["ids"])})
        for i, rec in r["ids"].items():
            cur = Y["identifiers"].get(i)
            if cur is None:
                Y["identifiers"][i] = {"lon": rec[0], "lat": rec[1], "first": rec[2], "last": rec[3], "rows": rec[4],
                                       "theme": r["theme"], "variable": rec[5], "coords": [list(c) for c in rec[6]]}
            else:
                cur["rows"] += rec[4]
                if rec[2] and (not cur["first"] or rec[2] < cur["first"]): cur["first"] = rec[2]
                if rec[3] > cur["last"]: cur["last"] = rec[3]
                for c in rec[6]:
                    if list(c) not in cur["coords"]: cur["coords"].append(list(c))
    for Y in census["years"].values():
        Y["months"] = sorted(Y["months"])
    census["total_files"] = len(files)
    census["total_rows"] = sum(Y["rows"] for Y in census["years"].values())
    census["total_bytes"] = sum(Y["bytes"] for Y in census["years"].values())
    census["seconds"] = round(time.time() - t0, 1)
    return census


CENSUS_FILE = DATA / "census.json"
if REBUILD_CENSUS or not CENSUS_FILE.exists():
    CENSUS = build_census()
    _out = ROOT / "data"
    _out.mkdir(parents=True, exist_ok=True)
    (_out / "census.json").write_text(json.dumps(CENSUS))
else:
    CENSUS = json.loads(CENSUS_FILE.read_text())
print("files %d, rows %d, %.1f GB" % (CENSUS["total_files"], CENSUS["total_rows"], CENSUS["total_bytes"] / 1e9))

files 914, rows 3644746934, 74.7 GB


### §2.2 · Spatial units and the coordinate projection

Coordinates are projected about the stated origin. An identifier further than
60 km from the origin on either axis is out of range and dropped. A *location*
is a distinct projected coordinate rounded to 0.1 m.

In [4]:
def proj(lon, lat):
    x = math.radians(lon - LON0) * R_EARTH * math.cos(math.radians(LAT0))
    y = math.radians(lat - LAT0) * R_EARTH
    return x, y


REC = {}        # year -> {identifier: record with x, y added}, in-range only
N_RAW_IDS = set()
N_NOCOORD = set()
N_OUT = set()
for _y in YEARS:
    REC[_y] = {}
    for _i, _r in CENSUS["years"][str(_y)]["identifiers"].items():
        N_RAW_IDS.add(_i)
        try:
            _lon, _lat = float(_r["lon"]), float(_r["lat"])
        except ValueError:
            N_NOCOORD.add(_i)
            continue
        _x, _yy = proj(_lon, _lat)
        if abs(_x) < LIMIT_M and abs(_yy) < LIMIT_M:
            REC[_y][_i] = dict(_r, x=_x, y=_yy)
        else:
            N_OUT.add(_i)


def ids_of(y):
    """Identifiers of a year, in census order."""
    return list(REC[y])


def ids_xy(y, keep=None):
    """Projected coordinates of a year's identifiers; `keep` filters on the record."""
    return np.array([(r["x"], r["y"]) for r in REC[y].values() if keep is None or keep(r)], float).reshape(-1, 2)


def dedup(P, q=1):
    """Distinct coordinates after rounding to 10^-q metres."""
    return np.unique(np.round(np.asarray(P, float), q), axis=0)


def locs_xy(y, keep=None):
    """A year's locations: identifier coordinates deduplicated at 0.1 m."""
    return dedup(ids_xy(y, keep))


def theme_of(y, i):
    return REC[y][i]["theme"]


ALL_XY = np.vstack([ids_xy(y) for y in YEARS])
HULL = ConvexHull(ALL_XY)
HULL_POLY = ALL_XY[HULL.vertices]          # counter-clockwise vertices of the fixed hull
A_HULL = float(HULL.volume)                # its area in square metres
print({y: (len(REC[y]), len(locs_xy(y))) for y in YEARS})
print("hull %.0f km2; raw identifiers %d, out of range %d" % (A_HULL / 1e6, len(N_RAW_IDS), len(N_OUT - set().union(*[set(REC[y]) for y in YEARS]))))

{2013: (45, 43), 2014: (111, 111), 2015: (380, 374), 2016: (694, 597), 2017: (893, 781), 2018: (1095, 974), 2019: (947, 839), 2020: (975, 850), 2021: (946, 831), 2022: (794, 660), 2023: (762, 605), 2024: (875, 631), 2025: (786, 584), 2026: (756, 564)}
hull 4446 km2; raw identifiers 2134, out of range 8


## §2.3 · Estimators shared by every later section

Every estimator the paper uses is defined once, here. The later sections call
these functions and define no estimator of their own.

### §2.3 · Geometry of a convex window

A convex polygon is held as the list of its edges, each an inward unit normal
and an offset. `eroded_inside` marks the points at least `r` inside every
edge, which is the window eroded by `r`. `in_hull` is the same test at
`r = 0`; a positive `tol` admits points up to `tol` metres outside, so hull
vertices count as inside. `sample_hull` draws uniform points by rejection from
the bounding box.

In [5]:
def inward_lines(poly):
    """Edges of a convex polygon as (inward unit normal, offset)."""
    poly = np.asarray(poly, float)
    cen = poly.mean(axis=0)
    out = []
    for k in range(len(poly)):
        a, b = poly[k], poly[(k + 1) % len(poly)]
        e = b - a
        n = np.array([-e[1], e[0]])
        n = n / np.linalg.norm(n)
        if np.dot(cen - a, n) < 0:
            n = -n
        out.append((n, float(np.dot(n, a))))
    return out


HULL_LINES = inward_lines(HULL_POLY)


def _lines(poly):
    return HULL_LINES if poly is HULL_POLY else inward_lines(poly)


def _eroded(P, lines, r):
    ok = np.ones(len(P), bool)
    for n, c in lines:
        ok &= (P @ n) >= (c + r)
    return ok


def eroded_inside(P, r, poly=HULL_POLY):
    """Mask of the points of P at least r inside every edge of the window."""
    return _eroded(np.asarray(P, float), _lines(poly), r)


def in_hull(P, poly, tol=0.0):
    """Mask of the points inside the polygon; tol > 0 makes the test closed."""
    return _eroded(np.asarray(P, float), _lines(poly), -tol)


def n_inside(P, r, poly=HULL_POLY):
    """Number of points that survive erosion of the window by r."""
    return int(eroded_inside(P, r, poly).sum())


def hull_of(P):
    """Convex hull of a point set: vertices in order, and area."""
    P = np.asarray(P, float)
    h = ConvexHull(P)
    return P[h.vertices], float(h.volume)


def hull_perimeter(poly):
    poly = np.asarray(poly, float)
    return float(sum(np.linalg.norm(poly[(k + 1) % len(poly)] - poly[k]) for k in range(len(poly))))


def sample_hull(n, g, poly=HULL_POLY):
    """n uniform points inside the polygon."""
    lines = _lines(poly)
    x0, y0 = poly.min(axis=0)
    x1, y1 = poly.max(axis=0)
    out = []
    while len(out) < n:
        c = np.column_stack([g.uniform(x0, x1, n * 2), g.uniform(y0, y1, n * 2)])
        out.extend(c[_eroded(c, lines, 0.0)])
    return np.array(out[:n])


def bbox_area(P):
    P = np.asarray(P, float)
    return float((P[:, 0].max() - P[:, 0].min()) * (P[:, 1].max() - P[:, 1].min()))

### §2.3 · Nearest-neighbour clustering indices

The Clark-Evans index divides the mean nearest-neighbour distance by its
expectation under complete spatial randomness at density `N / area`. Four
edge treatments are defined beside the plain index. `ce_inside` keeps the
points on or inside the hull edges. `ce_border` keeps a point only when its
distance to the boundary is at least its own nearest-neighbour distance, with
no weighting. `ce_guard` keeps the points at least `guard` metres inside and
takes their neighbours among the kept points. All three divide by the
expectation at the full `N`. `ce_donnelly` adds Donnelly's perimeter term to
the expectation.

In [6]:
def nn_dist(P):
    """Distance from each point to its nearest neighbour."""
    return cKDTree(P).query(P, k=2)[0][:, 1]


def ce_r(P, area):
    """Clark-Evans R for a reference area in square metres."""
    n = len(P)
    if n < 3 or area <= 0:
        return float("nan")
    return float(np.mean(nn_dist(P)) / (1.0 / (2.0 * math.sqrt(n / area))))


def ce_border(P, area, poly):
    n = len(P)
    if n < 5:
        return float("nan")
    lines = _lines(poly)
    d = nn_dist(P)
    keep = np.ones(n, bool)
    for nrm, c in lines:
        keep &= (P @ nrm) >= (c + d)
    if keep.sum() < 5:
        return float("nan")
    return float(np.mean(d[keep]) / (1.0 / (2.0 * math.sqrt(n / area))))


def ce_guard(P, area, poly, guard=500.0):
    keep = _eroded(P, _lines(poly), guard)
    if keep.sum() < 5:
        return float("nan")
    return float(np.mean(nn_dist(P[keep])) / (1.0 / (2.0 * math.sqrt(len(P) / area))))


def ce_inside(P, area, poly):
    return ce_guard(P, area, poly, 0.0)


def ce_donnelly(P, area, perimeter):
    n = len(P)
    expect = 1.0 / (2.0 * math.sqrt(n / area)) + (0.0514 + 0.041 / math.sqrt(n)) * (perimeter / n)
    return float(np.mean(nn_dist(P)) / expect)

### §3.3 · Registration-unit merge rules

A merged site joins identifiers closer than a tolerance. `single_labels`
gives the connected components of the graph that links every pair within
`r`, and `merge_single` keeps the first identifier of each component. The
input is the identifier coordinates, not the deduplicated locations.
`merge_complete` keeps one point per complete-linkage cluster of diameter at
most `r`, `merge_centroid` returns the centroid of each single-linkage
component, and `merge_grid` snaps coordinates to a grid of side `cell`.

In [7]:
from scipy.cluster import hierarchy


def single_labels(P, r):
    """Component label of each point under single linkage at radius r."""
    n = len(P)
    parent = np.arange(n)

    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a

    for i, j in cKDTree(P).query_pairs(r, output_type="ndarray"):
        ri, rj = find(int(i)), find(int(j))
        if ri != rj:
            parent[max(ri, rj)] = min(ri, rj)
    return np.array([find(i) for i in range(n)])


def _first_of(P, lab):
    return P[np.sort(np.unique(lab, return_index=True)[1])]


def merge_single(P, r):
    P = np.asarray(P, float)
    if r <= 0:
        return dedup(P)
    return _first_of(P, single_labels(P, r))


def complete_labels(P, r):
    if len(P) < 2:
        return np.zeros(len(P), int)
    return hierarchy.fcluster(hierarchy.linkage(P, method="complete"), t=r, criterion="distance") - 1


def merge_complete(P, r):
    P = np.asarray(P, float)
    return _first_of(P, complete_labels(P, r))


def merge_centroid(P, r):
    P = np.asarray(P, float)
    lab = single_labels(P, r)
    return np.array([P[lab == c].mean(axis=0) for c in np.unique(lab)])


def merge_grid(P, cell, origin=(0.0, 0.0)):
    P = np.asarray(P, float)
    Q = np.column_stack([np.round((P[:, 0] - origin[0]) / cell) * cell + origin[0],
                         np.round((P[:, 1] - origin[1]) / cell) * cell + origin[1]])
    return np.unique(Q, axis=0)

### §2.3/§3.6 · Second-order functions (K, L, g)

`K_border` is Ripley's K with the border correction on a convex window. The
reference points are those inside the window eroded by the radius; each
counts its neighbours within the radius, and the sum is scaled by
`area / (N * n_reference)`. `L_border` is `sqrt(K / pi) - r`, zero under
complete spatial randomness; it is undefined when fewer than five reference
points survive. `g_annular` is the pair correlation on annuli, the increment
of the same K divided by the annulus area.

In [8]:
RADII = np.array([100.0, 250.0, 500.0, 750.0, 1000.0, 1500.0, 2000.0])
ANNULI = [0.0, 25.0, 100.0, 250.0, 500.0, 750.0, 1000.0, 1500.0, 2000.0]


def K_border(P, radii=RADII, poly=HULL_POLY, area=A_HULL):
    P = np.asarray(P, float)
    radii = np.atleast_1d(np.asarray(radii, float))
    out = np.full(len(radii), np.nan)
    m = len(P)
    if m < 5:
        return out
    lines = _lines(poly)
    tree = cKDTree(P)
    for k, r in enumerate(radii):
        if r <= 0:
            out[k] = 0.0
            continue
        inside = _eroded(P, lines, r)
        n_er = int(inside.sum())
        if n_er < 5:
            continue
        cnt = tree.query_ball_point(P[inside], r, return_length=True)
        out[k] = area * float(np.sum(cnt - 1)) / (m * n_er)
    return out


def L_border(P, radii=RADII, poly=HULL_POLY, area=A_HULL):
    radii = np.atleast_1d(np.asarray(radii, float))
    K = K_border(P, radii, poly, area)
    return np.sqrt(np.maximum(K, 0.0) / math.pi) - radii


def g_annular(P, edges=ANNULI, poly=HULL_POLY, area=A_HULL):
    """Pair correlation on the annuli between consecutive edges."""
    K = K_border(P, edges, poly, area)
    e = np.asarray(edges, float)
    return (K[1:] - K[:-1]) / (math.pi * (e[1:] ** 2 - e[:-1] ** 2))

### §2.3/§3.4 · Concentration on a fixed grid (Gini, Ellison–Glaeser)

The grid is a square frame anchored at the south-west corner of all locations
of the record, with a side equal to the larger extent, cut into `k` by `k`
square cells. `gini_occupied` is the Gini coefficient of the counts in the
occupied cells. `eg_index` is the Ellison-Glaeser index with uniform cell
shares over all `k * k` cells and equally weighted locations. `shift` moves
the origin by a fraction of a cell on each axis.

In [9]:
_ALL_LOCS = np.vstack([locs_xy(y) for y in YEARS])
GRID_ORIGIN = (float(_ALL_LOCS[:, 0].min()), float(_ALL_LOCS[:, 1].min()))
GRID_SPAN = float(max(_ALL_LOCS[:, 0].max() - GRID_ORIGIN[0], _ALL_LOCS[:, 1].max() - GRID_ORIGIN[1]))


def cell_counts(P, k, shift=(0.0, 0.0)):
    """Counts in the occupied cells of the k-by-k grid, and the number of points on the grid."""
    P = np.asarray(P, float)
    cell = GRID_SPAN / k
    ix = np.floor((P[:, 0] - GRID_ORIGIN[0] - shift[0] * cell) / cell).astype(int)
    iy = np.floor((P[:, 1] - GRID_ORIGIN[1] - shift[1] * cell) / cell).astype(int)
    if shift == (0.0, 0.0):  # a point on the frame's far edge belongs to the last cell
        ix[P[:, 0] >= GRID_ORIGIN[0] + GRID_SPAN] = k - 1
        iy[P[:, 1] >= GRID_ORIGIN[1] + GRID_SPAN] = k - 1
    keep = (ix >= 0) & (ix < k) & (iy >= 0) & (iy < k)
    c = np.array(list(Counter(zip(ix[keep].tolist(), iy[keep].tolist())).values()), float)
    return c, int(keep.sum())


def gini_occupied(P, k, shift=(0.0, 0.0)):
    if len(P) < 3:
        return float("nan")
    c, n_in = cell_counts(P, k, shift)
    if n_in < 3:
        return float("nan")
    c = np.sort(c)
    n = len(c)
    return float(2.0 * np.sum(np.arange(1, n + 1) * c) / (n * c.sum()) - (n + 1.0) / n)


def eg_index(P, k, shift=(0.0, 0.0)):
    if len(P) < 3:
        return float("nan")
    c, n_in = cell_counts(P, k, shift)
    if n_in < 3:
        return float("nan")
    M = k * k
    x = 1.0 / M
    raw = float(np.sum((c / n_in - x) ** 2)) + (M - len(c)) * x * x
    H = 1.0 / n_in
    return float((raw - (1.0 - 1.0 / M) * H) / ((1.0 - 1.0 / M) * (1.0 - H)))

### §2.3/S1–S2 · Trend procedures (OLS, HAC, Theil–Sen, TOST, …)

Each procedure takes the years `Y` and a series `v`. `ols` is the ordinary
least-squares slope with its interval and p-value on `t(n - 2)`. `hac` keeps
the same slope and replaces the standard error by the Newey-West one with
Bartlett weights, without a small-sample factor, and an interval on
`t(n - 2)`. `theil_sen` is the median pairwise slope with Sen's rank-based
interval. `mann_kendall` is Kendall's tau of the series against time.
`perm_p` permutes the series. `prais_winsten` is the AR(1) generalised
least-squares fit, with the autocorrelation taken from the lag-one regression
of the residuals. `fixed_b` uses the Bartlett kernel at lag five, `b = 0.4`,
with the Kiefer-Vogelsang critical value 3.727. `block_bootstrap` resamples
circular blocks of three years. `tost` is the larger of the two one-sided
p-values against a symmetric margin, and `mde80` is the effect detectable
with 80 per cent power at the two-sided 5 per cent level.

In [10]:
FIXED_B_LAG, FIXED_B_CRIT = 5, 3.727


def ols(Y, v):
    Y, v = np.asarray(Y, float), np.asarray(v, float)
    r = stats.linregress(Y, v)
    df = len(Y) - 2
    tc = stats.t.ppf(0.975, df)
    return {"slope": float(r.slope), "se": float(r.stderr), "t": float(r.slope / r.stderr),
            "p": float(r.pvalue), "lo": float(r.slope - tc * r.stderr),
            "hi": float(r.slope + tc * r.stderr), "intercept": float(r.intercept), "df": df}


def _bartlett(Y, v, lag):
    Y, v = np.asarray(Y, float), np.asarray(v, float)
    X = np.column_stack([np.ones(len(Y)), Y])
    XtX = np.linalg.pinv(X.T @ X)
    b = XtX @ X.T @ v
    e = v - X @ b
    S = np.zeros((2, 2))
    for t in range(len(Y)):
        S += e[t] ** 2 * np.outer(X[t], X[t])
    for l in range(1, lag + 1):
        w = 1.0 - l / (lag + 1.0)
        for t in range(l, len(Y)):
            G = e[t] * e[t - l] * np.outer(X[t], X[t - l])
            S += w * (G + G.T)
    return float(b[1]), math.sqrt((XtX @ S @ XtX)[1, 1])


def hac(Y, v, lag=2):
    """Slope, Newey-West standard error and interval on t(n - 2)."""
    b, se = _bartlett(Y, v, lag)
    tc = stats.t.ppf(0.975, len(Y) - 2)
    return b, se, b - tc * se, b + tc * se


def theil_sen(Y, v):
    """Median pairwise slope and Sen's order-statistic 95 per cent interval."""
    Y, v = np.asarray(Y, float), np.asarray(v, float)
    n = len(Y)
    s = np.sort([(v[j] - v[i]) / (Y[j] - Y[i]) for i in range(n) for j in range(i + 1, n)])
    N = len(s)
    C = stats.norm.ppf(0.975) * math.sqrt(n * (n - 1) * (2 * n + 5) / 18.0)
    k = max(int(math.floor((N - C) / 2.0)), 0)
    return float(np.median(s)), float(s[k]), float(s[N - k - 1])


def mann_kendall(v, Y=None):
    v = np.asarray(v, float)
    r = stats.kendalltau(np.arange(len(v)) if Y is None else Y, v)
    return float(r.statistic), float(r.pvalue)


def perm_p(Y, v, g, n=20000):
    """Two-sided permutation p-value of the OLS slope."""
    Y, v = np.asarray(Y, float), np.asarray(v, float)
    x = Y - Y.mean()
    den = x @ x
    b = x @ v / den
    perm = np.array([x @ v[g.permutation(len(v))] / den for _ in range(n)])
    return float((np.abs(perm) >= abs(b)).mean())


def prais_winsten(Y, v):
    Y, v = np.asarray(Y, float), np.asarray(v, float)
    r = stats.linregress(Y, v)
    e = v - (r.intercept + r.slope * Y)
    rho = float((e[1:] @ e[:-1]) / (e[:-1] @ e[:-1]))
    s = math.sqrt(1.0 - rho ** 2)
    x = Y - Y.mean()
    vy = np.concatenate([[s * v[0]], v[1:] - rho * v[:-1]])
    vx = np.concatenate([[s * x[0]], x[1:] - rho * x[:-1]])
    c = np.concatenate([[s], np.full(len(Y) - 1, 1.0 - rho)])
    X = np.column_stack([c, vx])
    beta = np.linalg.lstsq(X, vy, rcond=None)[0]
    res = vy - X @ beta
    se = math.sqrt(((res @ res) / (len(Y) - 2) * np.linalg.inv(X.T @ X))[1, 1])
    tc = stats.t.ppf(0.975, len(Y) - 2)
    return {"slope": float(beta[1]), "rho": rho, "se": se,
            "lo": float(beta[1] - tc * se), "hi": float(beta[1] + tc * se)}


def fixed_b(Y, v):
    b, se = _bartlett(Y, v, FIXED_B_LAG)
    return {"slope": b, "se": se, "lo": b - FIXED_B_CRIT * se, "hi": b + FIXED_B_CRIT * se}


def block_bootstrap(Y, v, g, n=5000, L=3):
    """Percentile interval of the slope under a circular block bootstrap."""
    Y, v = np.asarray(Y, float), np.asarray(v, float)
    m = len(Y)
    out = np.empty(n)
    for i in range(n):
        starts = g.integers(0, m, size=m)
        idx = np.concatenate([[(s + j) % m for j in range(L)] for s in starts])[:m]
        out[i] = np.polyfit(Y[idx], v[idx], 1)[0]
    lo, hi = np.percentile(out, [2.5, 97.5])
    return float(lo), float(hi)


def tost(b, se, margin, df):
    pa = float(stats.t.sf((b + margin) / se, df))
    pb = float(stats.t.cdf((b - margin) / se, df))
    return max(pa, pb)


def mde80(se, df):
    return float((stats.t.ppf(0.975, df) + stats.t.ppf(0.80, df)) * se)

### §2.3/§3.3 · Resampling (rarefaction and parallel map)

`pmap` runs a function over a list in forked worker processes and returns
the results in order. `rarefy` draws, for each year, `M` subsamples of
`n_to` points without replacement and returns the Clark-Evans index of each
draw. Each year has its own random stream, named by the tag and the year's
position, so the result does not depend on how the work is scheduled.

In [11]:
import pickle


def pmap(fn, items, workers=WORKERS):
    """Map fn over items in forked processes; results come back in order."""
    items = list(items)
    k = max(1, min(workers, len(items)))
    if k == 1:
        return [fn(x) for x in items]
    chunks = [list(range(j, len(items), k)) for j in range(k)]
    kids = []
    for ch in chunks:
        rfd, wfd = os.pipe()
        pid = os.fork()
        if pid == 0:
            os.close(rfd)
            try:
                payload = pickle.dumps(("ok", [fn(items[i]) for i in ch]))
            except BaseException as exc:
                payload = pickle.dumps(("err", repr(exc)))
            with os.fdopen(wfd, "wb") as fh:
                fh.write(payload)
            os._exit(0)
        os.close(wfd)
        kids.append((pid, rfd, ch))
    out = [None] * len(items)
    for pid, rfd, ch in kids:
        with os.fdopen(rfd, "rb") as fh:
            status, res = pickle.loads(fh.read())
        os.waitpid(pid, 0)
        if status != "ok":
            raise RuntimeError("worker failed: %s" % res)
        for i, r in zip(ch, res):
            out[i] = r
    return out


def rarefy(points_by_year, n_to, areas, tag, M=1000):
    """D[year, draw]: Clark-Evans R of M subsamples of n_to points per year."""
    pts = [np.asarray(P, float) for P in points_by_year]
    areas = list(areas)

    def one(i):
        g = rng("%s:%d" % (tag, i))
        P, A = pts[i], areas[i]
        return [ce_r(P[g.choice(len(P), n_to, replace=False)], A) for _ in range(M)]

    return np.array(pmap(one, range(len(pts))))

### §3.6/S4 · Inhomogeneous nulls (kernel and cell)

Both nulls keep a pattern's first-order intensity and remove its
second-order structure. `kernel_null` lays grid nodes at half the bandwidth
inside the fixed hull, weights each node by the Gaussian kernel sum over its
nearest points, draws nodes without replacement and jitters them by a
Gaussian of a quarter of the bandwidth. `cell_null` bins the points on a
grid of 500 m cells, smooths the counts with a Gaussian, draws cell centres
inside the hull with replacement in proportion to the smoothed count, and
jitters each uniformly within its cell.

In [12]:
from matplotlib.path import Path as _MplPath
from scipy.ndimage import gaussian_filter

_KGRID = {}


def _kernel_grid(h):
    if h not in _KGRID:
        step = h / 2.0
        gx = np.arange(HULL_POLY[:, 0].min(), HULL_POLY[:, 0].max() + step, step)
        gy = np.arange(HULL_POLY[:, 1].min(), HULL_POLY[:, 1].max() + step, step)
        GX, GY = np.meshgrid(gx, gy)
        C = np.column_stack([GX.ravel(), GY.ravel()])
        _KGRID[h] = C[_eroded(C, HULL_LINES, -1e-9)]
    return _KGRID[h]


def kernel_null(P, n, h, g):
    """One draw of n points from the kernel-smoothed intensity of P."""
    P = np.asarray(P, float)
    C = _kernel_grid(h)
    d = cKDTree(P).query(C, k=min(len(P), 100))[0]
    w = np.clip(np.exp(-0.5 * (d / h) ** 2).sum(axis=1), 1e-12, None)
    ix = g.choice(len(C), n, replace=False, p=w / w.sum())
    return C[ix] + g.normal(0.0, h / 4.0, (n, 2))


_CGRID = {}


def _cell_grid(cell):
    if cell not in _CGRID:
        lo = ALL_XY.min(axis=0) - 2000.0
        hi = ALL_XY.max(axis=0) + 2000.0
        nx = int((hi[0] - lo[0]) / cell) + 1
        ny = int((hi[1] - lo[1]) / cell) + 1
        cx, cy = np.meshgrid(lo[0] + (np.arange(nx) + 0.5) * cell, lo[1] + (np.arange(ny) + 0.5) * cell)
        cen = np.column_stack([cx.ravel(), cy.ravel()])
        _CGRID[cell] = (lo, nx, ny, cen, _MplPath(HULL_POLY).contains_points(cen))
    return _CGRID[cell]


def cell_null(P, n, g, cell=500.0, sigma=750.0):
    """One draw of n points from the cell-smoothed intensity of P."""
    P = np.asarray(P, float)
    lo, nx, ny, cen, inside = _cell_grid(cell)
    ix = np.clip(((P[:, 0] - lo[0]) / cell).astype(int), 0, nx - 1)
    iy = np.clip(((P[:, 1] - lo[1]) / cell).astype(int), 0, ny - 1)
    H = np.zeros((ny, nx))
    np.add.at(H, (iy, ix), 1.0)
    dens = gaussian_filter(H, sigma=sigma / cell).ravel()[inside]
    pick = g.choice(int(inside.sum()), size=n, p=dens / dens.sum())
    return cen[inside][pick] + g.uniform(-cell / 2.0, cell / 2.0, size=(n, 2))

### Self-check (data-independent known-answer tests)

Each estimator is run on a pattern whose answer is known without the census:
uniform points in the hull, a square lattice, two points a known distance
apart, and an exact straight line. A failure here stops the notebook.

In [13]:
def _check(name, got, want, tol):
    ok = abs(got - want) <= tol
    print("%-52s %12.5f  expected %-8s %s" % (name, got, want, "ok" if ok else "FAIL"))
    assert ok, name


_g = rng("selfcheck")
_U = sample_hull(4000, _g)
_check("Clark-Evans R, uniform points in the hull", ce_r(_U, A_HULL), 1.0, 0.03)
_check("L(1 km), uniform points in the hull (m)", L_border(_U, [1000.0])[0], 0.0, 150.0)
_s = 100.0
_G = np.array([(i * _s, j * _s) for i in range(60) for j in range(60)], float)
_check("Clark-Evans R, square lattice", ce_r(_G, 3600 * _s * _s), 2.0, 1e-9)
_T = np.array([(0.0, 0.0), (10.0, 0.0), (5000.0, 0.0)])
_check("sites at 25 m, two points 10 m apart and one far", len(merge_single(_T, 25.0)), 2, 0)
_check("sites at 5 m, same three points", len(merge_single(_T, 5.0)), 3, 0)
_Y = np.array(YEARS, float)
_line = 3.0 - 0.5 * (_Y - _Y[0])
_check("OLS slope of an exact line", ols(_Y, _line)["slope"], -0.5, 1e-12)
_check("Theil-Sen slope of an exact line", theil_sen(_Y, _line)[0], -0.5, 1e-12)
_check("Mann-Kendall tau of a falling line", mann_kendall(_line)[0], -1.0, 1e-12)
_check("Gini of equal cell counts", gini_occupied(np.array([(GRID_ORIGIN[0] + (i + 0.5) * GRID_SPAN / 20,
       GRID_ORIGIN[1] + (j + 0.5) * GRID_SPAN / 20) for i in range(20) for j in range(20)]), 20), 0.0, 1e-12)
_P = locs_xy(YEARS[-1])
_n = len(_P)
_c, _ = cell_counts(_P, 20)
_q = float(np.sum(_c * (_c - 1)) / (_c.sum() * (_c.sum() - 1)))
_check("Ellison-Glaeser index against its pair form", eg_index(_P, 20), (_q - 1 / 400) / (1 - 1 / 400), 1e-12)
_d = float(np.mean(nn_dist(_P)))
_check("ln R identity on the last year", math.log(ce_r(_P, A_HULL)),
       math.log(_d) + 0.5 * math.log(_n) - 0.5 * math.log(A_HULL) + math.log(2.0), 1e-12)
_check("pmap order", sum(pmap(lambda i: i * i, range(100))), sum(i * i for i in range(100)), 0)

Clark-Evans R, uniform points in the hull                 1.02421  expected 1.0      ok
L(1 km), uniform points in the hull (m)                   3.58803  expected 0.0      ok
Clark-Evans R, square lattice                             2.00000  expected 2.0      ok
sites at 25 m, two points 10 m apart and one far          2.00000  expected 2        ok
sites at 5 m, same three points                           3.00000  expected 3        ok
OLS slope of an exact line                               -0.50000  expected -0.5     ok
Theil-Sen slope of an exact line                         -0.50000  expected -0.5     ok
Mann-Kendall tau of a falling line                       -1.00000  expected -1.0     ok
Gini of equal cell counts                                 0.00000  expected 0.0      ok
Ellison-Glaeser index against its pair form               0.09643  expected 0.09642718463644592 ok
ln R identity on the last year                           -1.58845  expected -1.5884478630511865 ok
pmap order

/tmp/ipykernel_16/3432440229.py:9: RuntimeWarning: divide by zero encountered in scalar divide
  return {"slope": float(r.slope), "se": float(r.stderr), "t": float(r.slope / r.stderr),
/tmp/ipykernel_16/2227444663.py:14: DeprecationWarning: This process (pid=16) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


## §3.1–§3.6 · The annual series every later section shares

Each series is computed once here and read by name afterwards, so two
sections cannot hold two versions of the same quantity.

In [14]:
MEMO = {}


def memo(tag, fn):
    """Compute a named result once and return the stored copy afterwards."""
    if tag not in MEMO:
        MEMO[tag] = fn()
    return MEMO[tag]


Y = np.array(YEARS, float)
IDS = {y: ids_xy(y) for y in YEARS}                     # identifier coordinates
LOCS = {y: locs_xy(y) for y in YEARS}                   # locations, 0.1 m
SITES10 = {y: merge_single(IDS[y], 10.0) for y in YEARS}  # 10 m single-linkage sites
SITES25 = {y: merge_single(IDS[y], 25.0) for y in YEARS}  # 25 m single-linkage sites
N_IDS = np.array([len(IDS[y]) for y in YEARS])
N_LOCS = np.array([len(LOCS[y]) for y in YEARS])
N_SITES25 = np.array([len(SITES25[y]) for y in YEARS])
UNITS = {"ids": IDS, "loc": LOCS, "site10": SITES10, "site25": SITES25}

OWN_POLY = {y: hull_of(LOCS[y])[0] for y in YEARS}      # each year's own convex hull
A_OWN = np.array([hull_of(LOCS[y])[1] for y in YEARS])  # and its area
A_BOX = np.array([bbox_area(LOCS[y]) for y in YEARS])   # each year's bounding box, locations
A_BOX_IDS = np.array([bbox_area(IDS[y]) for y in YEARS])


def r_series(unit, areas=None):
    """Clark-Evans R of a unit by year; `areas` defaults to the fixed hull."""
    a = [A_HULL] * len(YEARS) if areas is None else areas
    return np.array([ce_r(UNITS[unit][y], a[k]) for k, y in enumerate(YEARS)])


R_LOC = r_series("loc")               # the headline series
R_IDS = r_series("ids")
R_SITE25 = r_series("site25")
R_LOC_OWN = r_series("loc", A_OWN)
R_SITE25_OWN = r_series("site25", A_OWN)
R_LOC_BOX = r_series("loc", A_BOX)
R_IDS_BOX = r_series("ids", A_BOX_IDS)
L_LOC = np.array([L_border(LOCS[y]) for y in YEARS])    # L at RADII, fixed hull, by year


def rarefied(unit, frame="fixed", n_to=None, years=None):
    """Rarefied R draws D[year, draw] for a unit on the fixed hull or each year's own hull."""
    ys = YEARS if years is None else years
    pts = [UNITS[unit][y] for y in ys]
    n = min(len(p) for p in pts) if n_to is None else n_to
    areas = [A_HULL] * len(ys) if frame == "fixed" else [A_OWN[YEARS.index(y)] for y in ys]
    tag = "rarefy:%s:%s:%d:%d-%d" % (unit, frame, n, ys[0], ys[-1])
    return memo(tag, lambda: rarefy(pts, n, areas, tag))


def fall(v):
    """Endpoint fall of a series, first year to last, as a fraction."""
    return float((v[0] - v[-1]) / v[0])


print(np.round(R_LOC, 3), np.round(L_LOC[:, 4]).astype(int))

[0.417 0.334 0.279 0.264 0.237 0.418 0.435 0.384 0.41  0.279 0.246 0.198
 0.197 0.204] [12307  6929  7307  7589  8506  6947  6786  7980  6570  7652  5861  5211
  4163  4005]


## §2.3/§3.3/S1–S2 · Trend inference on the clustering index

This section fits the fixed-hull Clark-Evans series under every interval
procedure the paper names, simulates the size of the two parametric tests at
fourteen points, and tests the rarefied series for equivalence to zero. It
fills Tables `si:trend`, `si:size`, `si:procs` and `si:seed`.

In [15]:
INF_NSIM = 60000                      # replicates per row of the size simulation
INF_CHUNKS = 8                        # each row is simulated in this many parallel parts
INF_RHOS = (0.0, 0.15, 0.3, 0.5)      # AR(1) coefficients simulated
INF_T = len(YEARS)
INF_DF = INF_T - 2
INF_LAG = 2                           # Newey-West lag
INF_NPERM = 20000
INF_NBOOT = 5000
INF_BLOCK = 3
INF_STREAMS = ("canonical", "second", "third")

f3, f4, f5 = "{:.3f}", "{:.4f}", "{:.5f}"
s3, s4, s5 = "{:+.3f}", "{:+.4f}", "{:+.5f}"


def _iv(lo, hi, d=4, sep=","):
    return ("$[{:+.%df}%s{:+.%df}]$" % (d, sep, d)).format(lo, hi)


def _tp(p):
    """An equivalence p-value: four decimals below 0.01, three above."""
    return ("{:.4f}" if p < 0.01 else "{:.3f}").format(p)

### §3.1/S1 · The fixed-hull trend under each procedure

Four series are fitted: locations, identifiers, locations without 2026, and
locations without 2017, 2018 and 2026. Each gets the OLS slope and p-value, a
permutation p-value, the Mann-Kendall p-value and the Newey-West interval.

In [16]:
_keep26 = np.array([y != 2026 for y in YEARS])
_keep3 = np.array([y not in (2017, 2018, 2026) for y in YEARS])
INF_TREND = [
    ("loc", "Hull $R$, locations", Y, R_LOC),
    ("ids", "Hull $R$, identifiers", Y, R_IDS),
    ("no26", "Hull $R$, locations, no 2026", Y[_keep26], R_LOC[_keep26]),
    ("no3", "Hull $R$, locations, no 2017--18, 2026", Y[_keep3], R_LOC[_keep3]),
]
INF_FIT = {}
_rows = []
for _k, _label, _yy, _v in INF_TREND:
    _o = ols(_yy, _v)
    _b, _se, _lo, _hi = hac(_yy, _v, INF_LAG)
    _pp = perm_p(_yy, _v, rng("inf:perm:" + _k), INF_NPERM)
    _mk = mann_kendall(_v)[1]
    INF_FIT[_k] = dict(_o, nw_se=_se, nw_lo=_lo, nw_hi=_hi, perm=_pp, mk=_mk, n=len(_yy))
    put("inf.%s.slope" % _k, _o["slope"], s5)
    put("inf.%s.se" % _k, _o["se"], f4)
    put("inf.%s.p" % _k, _o["p"], f3)
    put("inf.%s.perm" % _k, _pp, f3)
    put("inf.%s.mk" % _k, _mk, f3)
    put("inf.%s.n" % _k, len(_yy), "{:d}")
    _rows.append("%s & $%s$ & %s & %s & %s & %s & %d" % (
        _label, s5.format(_o["slope"]), f3.format(_o["p"]), f3.format(_pp), f3.format(_mk),
        _iv(_lo, _hi, 5, ", "), len(_yy)))
table("si-trend", _rows)

put("inf.macro.locslope", INF_FIT["loc"]["slope"], f5, macro="rHullLocSlope")
put("inf.macro.locp", INF_FIT["loc"]["p"], f3, macro="hullLocOlsP")
put("inf.macro.idsp", INF_FIT["ids"]["p"], f3)
put("inf.macro.idsslope", INF_FIT["ids"]["slope"], f5)
put("inf.macro.idsse", INF_FIT["ids"]["se"], f4)

0.005319711927585988

### §2.3/S2 · Size of the tests at fourteen points

Series with no trend are simulated at the length of the record. The AR(1)
rows start from the stationary distribution; one of them uses the lag-1
residual autocorrelation of the headline location series, and the last row is
independent Gaussian noise. A test is counted as rejecting when its two-sided statistic
passes the 97.5th percentile of `t(12)`.

In [17]:
def _size_row(job):
    name, rho, part = job
    g = rng("inf:size:%s:%d" % (name, part))
    tc = stats.t.ppf(0.975, INF_DF)
    x = Y - Y.mean()
    sxx = float(x @ x)
    nw = ol = 0
    for _ in range(INF_NSIM // INF_CHUNKS):
        if rho is None:
            e = g.normal(size=INF_T)
        else:
            e = np.empty(INF_T)
            e[0] = g.normal() / math.sqrt(1.0 - rho ** 2)
            for t in range(1, INF_T):
                e[t] = rho * e[t - 1] + g.normal()
        b, se = _bartlett(Y, e, INF_LAG)
        nw += abs(b / se) > tc
        res = e - e.mean() - b * x
        ol += abs(b / math.sqrt(res @ res / INF_DF / sxx)) > tc
    return name, nw, ol


INF_RHOHAT = round(float(prais_winsten(Y, R_LOC)["rho"]), 2)   # autocorrelation of the headline series
assert 0.0 < INF_RHOHAT < 0.95, INF_RHOHAT
put("inf.size.rhohat", INF_RHOHAT, "{:.2f}")
_jobs = [("rho%g" % r, r) for r in INF_RHOS] + [("rhohat", INF_RHOHAT), ("iid", None)]
assert INF_NSIM % INF_CHUNKS == 0
INF_SIZE = {n: [0, 0] for n, _ in _jobs}
for _n, _a, _b in pmap(_size_row, [(n, r, c) for n, r in _jobs for c in range(INF_CHUNKS)]):
    INF_SIZE[_n][0] += _a / INF_NSIM
    INF_SIZE[_n][1] += _b / INF_NSIM
put("inf.size.nsim", INF_NSIM, "{:d}")
put("inf.baseseed", BASE_SEED, "{:d}")
_lab = {"rho0": r"AR(1) $\rho = 0$", "rho0.15": r"AR(1) $\rho = 0.15$", "rho0.3": r"AR(1) $\rho = 0.3$",
        "rho0.5": r"AR(1) $\rho = 0.5$", "iid": r"i.i.d.\ Gaussian",
        "rhohat": r"AR(1) $\rho = \hat\rho = %.2f$" % INF_RHOHAT}
_rows = []
for _n, _r in _jobs:
    _nw, _ol = INF_SIZE[_n]
    _key = _n.replace(".", "")
    put("inf.size.%s.nw" % _key, _nw, f3)
    put("inf.size.%s.ols" % _key, _ol, f3)
    put("inf.size.%s.nw.pct" % _key, 100 * _nw, "{:.1f}")
    put("inf.size.%s.ols.pct" % _key, 100 * _ol, "{:.1f}")
    _rows.append("%s & %s & %s" % (_lab[_n], f3.format(_nw), f3.format(_ol)))
table("si-size", _rows)
put("inf.macro.nwsize", 100 * INF_SIZE["rho0"][0], "{:.1f}", macro="nwSize")
put("inf.macro.nwsizear", 100 * INF_SIZE["rhohat"][0], "{:.1f}")
put("inf.macro.olsiid", 100 * INF_SIZE["iid"][1], "{:.1f}", macro="olsSizeIid")
put("inf.macro.olsar", 100 * INF_SIZE["rhohat"][1], "{:.1f}", macro="olsSizeAr")

# the prose says both tests over-reject once errors are autocorrelated
assert INF_SIZE["rho0"][0] > 0.10 and INF_SIZE["rho0.5"][1] > 0.10
assert INF_SIZE["rhohat"][0] > INF_SIZE["rho0"][0] and INF_SIZE["rhohat"][1] > INF_SIZE["iid"][1]
assert abs(INF_SIZE["iid"][1] - 0.05) < 0.01

/tmp/ipykernel_16/2227444663.py:14: DeprecationWarning: This process (pid=16) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


### §2.3/S2 · A Newey-West test with its critical value simulated at this length

The table above shows the Newey-West test over-rejecting at fourteen points
when it is referred to `t(12)`. Here the same statistic is referred to its
own null distribution instead: series with no trend and AR(1) errors are
simulated on each series' own set of years, and the p-value is the share of
simulated statistics at least as large as the observed one. A test built this
way has its nominal size under the simulated error model by construction.

In [18]:
INF_NCAL = 40000                      # null series per design and autocorrelation
INF_CAL_RHOS = (INF_RHOHAT,)          # the lag-1 autocorrelation of the headline series


def _cal_part(job):
    key, rho, part = job
    mask = INF_MASK[key]
    yy = Y[mask]
    g = rng("inf:cal:%s:%g:%d" % (key, rho, part))
    out = np.empty(INF_NCAL // INF_CHUNKS)
    for i in range(len(out)):
        e = np.empty(INF_T)
        e[0] = g.normal() / math.sqrt(1.0 - rho ** 2)
        for t in range(1, INF_T):
            e[t] = rho * e[t - 1] + g.normal()
        b, se = _bartlett(yy, e[mask], INF_LAG)
        out[i] = abs(b / se)
    return key, rho, out


INF_MASK = {"loc": np.ones(INF_T, bool), "ids": np.ones(INF_T, bool), "no26": _keep26, "no3": _keep3}
assert INF_NCAL % INF_CHUNKS == 0
_null = defaultdict(list)
for _k, _r, _o in pmap(_cal_part, [(k, r, c) for k in INF_MASK for r in INF_CAL_RHOS for c in range(INF_CHUNKS)]):
    _null[(_k, _r)].append(_o)
INF_CAL = {}
for (_k, _r), _parts in _null.items():
    _sim = np.concatenate(_parts)
    _obs = abs(INF_FIT[_k]["slope"] / INF_FIT[_k]["nw_se"])
    INF_CAL[(_k, _r)] = {"p": float((1 + (_sim >= _obs).sum()) / (1 + len(_sim))),
                         "crit": float(np.percentile(_sim, 95.0)), "t": float(_obs)}
put("inf.cal.nsim", INF_NCAL, "{:d}")
put("inf.cal.tcrit", float(stats.t.ppf(0.975, INF_DF)), "{:.2f}")
for _k in INF_MASK:
    _c = INF_CAL[(_k, INF_RHOHAT)]
    put("inf.cal.%s.p" % _k, _c["p"], pval)
    put("inf.cal.%s.crit" % _k, _c["crit"], "{:.2f}")
    put("inf.cal.%s.t" % _k, INF_CAL[(_k, INF_RHOHAT)]["t"], "{:.2f}")
_c = INF_CAL[("loc", INF_RHOHAT)]
put("inf.cal.loc.lo", INF_FIT["loc"]["slope"] - _c["crit"] * INF_FIT["loc"]["nw_se"], s5)
put("inf.cal.loc.hi", INF_FIT["loc"]["slope"] + _c["crit"] * INF_FIT["loc"]["nw_se"], s5)
# The prose: the simulated critical value is above that of t(12), and under it no fixed-hull
# slope is distinguishable from zero.
assert all(c["crit"] > stats.t.ppf(0.975, INF_DF) for c in INF_CAL.values())
assert all(c["p"] > 0.05 for c in INF_CAL.values())
print({k: (round(c["t"], 2), round(c["crit"], 2), round(c["p"], 4)) for k, c in INF_CAL.items()})

/tmp/ipykernel_16/2227444663.py:14: DeprecationWarning: This process (pid=16) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


{('loc', 0.57): (2.42, 5.33, 0.2892), ('ids', 0.57): (3.15, 5.27, 0.1846), ('no26', 0.57): (1.79, 5.53, 0.4343), ('no3', 0.57): (1.99, 6.07, 0.4247)}


### §3.3/S2 · Every procedure on the headline and rarefied series

The rarefied series are the annual means of 1000 draws to the smallest
year's count, locations and identifiers. Each of the four series gets the
same eight procedures. The rarefied columns also get the percentile range of
the slopes fitted to one draw per year, and the equivalence tests.

In [19]:
INF_D = {"loc": rarefied("loc"), "ids": rarefied("ids")}
INF_NTO = {"loc": int(N_LOCS.min()), "ids": int(N_IDS.min())}
INF_MEAN = {u: INF_D[u].mean(axis=1) for u in INF_D}
INF_COLS = [("loc", R_LOC), ("ids", R_IDS), ("rloc", INF_MEAN["loc"]), ("rids", INF_MEAN["ids"])]

INF_P = {}
for _k, _v in INF_COLS:
    _o = ols(Y, _v)
    _h = hac(Y, _v, INF_LAG)
    _ts = theil_sen(Y, _v)
    _pw = prais_winsten(Y, _v)
    _fb = fixed_b(Y, _v)
    _bb = block_bootstrap(Y, _v, rng("inf:boot:" + _k), INF_NBOOT, INF_BLOCK)
    INF_P[_k] = {"ols": _o, "nw": (_h[2], _h[3]), "nw_se": _h[1], "ts": _ts, "pw": _pw,
                 "fb": (_fb["lo"], _fb["hi"]), "bb": _bb}
    put("inf.procs.%s.slope" % _k, _o["slope"], s5)
    put("inf.procs.%s.pw.slope" % _k, _pw["slope"], s5)
    put("inf.procs.%s.pw.iv" % _k, (_pw["lo"], _pw["hi"]), lambda t: interval(t[0], t[1], 4))
    put("inf.procs.%s.pw.rho" % _k, _pw["rho"], s3)


def _contains(pairs):
    return ", ".join("yes" if lo <= 0.0 <= hi else "no" for lo, hi in pairs)


def _row(label, cells, last):
    return "%s & %s & %s" % (label, " & ".join(cells), last)


_ks = [k for k, _ in INF_COLS]
_P = INF_P
_rows = [
    _row("OLS slope", ["$%s$" % s5.format(_P[k]["ols"]["slope"]) for k in _ks], "--"),
    _row("OLS interval ($t_{12}$)", [_iv(_P[k]["ols"]["lo"], _P[k]["ols"]["hi"]) for k in _ks],
         _contains([(_P[k]["ols"]["lo"], _P[k]["ols"]["hi"]) for k in _ks])),
    _row("Newey--West (HAC, lag %d)" % INF_LAG, [_iv(*_P[k]["nw"]) for k in _ks],
         _contains([_P[k]["nw"] for k in _ks])),
    _row("Theil--Sen", ["$%s$" % s5.format(_P[k]["ts"][0]) for k in _ks], "--"),
    _row("Theil--Sen interval", [_iv(_P[k]["ts"][1], _P[k]["ts"][2]) for k in _ks],
         _contains([(_P[k]["ts"][1], _P[k]["ts"][2]) for k in _ks])),
    _row("Moving-block bootstrap ($L=%d$)" % INF_BLOCK, [_iv(*_P[k]["bb"]) for k in _ks],
         _contains([_P[k]["bb"] for k in _ks])),
    _row("Prais--Winsten slope", ["$%s$" % s5.format(_P[k]["pw"]["slope"]) for k in _ks], "--"),
    _row("Prais--Winsten interval", [_iv(_P[k]["pw"]["lo"], _P[k]["pw"]["hi"]) for k in _ks],
         _contains([(_P[k]["pw"]["lo"], _P[k]["pw"]["hi"]) for k in _ks])),
    _row("Fixed-$b$ (Bartlett, $b=0.4$, critical %.3f)" % FIXED_B_CRIT, [_iv(*_P[k]["fb"]) for k in _ks],
         _contains([_P[k]["fb"] for k in _ks])),
    _row(r"Prais--Winsten $\hat\rho$", ["$%s$" % s3.format(_P[k]["pw"]["rho"]) for k in _ks], "--"),
    _row("Permutation $p$", [f3.format(INF_FIT["loc"]["perm"]), f3.format(INF_FIT["ids"]["perm"]), "--", "--"], "--"),
    _row("Mann--Kendall $p$", [f3.format(INF_FIT["loc"]["mk"]), f3.format(INF_FIT["ids"]["mk"]), "--", "--"], "--"),
]

/tmp/ipykernel_16/2227444663.py:14: DeprecationWarning: This process (pid=16) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


### §3.3/S2 · Equivalence of the rarefied slope to zero (TOST)

The margin is the size of the unrarefied slope on the same unit and frame.
The primary test uses the OLS standard error of the annual-mean slope. The
sensitivity test adds the standard deviation of the slopes fitted to single
draws, so it speaks about one rarefied draw and not about the expected
trajectory. Both are one-sided on `t(12)`.

In [20]:
INF_E = {}
_x = Y - Y.mean()
for _u, _unrar in (("loc", "loc"), ("ids", "ids")):
    _D = INF_D[_u]
    _o = INF_P["r" + _u]["ols"]
    _rep = _x @ (_D - _D.mean(axis=0)) / (_x @ _x)           # one slope per draw
    _srep = float(np.std(_rep, ddof=1))
    _s = math.sqrt(_o["se"] ** 2 + _srep ** 2)
    _m = abs(INF_FIT[_unrar]["slope"])
    _lo, _hi = (float(q) for q in np.percentile(_rep, [2.5, 97.5]))
    INF_E[_u] = {"slope": _o["slope"], "smean": _o["se"], "srep": _srep, "s": _s, "margin": _m,
                 "rep": (_lo, _hi), "mde": mde80(_o["se"], INF_DF),
                 "p_full": tost(_o["slope"], _o["se"], _m, INF_DF),
                 "p_half": tost(_o["slope"], _o["se"], _m / 2, INF_DF),
                 "q_full": tost(_o["slope"], _s, _m, INF_DF),
                 "q_half": tost(_o["slope"], _s, _m / 2, INF_DF)}
    _e = INF_E[_u]
    put("inf.tost.%s.smean" % _u, _e["smean"], f5)
    put("inf.tost.%s.srep" % _u, _e["srep"], f5)
    put("inf.tost.%s.s" % _u, _e["s"], f5)
    put("inf.tost.%s.margin" % _u, _m, f5)
    put("inf.tost.%s.half" % _u, _m / 2, f5)
    put("inf.tost.%s.pfull" % _u, _e["p_full"], _tp)
    put("inf.tost.%s.phalf" % _u, _e["p_half"], _tp)
    put("inf.tost.%s.qfull" % _u, _e["q_full"], _tp)
    put("inf.tost.%s.qhalf" % _u, _e["q_half"], _tp)
    put("inf.tost.%s.mde" % _u, _e["mde"], f5)
    put("inf.rare.%s.n" % _u, INF_NTO[_u], "{:d}")

def _names(xs):
    return xs[0] if len(xs) == 1 else ", ".join(xs[:-1]) + " and " + xs[-1]


_PROC = [("OLS", lambda q: (q["ols"]["lo"], q["ols"]["hi"])), ("Newey--West", lambda q: q["nw"]),
         ("Theil--Sen", lambda q: (q["ts"][1], q["ts"][2])), ("moving-block bootstrap", lambda q: q["bb"]),
         ("Prais--Winsten", lambda q: (q["pw"]["lo"], q["pw"]["hi"])), ("fixed-$b$", lambda q: q["fb"])]
for _k in ("loc", "ids"):       # which intervals hold zero on the two unrarefied series
    _in = [n for n, f in _PROC if f(_P[_k])[0] <= 0.0 <= f(_P[_k])[1]]
    _out = [n for n, f in _PROC if n not in _in]
    assert _in and _out, (_k, _in, _out)
    put("inf.procs.%s.contain" % _k, _names(_in))
    put("inf.procs.%s.exclude" % _k, _names(_out))

_el, _ei = INF_E["loc"], INF_E["ids"]
_rows += [
    _row(r"Complete-replicate 95\%", ["--", "--", _iv(*_el["rep"]), _iv(*_ei["rep"])],
         "--, --, " + _contains([_el["rep"], _ei["rep"]])),
    _row(r"TOST ($s_{\mathrm{mean}}$, $t_{12}$), full margin",
         ["--", "--", "$p=%s$" % _tp(_el["p_full"]), "$p=%s$" % _tp(_ei["p_full"])], "--"),
    _row(r"TOST ($s_{\mathrm{mean}}$, $t_{12}$), half margin",
         ["--", "--", "$p=%s$" % _tp(_el["p_half"]), "$p=%s$" % _tp(_ei["p_half"])], "--"),
    _row(r"TOST (single draw, $t_{12}$), full margin",
         ["--", "--", "$p=%s$" % _tp(_el["q_full"]), "$p=%s$" % _tp(_ei["q_full"])], "--"),
    _row(r"TOST (single draw, $t_{12}$), half margin",
         ["--", "--", "$p=%s$" % _tp(_el["q_half"]), "$p=%s$" % _tp(_ei["q_half"])], "--"),
]
table("si-procedures", _rows)

put("inf.macro.rareslope", _el["slope"], s5, macro="rRareLocSlope")
put("inf.macro.rarelo", _el["rep"][0], s5, macro="rRareLocLO")
put("inf.macro.rarehi", _el["rep"][1], s5, macro="rRareLocHI")
put("inf.macro.raren", INF_NTO["loc"], "{:d}", macro="rRareLocN")
put("inf.macro.rarefirst", INF_MEAN["loc"][0], f3)
put("inf.macro.rarelast", INF_MEAN["loc"][-1], f3)
put("inf.macro.raremde", _el["mde"], f5, macro="rRareMDE")

0.008789466657282566

### S2 · Which standard error a single rarefied draw carries

The sensitivity test combines the scatter of the annual means with the spread
of the replicate slopes. An analyst holding one draw would instead read the
ordinary OLS standard error of that draw, which weights every year's
subsampling variance equally. Both are recorded, with the drift the margin
allows over the record and the smallest sampling fraction.

In [21]:
INF_SD = {}
_xc = Y - Y.mean()
for _u, _series in (("loc", R_LOC), ("ids", R_IDS)):
    _D = INF_D[_u]
    _bm = _xc @ (_D - _D.mean(axis=0)) / (_xc @ _xc)
    _res = _D - _D.mean(axis=0) - np.outer(_xc, _bm)
    _sd = float(np.median(np.sqrt((_res ** 2).sum(axis=0) / INF_DF / (_xc @ _xc))))
    _e = INF_E[_u]
    INF_SD[_u] = {"se": _sd, "full": tost(_e["slope"], _sd, _e["margin"], INF_DF),
                  "half": tost(_e["slope"], _sd, _e["margin"] / 2, INF_DF)}
    put("inf.tost.%s.sdraw" % _u, _sd, f5)
    put("inf.tost.%s.dfull" % _u, INF_SD[_u]["full"], _tp)
    put("inf.tost.%s.dhalf" % _u, INF_SD[_u]["half"], _tp)
    put("inf.tost.%s.drift" % _u, _e["margin"] * (YEARS[-1] - YEARS[0]), f3)
    put("inf.tost.%s.driftpct" % _u, 100.0 * _e["margin"] * (YEARS[-1] - YEARS[0]) / _series[0], "{:.0f}")
_frac = INF_NTO["loc"] / N_LOCS
put("inf.frac.min", float(_frac.min()), f3)
put("inf.frac.minyear", YEARS[int(_frac.argmin())], "{:d}")
put("inf.frac.maxn", int(N_LOCS.max()), "{:d}")
# The prose says the single-draw OLS error is the larger of the two, and that under it
# equivalence of a single location draw is not established.
assert all(INF_SD[u]["se"] > INF_E[u]["s"] for u in INF_SD), "the single-draw OLS error is not the larger"
assert INF_SD["loc"]["full"] >= 0.05 > INF_E["loc"]["q_full"], "the two single-draw tests now agree for locations"
print({u: {k: round(v, 5) for k, v in d.items()} for u, d in INF_SD.items()})

{'loc': {'se': 0.00602, 'full': 0.06229, 'half': 0.23699}, 'ids': {'se': 0.00593, 'full': 0.04791, 'half': 0.28347}}


The sentences that read these tables are checked here, so a change in the
data that made one of them false stops the notebook.

In [22]:
INF_CLAIMS = {
    "every interval in the rarefied columns contains zero": all(
        lo <= 0 <= hi for k in ("rloc", "rids") for lo, hi in (
            (_P[k]["ols"]["lo"], _P[k]["ols"]["hi"]), _P[k]["nw"], (_P[k]["ts"][1], _P[k]["ts"][2]),
            _P[k]["bb"], (_P[k]["pw"]["lo"], _P[k]["pw"]["hi"]), _P[k]["fb"])),
    "fixed-b contains zero on all four series": all(_P[k]["fb"][0] <= 0 <= _P[k]["fb"][1] for k in _ks),
    "block bootstrap contains zero on all four series": all(_P[k]["bb"][0] <= 0 <= _P[k]["bb"][1] for k in _ks),
    "replicate ranges contain zero": all(INF_E[u]["rep"][0] <= 0 <= INF_E[u]["rep"][1] for u in INF_E),
    "expected trajectory equivalent at the full margin on both units": _el["p_full"] < 0.05 and _ei["p_full"] < 0.05,
    "expected trajectory not equivalent at the half margin on either unit": _el["p_half"] >= 0.05 and _ei["p_half"] >= 0.05,
    "single draw equivalent at the full margin on both units": _el["q_full"] < 0.05 and _ei["q_full"] < 0.05,
    "single draw not equivalent at the half margin on either unit": _el["q_half"] >= 0.05 and _ei["q_half"] >= 0.05,
    "rarefied slope smaller in size than the unrarefied one on both units": abs(_el["slope"]) < _el["margin"] and abs(_ei["slope"]) < _ei["margin"],
    "MDE below the unrarefied slope": _el["mde"] < _el["margin"],
    "neither test attains its size at the headline autocorrelation": min(INF_SIZE["rhohat"]) > 0.075,
}
for _c, _ok in INF_CLAIMS.items():
    print("%-66s %s" % (_c, "holds" if _ok else "FAILS"))

every interval in the rarefied columns contains zero               holds
fixed-b contains zero on all four series                           holds
block bootstrap contains zero on all four series                   holds
replicate ranges contain zero                                      holds
expected trajectory equivalent at the full margin on both units    holds
expected trajectory not equivalent at the half margin on either unit holds
single draw equivalent at the full margin on both units            holds
single draw not equivalent at the half margin on either unit       holds
rarefied slope smaller in size than the unrarefied one on both units holds
MDE below the unrarefied slope                                     holds
neither test attains its size at the headline autocorrelation      holds


### S20 · Sensitivity to the random stream

The rarefaction is repeated on two further random streams for each unit. The
standard errors here are Newey-West.

In [23]:
INF_SEED = {}
_rows = []
for _j, _name in enumerate(INF_STREAMS):
    _cells = []
    for _u, _unit in (("loc", LOCS), ("ids", IDS)):
        if _j == 0:
            _Dj = INF_D[_u]
        else:
            _tag = "inf:stream:%s:%s" % (_u, _name)
            _Dj = memo(_tag, lambda: rarefy([_unit[y] for y in YEARS], INF_NTO[_u], [A_HULL] * INF_T, _tag))
        _b, _se, _lo, _hi = hac(Y, _Dj.mean(axis=1), INF_LAG)
        INF_SEED[(_u, _name)] = (_b, _se, _lo, _hi)
        put("inf.seed.%s.%d" % (_u, _j + 1), _b, s5)
        _cells.append((_b, _se, _lo <= 0 <= _hi))
    (_bl, _sl, _zl), (_bi, _si, _zi) = _cells
    _rows.append(r"%s & $%s$ ($\mathrm{se}\,%s$) & %s & $%s$ ($\mathrm{se}\,%s$) & %s" % (
        _name.capitalize(), s5.format(_bl), f5.format(_sl), "yes" if _zl else "no",
        s5.format(_bi), f5.format(_si), "yes" if _zi else "no"))
table("si-seeds", _rows)
INF_CLAIMS["every stream's interval contains zero"] = all(lo <= 0 <= hi for _, _, lo, hi in INF_SEED.values())
print("streams:", {k: round(v[0], 5) for k, v in INF_SEED.items()})
assert all(INF_CLAIMS.values()), [c for c, ok in INF_CLAIMS.items() if not ok]

/tmp/ipykernel_16/2227444663.py:14: DeprecationWarning: This process (pid=16) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


streams: {('loc', 'canonical'): -0.00104, ('ids', 'canonical'): -0.00373, ('loc', 'second'): -0.00126, ('ids', 'second'): -0.00382, ('loc', 'third'): -0.00109, ('ids', 'third'): -0.00388}


## §3.3–§3.4/S3/S5/S15 · Rarefaction, reference frames and concentration

This section produces the supplement's sections on rarefaction and the anchor
year, on reference frames and the grid, and on the grid origin, together with
the concentration paragraphs of the main text.

Three things are measured. The clustering index is rarefied to a common
sample size, with the anchor year dropped and at larger sizes. The clustering
index is recomputed under other reference windows. The two concentration
indices, the Gini coefficient of occupied cells and the Ellison-Glaeser
index, are computed on four grids, rarefied, and recomputed with the grid
origin shifted.

In [24]:
CON_M = 1000                      # rarefaction draws per year
CON_RES = (20, 30, 40, 50)        # grid cells per side
CON_SIZES = (150, 200)            # larger common sample sizes
CON_LATTICE = np.linspace(0.0, 1.0, 5)   # origin shifts, in cell fractions
CON_LATTICE_OFFSET = -0.02        # the lattice starts just outside the frame
CON_EQ_TRIALS = 2000              # random count vectors for the Gini formula check


def con_iv(lo, hi, d=4, sep=", "):
    """An interval as printed, with explicit signs."""
    return ("[{:+.%df}%s{:+.%df}]" % (d, sep, d)).format(lo, hi)


def con_years_text(ys):
    """A list of years in words: runs become ranges."""
    ys = sorted(ys)
    runs, a = [], ys[0]
    for p, q in zip(ys, ys[1:] + [None]):
        if q != p + 1:
            runs.append("%d" % a if a == p else "%d--%d" % (a, p))
            a = q
    return runs[0] if len(runs) == 1 else ", ".join(runs[:-1]) + " and " + runs[-1]


def con_rep_slopes(Yv, D):
    """OLS slope of every draw; D is draws[year, draw]."""
    x = np.asarray(Yv, float) - np.mean(Yv)
    return (x @ (D - D.mean(axis=0))) / (x @ x)


def con_rep95(Yv, D):
    s = con_rep_slopes(Yv, D)
    return float(np.percentile(s, 2.5)), float(np.percentile(s, 97.5))

### §3.3/S3 · The rarefied clustering index, year by year

Locations are rarefied to the smallest year's location count and identifiers
to the smallest year's identifier count. The table gives the mean and the
standard deviation over the draws beside the unrarefied value. The smallest
year is its own sample, so its standard deviation is zero and is printed as a
dash.

In [25]:
con_D = {"loc": rarefied("loc"), "ids": rarefied("ids")}
con_nmin = {"loc": int(N_LOCS.min()), "ids": int(N_IDS.min())}
put("con.nmin.loc", con_nmin["loc"], "{:d}")
put("con.nmin.ids", con_nmin["ids"], "{:d}")
put("con.draws", CON_M, "{:d}")

con_rows = []
for k, y in enumerate(YEARS):
    cells = ["%d" % y]
    for u, raw in (("loc", R_LOC), ("ids", R_IDS)):
        sd = con_D[u][k].std(ddof=1)
        cells += ["%.3f" % con_D[u][k].mean(), "--" if sd < 1e-12 else "%.3f" % sd, "%.3f" % raw[k]]
    con_rows.append(" & ".join(cells))
table("si-rarefied", con_rows)

con_rloc = con_D["loc"].mean(axis=1)
con_rids = con_D["ids"].mean(axis=1)
_b, _se, _lo, _hi = hac(Y, con_rloc)
put("con.rare.loc.slope", ols(Y, con_rloc)["slope"], "{:+.5f}")
put("con.rare.loc.nw", con_iv(_lo, _hi, 4, ","))
put("con.rare.ids.slope", ols(Y, con_rids)["slope"], "{:+.5f}")
assert _lo < 0 < _hi and hac(Y, con_rids)[2] < 0 < hac(Y, con_rids)[3], "a rarefied series has a detectable trend"
# The prose names the first year as the smallest one, the anchor of the rarefaction.
assert int(np.argmin(N_LOCS)) == 0 and int(np.argmin(N_IDS)) == 0 and int(np.argmin(N_SITES25)) == 0, "the smallest year is not the first"

### S3 · The anchor year and larger sample sizes

The smallest year fixes the common sample size and, being its own sample, has
no sampling variance. The fit is repeated without it, and at larger sample
sizes on the years that reach them, on locations and on the 25 m merged
sites. Each row gives the slope of the annual means with its interval and
p-value, the Theil-Sen interval, and the central 95 per cent of the slopes
fitted to single draws.

In [26]:
def con_rs_row(unit, n_to, years, D, label):
    Yv = np.array(years, float)
    m = D.mean(axis=1)
    o = ols(Yv, m)
    ts = theil_sen(Yv, m)
    rep = con_rep95(Yv, D)
    row = " & ".join([label, "%d" % n_to, "%d" % len(years), "$%+.5f$" % o["slope"],
                      "$%s$" % con_iv(o["lo"], o["hi"], 4, ","), "%.3f" % o["p"],
                      "$%s$" % con_iv(ts[1], ts[2], 4, ","), "$%s$" % con_iv(rep[0], rep[1], 4, ",")])
    spans = [(o["lo"], o["hi"]), (ts[1], ts[2]), rep]
    return row, o["slope"], all(lo < 0 < hi for lo, hi in spans)


con_rs = {}
con_rs_body = []
for unit, name in (("loc", "Locations"), ("site25", r"Merged \SI{25}{\meter}")):
    counts = np.array([len(UNITS[unit][y]) for y in YEARS])
    nmin = int(counts.min())
    second = int(np.sort(counts)[1])
    D0 = rarefied(unit)
    rows = [con_rs_row(unit, nmin, YEARS, D0, name),
            con_rs_row(unit, nmin, YEARS[1:], D0[1:], name + ", no %d" % YEARS[0])]
    omitted = {}
    for n_to in (second,) + CON_SIZES:
        keep = [y for k, y in enumerate(YEARS) if counts[k] >= n_to]
        omitted[n_to] = [y for y in YEARS if y not in keep]
        rows.append(con_rs_row(unit, n_to, keep, rarefied(unit, "fixed", n_to, keep), name))
    con_rs[unit] = {"second": second, "omitted": omitted, "slopes": [r[1] for r in rows],
                    "all_span_zero": all(r[2] for r in rows)}
    con_rs_body.append(" \\\\\n\\rowrule\n".join(r[0] for r in rows))
table("si-rarersens", [" \\\\\n\\midrule\n".join(con_rs_body)])

put("con.rs.second.loc", con_rs["loc"]["second"], "{:d}")
put("con.rs.second.site", con_rs["site25"]["second"], "{:d}")
put("con.rs.omit.loc", con_years_text(con_rs["loc"]["omitted"][CON_SIZES[-1]]))
put("con.rs.omit.site", con_years_text(con_rs["site25"]["omitted"][CON_SIZES[-1]]))
# The prose says every interval holds zero and the slope changes sign from row to row.
assert con_rs["loc"]["all_span_zero"] and con_rs["site25"]["all_span_zero"], "a rarefied-R interval excludes zero"
_s = con_rs["loc"]["slopes"] + con_rs["site25"]["slopes"]
assert min(_s) < 0 < max(_s), "the rarefied slopes no longer change sign between rows"
assert (con_rs["loc"]["omitted"][CON_SIZES[0]] == con_rs["loc"]["omitted"][CON_SIZES[1]]
        == con_rs["site25"]["omitted"][CON_SIZES[0]] == con_rs["site25"]["omitted"][CON_SIZES[1]]), "the two larger sizes omit different years"
put("con.rs.omit.both", con_years_text(con_rs["loc"]["omitted"][CON_SIZES[1]]))


/tmp/ipykernel_16/2227444663.py:14: DeprecationWarning: This process (pid=16) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


'2013--2014'

### S3 · The scale the rarefied index can resolve

At the common sample size the expected nearest-neighbour distance under
complete spatial randomness in the fixed hull is several kilometres, which
bounds what the rarefied index can see.

In [27]:
con_enn = {u: 0.5 * math.sqrt(A_HULL / con_nmin[u]) for u in ("ids", "loc")}
put("con.scale.enn.ids", con_enn["ids"], "{:.0f}")
put("con.scale.enn.loc", con_enn["loc"], "{:.0f}")
put("con.scale.enn.diff", round(con_enn["loc"]) - round(con_enn["ids"]), "{:.0f}")
put("con.hull.km2", A_HULL / 1e6, "{:.0f}")
con_sizes_text = ", ".join("%d" % n for n in (con_nmin["loc"],) + CON_SIZES)
table("si-scale", [
    "Rarefaction sample size (identifiers) & %d" % con_nmin["ids"],
    "Hull area (km$^2$) & %.0f" % (A_HULL / 1e6),
    "Expected nearest-neighbour distance (m) & %.0f / %.0f" % (con_enn["ids"], con_enn["loc"]),
    "Location sample size, smallest year & %d" % con_nmin["loc"],
    "Anchor sizes tested for the concentration index & %s" % con_sizes_text,
])

### §3.4/S3 · The Gini coefficient of occupied cells, rarefied

The Gini coefficient is computed on the fixed grid at four resolutions and
rarefied to the smallest year's location count. A year that holds exactly the
common sample size is not drawn. The twenty-cell grid is also fitted without
the anchor year, without the anchor and the last year, and at larger common
sample sizes.

In [28]:
def con_grid_draws(fn, name, k, n_to, years):
    """draws[year, draw] of a grid index at a common sample size."""
    def one(j):
        P = LOCS[years[j]]
        if len(P) <= n_to:
            return [fn(P, k)] * CON_M
        g = rng("con:%s:%d:%d:%d" % (name, k, n_to, years[j]))
        return [fn(P[g.choice(len(P), n_to, replace=False)], k) for _ in range(CON_M)]
    tag = "con:%s:%d:%d:%d" % (name, k, n_to, len(years))
    return memo(tag, lambda: np.array(pmap(one, range(len(years)))))


def con_summary(Yv, D, tag):
    m = D.mean(axis=1)
    o = ols(Yv, m)
    ts = theil_sen(Yv, m)
    return {"mean": m, "slope": o["slope"], "se": o["se"], "p": o["p"], "ts": ts,
            "perm": perm_p(Yv, m, rng("con:perm:" + tag)), "mk": mann_kendall(m, Yv)[1],
            "rep": con_rep95(Yv, D)}


con_n = con_nmin["loc"]
con_gini_raw = {k: np.array([gini_occupied(LOCS[y], k) for y in YEARS]) for k in CON_RES}
con_gini_D = {k: con_grid_draws(gini_occupied, "gini", k, con_n, YEARS) for k in CON_RES}
con_gini = {k: con_summary(Y, con_gini_D[k], "gini%d" % k) for k in CON_RES}
con_gini_rawslope = {k: ols(Y, con_gini_raw[k])["slope"] for k in CON_RES}

put("con.grid.x0", GRID_ORIGIN[0], "{:.0f}")
put("con.grid.y0", GRID_ORIGIN[1], "{:.0f}")
put("con.grid.span.km", GRID_SPAN / 1e3, "{:.1f}")
put("con.gini.raw20", con_gini_rawslope[20], "{:+.4f}")
put("con.gini.raw20.abs", con_gini_rawslope[20], "{:.4f}")
put("con.gini.raw20.macro", con_gini_rawslope[20], "{:.3f}")
put("con.gini.rare20", con_gini[20]["slope"], "{:+.4f}", macro="giniRareSlope")
put("con.gini.rep20", con_iv(*con_gini[20]["rep"]))
put("con.gini.mde", mde80(con_gini[20]["se"], len(YEARS) - 2), "{:.4f}")

table("si-ginigrid", [
    " & ".join(["%d" % k, "$%+.4f$" % con_gini[k]["slope"],
                "$%s$" % con_iv(con_gini[k]["ts"][1], con_gini[k]["ts"][2]),
                "$%+.4f$" % con_gini_rawslope[k], "%.0f" % (GRID_SPAN / k)])
    for k in CON_RES])

# The prose rests on a sign reversal at every resolution and on these intervals.
assert all(con_gini_rawslope[k] > 0 > con_gini[k]["slope"] for k in CON_RES), "no sign reversal at some resolution"
# The uncorrected series rises to a peak and falls back, so its rank interval holds zero; the rarefied one does not.
_ts_raw = theil_sen(Y, con_gini_raw[20])
assert _ts_raw[1] < 0 < _ts_raw[2] and con_gini[20]["ts"][2] < 0, "the rank intervals of the Gini series changed sides"
con_gini_peak = int(np.argmax(con_gini_raw[20]))
assert 0 < con_gini_peak < len(YEARS) - 1, "the uncorrected Gini no longer peaks inside the record"
put("con.gini.raw.peakyear", YEARS[con_gini_peak], "{:d}")
# The figure caption says the uncorrected series follows the population.
assert np.corrcoef(con_gini_raw[20], N_LOCS)[0, 1] > 0.8, "the uncorrected Gini no longer follows the location count"
assert con_gini[20]["rep"][0] < 0 < con_gini[20]["rep"][1], "the replicate interval of the rarefied Gini excludes zero"
assert abs(con_gini[20]["slope"]) < mde80(con_gini[20]["se"], len(YEARS) - 2) < con_gini_rawslope[20]

/tmp/ipykernel_16/2227444663.py:14: DeprecationWarning: This process (pid=16) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


In [29]:
con_drop = {"all": (Y, con_gini_D[20]), "no_first": (Y[1:], con_gini_D[20][1:]),
            "no_ends": (Y[1:-1], con_gini_D[20][1:-1])}
con_drop = {name: con_summary(Yv, D, "drop:" + name) for name, (Yv, D) in con_drop.items()}
con_drop_label = {"all": "Rarefied Gini, all fourteen years",
                  "no_first": "Rarefied Gini, %d anchor dropped" % YEARS[0],
                  "no_ends": "Rarefied Gini, %d and %d dropped" % (YEARS[0], YEARS[-1])}
table("si-anchor", [
    " & ".join([con_drop_label[n], "$%+.4f$" % s["slope"], "%.3f" % s["p"], "%.3f" % s["perm"],
                "%.3f" % s["mk"], "$%s$" % con_iv(s["rep"][0], s["rep"][1], 4, ",")])
    for n, s in con_drop.items()])
put("con.gini.nofirst.slope", con_drop["no_first"]["slope"], "{:+.4f}")
put("con.gini.nofirst.p", con_drop["no_first"]["p"], "{:.3f}")
assert abs(con_drop["no_first"]["slope"]) < abs(con_drop["all"]["slope"]), "dropping the anchor does not flatten the slope"

con_anchor_rows = []
con_anchor_omit = {}
for n_to in (con_n,) + CON_SIZES:
    keep = [y for k, y in enumerate(YEARS) if N_LOCS[k] >= n_to]
    con_anchor_omit[n_to] = [y for y in YEARS if y not in keep]
    Yk = np.array(keep, float)
    D = con_gini_D[20] if n_to == con_n else con_grid_draws(gini_occupied, "gini", 20, n_to, keep)
    s = con_drop["all"] if n_to == con_n else con_summary(Yk, D, "anchor%d" % n_to)
    if n_to != con_n:  # the main text says the larger sizes exclude zero
        assert s["rep"][1] < 0, "the replicate interval at %d locations reaches zero" % n_to
    con_anchor_rows.append(" & ".join([
        r"\num{%d} locations" % n_to, "%d" % len(keep), "$%+.4f$" % s["slope"], "$%+.4f$" % s["ts"][0],
        "$%s$" % con_iv(s["ts"][1], s["ts"][2]), "$%+.3f$" % ((YEARS[-1] - YEARS[0]) * s["slope"]),
        "$%s$" % con_iv(s["rep"][0], s["rep"][1], 4, ",")]))
table("si-anchors", con_anchor_rows)
put("con.anchor.omit.mid", con_years_text(con_anchor_omit[CON_SIZES[0]]))
put("con.anchor.omit.big", con_years_text(con_anchor_omit[CON_SIZES[1]]))
put("con.horizon", YEARS[-1] - YEARS[0], "{:d}")
assert con_anchor_omit[CON_SIZES[0]] == con_anchor_omit[CON_SIZES[1]], "the two larger anchors drop different years"


/tmp/ipykernel_16/2227444663.py:14: DeprecationWarning: This process (pid=16) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


### S3 · The gap-filled Gini

An identifier absent between two years in which it reports is treated as
present throughout. The uncorrected Gini slope on that roster is recorded for
the main text.

In [30]:
def con_gapfilled_gini(k=20):
    span, where = {}, {}
    for y in YEARS:
        for i, r in REC[y].items():
            span.setdefault(i, [y, y])[1] = y
            where[i] = (round(r["x"], 1), round(r["y"], 1))
    out = []
    for y in YEARS:
        P = np.unique(np.array([where[i] for i, (a, b) in span.items() if a <= y <= b], float), axis=0)
        out.append(gini_occupied(P, k))
    return np.array(out)


put("con.gini.gapfilled", ols(Y, con_gapfilled_gini())["slope"], "{:+.3f}")

0.012061887372059304

### §2.3/S5 · Reference windows and the per-year bounding box

The clustering index is recomputed with the area of the bounding box of the
whole record, with each year's own bounding box, and inside each year's
5th-95th percentile core rectangle with that rectangle's area.

In [31]:
con_allbox = bbox_area(np.vstack([LOCS[y] for y in YEARS]))


def con_core(P):
    lo, hi = np.percentile(P, 5, axis=0), np.percentile(P, 95, axis=0)
    keep = (P[:, 0] >= lo[0]) & (P[:, 0] <= hi[0]) & (P[:, 1] >= lo[1]) & (P[:, 1] <= hi[1])
    return P[keep], float((hi[0] - lo[0]) * (hi[1] - lo[1]))


con_core_area = np.array([con_core(LOCS[y])[1] for y in YEARS])
con_R_core = np.array([ce_r(*con_core(LOCS[y])) for y in YEARS])
con_R_allbox = r_series("loc", [con_allbox] * len(YEARS))


def con_frame_row(label, area_text, v):
    o = ols(Y, v)
    return " & ".join([label, area_text, "%.3f" % v[0], "%.3f" % v[-1], "$%+.4f$" % o["slope"],
                       ("$%s$" % pval(o["p"])) if o["p"] < 0.001 else "%.3f" % o["p"]])


table("si-frametab", [
    con_frame_row("Fixed hull (headline)", "%.0f" % (A_HULL / 1e6), R_LOC),
    con_frame_row("Full-record bounding box", "%.0f" % (con_allbox / 1e6), con_R_allbox),
    con_frame_row("Core, 5th--95th pct", "%.0f--%.0f" % (con_core_area.min() / 1e6, con_core_area.max() / 1e6),
                  con_R_core),
])
table("si-peryear", ["%d & %d & %.0f & %.3f" % (y, N_LOCS[k], A_BOX[k] / 1e6, R_LOC_BOX[k])
                      for k, y in enumerate(YEARS)])

con_kmax = int(np.argmax(A_BOX))
put("con.box.first", A_BOX[0] / 1e6, "{:.0f}")
put("con.box.max", A_BOX[con_kmax] / 1e6, "{:.0f}")
put("con.box.max.year", YEARS[con_kmax], "{:d}")
put("con.box.last", A_BOX[-1] / 1e6, "{:.0f}")
put("con.box.factor", A_BOX[con_kmax] / A_BOX[0], "{:.1f}")
put("con.box.fall", 100 * fall(R_LOC_BOX), "{:.1f}")
put("con.hull.fall", 100 * fall(R_LOC), "{:.1f}")
put("con.own.first.km2", A_OWN[0] / 1e6, "{:.0f}")
put("con.own.last.km2", A_OWN[-1] / 1e6, "{:.0f}")
assert A_BOX[con_kmax] > A_BOX[-1] > A_BOX[0], "the per-year box no longer rises and falls back"
assert (A_OWN[1:] > A_OWN[0]).all(), "a later footprint is smaller than the first year's"

### §3.4/S5 · The Ellison–Glaeser dartboard index

With uniform cell shares and equally weighted locations the index is the
share of location pairs that fall in one cell, rescaled. That share is an
unbiased pair statistic, so rarefying leaves its expectation unchanged: the
rarefied annual means differ from the uncorrected values by simulation noise
only, and the spread of single-draw slopes is the one new quantity.

In [32]:
con_eg_raw = {k: np.array([eg_index(LOCS[y], k) for y in YEARS]) for k in CON_RES}
con_eg_D = {k: con_grid_draws(eg_index, "eg", k, con_n, YEARS) for k in CON_RES}
con_eg_rows = []
con_eg = {}
for k in CON_RES:
    hr = hac(Y, con_eg_raw[k])
    m = con_eg_D[k].mean(axis=1)
    hm = hac(Y, m)
    rep = con_rep95(Y, con_eg_D[k])
    con_eg[k] = {"raw": hr, "rare": hm, "rep": rep, "mean": m}
    con_eg_rows.append(" & ".join([
        "%d" % k, "$%+.4f$" % hr[0], "$%s$" % con_iv(hr[2], hr[3]), "%.0f" % (GRID_SPAN / k),
        "$%+.4f$" % hm[0], "$%s$" % con_iv(hm[2], hm[3]), "$%s$" % con_iv(rep[0], rep[1])]))
    put("con.eg.slope%d" % k, hr[0], "{:+.4f}")
table("si-egtable", con_eg_rows)
table("si-egannual", ["%d & $%+.3f$ & $%+.3f$" % (y, con_eg_raw[20][j], con_eg[20]["mean"][j])
                       for j, y in enumerate(YEARS)])
put("con.eg.rep20", con_iv(*con_eg[20]["rep"]))


def con_pair_form(P, k):
    """The pair-collision form of the index, for the identity check."""
    c, n = cell_counts(P, k)
    q = float(np.sum(c * (c - 1.0))) / (n * (n - 1.0))
    return (q - 1.0 / (k * k)) / (1.0 - 1.0 / (k * k))


con_eg_identity = max(abs(con_pair_form(LOCS[y], k) - con_eg_raw[k][j]) for k in CON_RES for j, y in enumerate(YEARS))
assert con_eg_identity < 1e-12, "the pair form and the printed form of the index differ"
assert all(con_eg[k]["raw"][0] < 0 for k in CON_RES), "the Ellison-Glaeser slope is not negative everywhere"
# The main text says the replicate interval reaches zero at 20 cells and not at the finer grids.
con_eg_rep_zero = {k: con_eg[k]["rep"][0] < 0 < con_eg[k]["rep"][1] for k in CON_RES}
assert con_eg_rep_zero == {k: k == CON_RES[0] for k in CON_RES}, "the replicate intervals no longer split as the text says: %s" % con_eg_rep_zero

/tmp/ipykernel_16/2227444663.py:14: DeprecationWarning: This process (pid=16) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


### S15 · Grid-origin sensitivity for the concentration index

Both indices are recomputed with the grid origin shifted over a five-by-five
lattice of sub-cell positions, at 20 and at 40 cells per side. The Gini
formula as printed in the main text is also checked against its textbook
form on random count vectors.

In [33]:
con_shifts = [(CON_LATTICE_OFFSET + fx, CON_LATTICE_OFFSET + fy) for fx in CON_LATTICE for fy in CON_LATTICE]
con_origin = {}
for k in (20, 40):
    gs = np.array([ols(Y, [gini_occupied(LOCS[y], k, sh) for y in YEARS])["slope"] for sh in con_shifts])
    es = [np.array([eg_index(LOCS[y], k, sh) for y in YEARS]) for sh in con_shifts]
    eh = np.array([hac(Y, v) for v in es])
    con_origin[k] = {"gini": gs, "eg": eh[:, 0], "eg_up": eh[:, 3],
                     "eg_ols_up": np.array([ols(Y, v)["hi"] for v in es])}
con_no = len(con_shifts)
put("con.origin.n", con_no, "{:d}")
table("si-origintab", [
    " & ".join(["%d" % k, "$%+.4f$ to $%+.4f$" % (o["gini"].min(), o["gini"].max()), "$%+.4f$" % o["gini"].mean(),
                "%.4f" % o["gini"].std(ddof=1), "yes" if (o["gini"] > 0).all() else "no"])
    for k, o in con_origin.items()])
table("si-egorigin", [
    " & ".join(["%d" % k, "$%+.4f$ to $%+.4f$" % (o["eg"].min(), o["eg"].max()),
                "%d of %d" % ((o["eg"] < 0).sum(), con_no),
                "$%+.4f$ to $%+.4f$" % (o["eg_up"].min(), o["eg_up"].max()),
                "%d of %d" % ((o["eg_up"] < 0).sum(), con_no),
                "%d of %d" % ((o["eg_ols_up"] < 0).sum(), con_no)])
    for k, o in con_origin.items()])
for k in (20, 40):
    g = con_origin[k]["gini"]
    put("con.origin.gini.spread%d" % k, 100 * (g.max() - g.min()) / g.min(), "{:.0f}")
for k in (20, 40):
    put("con.origin.eg.sig%d" % k, int((con_origin[k]["eg_up"] < 0).sum()), "{:d}")
    put("con.origin.eg.olssig%d" % k, int((con_origin[k]["eg_ols_up"] < 0).sum()), "{:d}")
assert all((con_origin[k]["gini"] > 0).all() and (con_origin[k]["eg"] < 0).all() for k in (20, 40)), "an origin changes a sign"
# The prose says the significance does not hold at every origin, at either resolution.
assert all((con_origin[k]["eg_up"] >= 0).any() for k in (20, 40)), "every interval excludes zero at some resolution"


def con_gini_printed(c):
    """Eq. (1) of the main text, as printed."""
    c = np.sort(np.asarray(c, float))
    n = len(c)
    return (n + 1) / n - (2.0 / (n * c.sum())) * np.sum((n - np.arange(1, n + 1) + 1) * c)


def con_gini_textbook(c):
    c = np.sort(np.asarray(c, float))
    n = len(c)
    return 2.0 * np.sum(np.arange(1, n + 1) * c) / (n * c.sum()) - (n + 1.0) / n


_g = rng("con:gini-equation")
con_eq_worst = 0.0
for _ in range(CON_EQ_TRIALS):
    _c = _g.integers(1, 20, size=int(_g.integers(3, 60)))
    con_eq_worst = max(con_eq_worst, abs(con_gini_printed(_c) - con_gini_textbook(_c)))
assert con_eq_worst < 1e-12
_m, _e = ("%.1e" % con_eq_worst).split("e")
put("con.gini.eqdiff", r"%s\times10^{%d}" % (_m, int(_e)))
put("con.gini.eqtrials", CON_EQ_TRIALS, "{:d}")

2000

### §3.4/S5 · The dartboard index on each year's own support, and under every trend procedure

The frame of the dartboard index is fixed while the footprint grows, so the
index is recomputed with the uniform benchmark spread only over the grid
cells a year's own convex hull meets, on the ground the 2013 network
occupied, and on the 25 m merged sites. The headline series is also fitted
with every interval procedure the clustering index is given.

In [34]:
def con_cells_meeting(poly, k):
    """mask[k, k] of the grid cells a convex polygon meets (separating-axis test on the cell corners)."""
    cell = GRID_SPAN / k
    X0, Y0 = np.meshgrid(GRID_ORIGIN[0] + cell * np.arange(k), GRID_ORIGIN[1] + cell * np.arange(k), indexing="ij")
    corners = np.stack([np.stack([X0 + dx, Y0 + dy], axis=-1) for dx in (0.0, cell) for dy in (0.0, cell)])
    ok = ((X0 + cell >= poly[:, 0].min()) & (X0 <= poly[:, 0].max())
          & (Y0 + cell >= poly[:, 1].min()) & (Y0 <= poly[:, 1].max()))
    for n, c in inward_lines(poly):
        ok &= (corners @ n).max(axis=0) >= c
    return ok


def con_eg_support(P, k, mask):
    """Dartboard index with the uniform benchmark spread over the cells of `mask` only."""
    P = np.asarray(P, float)
    cell = GRID_SPAN / k
    ix = np.clip(np.floor((P[:, 0] - GRID_ORIGIN[0]) / cell).astype(int), 0, k - 1)
    iy = np.clip(np.floor((P[:, 1] - GRID_ORIGIN[1]) / cell).astype(int), 0, k - 1)
    keep = mask[ix, iy]
    n, M = int(keep.sum()), int(mask.sum())
    c = np.bincount(ix[keep] * k + iy[keep], minlength=k * k).astype(float)
    q = float(np.sum(c * (c - 1.0))) / (n * (n - 1.0))
    return (q - 1.0 / M) / (1.0 - 1.0 / M)


assert max(abs(con_eg_support(LOCS[y], k, np.ones((k, k), bool)) - con_eg_raw[k][j])
           for k in CON_RES for j, y in enumerate(YEARS)) < 1e-12, "the support form differs from the index on the full frame"

con_fp = OWN_POLY[YEARS[0]]                       # the ground the first year's network occupied
CON_EGS = {
    "frame.loc": ("Full frame, exact locations", lambda y, k: eg_index(LOCS[y], k)),
    "frame.site": (r"Full frame, merged \SI{25}{\meter}", lambda y, k: eg_index(SITES25[y], k)),
    "own.loc": ("Own-hull cells, exact locations",
                lambda y, k: con_eg_support(LOCS[y], k, con_cells_meeting(OWN_POLY[y], k))),
    "own.site": (r"Own-hull cells, merged \SI{25}{\meter}",
                 lambda y, k: con_eg_support(SITES25[y], k, con_cells_meeting(OWN_POLY[y], k))),
    "fp.loc": ("%d footprint, exact locations" % YEARS[0],
               lambda y, k: con_eg_support(LOCS[y][in_hull(LOCS[y], con_fp, 1e-6)], k, con_cells_meeting(con_fp, k))),
    "fp.site": (r"%d footprint, merged \SI{25}{\meter}" % YEARS[0],
                lambda y, k: con_eg_support(SITES25[y][in_hull(SITES25[y], con_fp, 1e-6)], k,
                                            con_cells_meeting(con_fp, k))),
}
con_egs = {v: {k: ols(Y, np.array([fn(y, k) for y in YEARS])) for k in CON_RES} for v, (_, fn) in CON_EGS.items()}
table("si-egsupport", [
    " & ".join([label] + ["$%+.4f$ ($%s$)" % (con_egs[v][k]["slope"], pval(con_egs[v][k]["p"])) for k in CON_RES])
    for v, (label, _) in CON_EGS.items()])
for v in CON_EGS:
    for k in CON_RES:
        put("con.egs.%s.slope%d" % (v, k), con_egs[v][k]["slope"], "{:+.4f}")
        put("con.egs.%s.p%d" % (v, k), con_egs[v][k]["p"], pval)
_ratio = [con_egs["own.loc"][k]["slope"] / con_egs["frame.loc"][k]["slope"] for k in CON_RES]
put("con.egs.own.ratio.lo", 100.0 * min(_ratio), "{:.0f}")
put("con.egs.own.ratio.hi", 100.0 * max(_ratio), "{:.0f}")
put("con.egs.fp.n.first", int(in_hull(LOCS[YEARS[0]], con_fp, 1e-6).sum()), "{:d}")
put("con.egs.fp.n.last", int(in_hull(LOCS[YEARS[-1]], con_fp, 1e-6).sum()), "{:d}")
put("con.egs.own.cells.first", int(con_cells_meeting(OWN_POLY[YEARS[0]], CON_RES[0]).sum()), "{:d}")
put("con.egs.own.cells.max", max(int(con_cells_meeting(OWN_POLY[y], CON_RES[0]).sum()) for y in YEARS), "{:d}")
put("con.egs.own.cells.last", int(con_cells_meeting(OWN_POLY[YEARS[-1]], CON_RES[0]).sum()), "{:d}")
put("con.egs.frame.cells", CON_RES[0] ** 2, "{:d}")
# The prose: every variant declines; from 30 cells per side the OLS interval excludes zero under every
# variant; at 20 cells it does on the headline series and not on the own-hull benchmark or the merged sites.
assert all(con_egs[v][k]["slope"] < 0 for v in CON_EGS for k in CON_RES), "a variant no longer declines"
assert all(con_egs[v][k]["p"] < 0.05 for v in CON_EGS for k in CON_RES[1:]), "a variant loses significance above 20 cells"
assert con_egs["frame.loc"][CON_RES[0]]["p"] < 0.05 <= min(con_egs["own.loc"][CON_RES[0]]["p"], con_egs["frame.site"][CON_RES[0]]["p"])
assert min(_ratio) > 0.8, "the own-hull benchmark removes more than a fifth of a slope"
put("con.egs.n20.sig", sum(1 for v in CON_EGS if con_egs[v][CON_RES[0]]["p"] < 0.05),
    lambda n: ("none", "one", "two", "three", "four", "five", "six")[n])
put("con.egs.n.rows", len(CON_EGS), lambda n: ("none", "one", "two", "three", "four", "five", "six")[n])
# The prose: merging makes the full-frame slope shallower at every resolution, and on the first
# year's ground the slope is no shallower than on the full frame.
assert all(abs(con_egs["frame.site"][k]["slope"]) < abs(con_egs["frame.loc"][k]["slope"]) for k in CON_RES)
assert all(con_egs["fp.loc"][k]["slope"] / con_egs["frame.loc"][k]["slope"] > 0.95 for k in CON_RES)

con_egp = {}
_rows = []
for k in CON_RES:
    v = con_eg_raw[k]
    o, h, ts, pw, fb = ols(Y, v), hac(Y, v), theil_sen(Y, v), prais_winsten(Y, v), fixed_b(Y, v)
    ivs = [(o["lo"], o["hi"]), (h[2], h[3]), (ts[1], ts[2]), (pw["lo"], pw["hi"]), (fb["lo"], fb["hi"])]
    con_egp[k] = sum(1 for lo, hi in ivs if not lo <= 0.0 <= hi)
    _rows.append(" & ".join(["%d" % k] + ["$%s$" % con_iv(lo, hi) for lo, hi in ivs]
                            + ["$%s$" % pval(o["p"]), "$%s$" % pval(mann_kendall(v, Y)[1]),
                               "%d of %d" % (con_egp[k], len(ivs))]))
    put("con.egp.excl%d" % k, con_egp[k], lambda n: ("none", "one", "two", "three", "four", "all five")[n])
    put("con.egp.olsp%d" % k, o["p"], pval)
table("si-egprocs", _rows)
put("con.egp.fb20.hi", fixed_b(Y, con_eg_raw[CON_RES[0]])["hi"], "{:+.5f}")
assert all(con_egp[k] == 5 for k in CON_RES[1:]), "an interval reaches zero above 20 cells per side"
print("EG own-hull share of the full-frame slope:", [round(r, 3) for r in _ratio], "intervals excluding zero:", con_egp)

EG own-hull share of the full-frame slope: [0.854, 0.948, 0.972, 0.962] intervals excluding zero: {20: 5, 30: 5, 40: 5, 50: 5}


## §2.3/§3.6/S4 · The multiscale function, its control and its nulls

This section covers four things. A known-answer control shows what the
border-corrected $L$ returns on patterns whose structure is fixed by
construction. The kernel null asks, year by year, whether $L$ on locations
exceeds what a pattern with the same smoothed intensity gives, and the
bandwidth sweep repeats that test at five kernel widths. The cell null
repeats it on the 10 m and 25 m merged sites. The last part counts the
clusters that the 25 m merge forms in 2026 and the themes inside them.

In [35]:
MSN_NNULL = 199                                   # draws per year in every null
MSN_H = 750.0                                     # headline kernel bandwidth, metres
MSN_BANDWIDTHS = [250.0, 500.0, 750.0, 1000.0, 1500.0]
MSN_TOL = 25.0                                    # merge radius audited, metres
MSN_SPREAD_STREAMS = 3                            # extra streams for the seed spread
MSN_WORDS = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine",
             "ten", "eleven", "twelve", "thirteen", "fourteen"]
MSN_I = {int(r): k for k, r in enumerate(RADII)}  # column of each scale in an L vector


def msn_word(n):
    return MSN_WORDS[int(n)]


def msn_cell(v, d=1):
    """A table cell: plus sign for positive values, math minus for negative ones."""
    s = ("{:+.%df}" % d).format(v)
    return s if v >= 0 else "$%s$" % s

### S4 · Known-answer control for the multiscale estimator

Three patterns are drawn inside the fixed hull: uniform points, tight Gaussian
clusters, and a jittered lattice. The border-corrected $L$ must be near zero
on the first, positive on the second and negative on the third. The uniform
pattern is also passed through an estimator with no edge correction, which
counts pairs over the whole window and loses those that fall outside it.

In [36]:
MSN_CTL_N = 300        # uniform points
MSN_CTL_BLOBS = 40     # cluster centres
MSN_CTL_PER = 8        # points per cluster
MSN_CTL_SD = 150.0     # cluster spread, metres
MSN_CTL_GRID = 26      # lattice nodes per side
MSN_CTL_JIT = 60.0     # lattice jitter, metres


def msn_L_uncorrected(P):
    """L from the raw pair count, with no edge correction."""
    tree, m = cKDTree(P), len(P)
    out = []
    for r in RADII:
        k = A_HULL * len(tree.query_pairs(r, output_type="ndarray")) / (m * (m - 1))
        out.append(math.sqrt(k / math.pi) - r)
    return np.array(out)


_g = rng("msn:control")
_csr = sample_hull(MSN_CTL_N, _g)
_cen = sample_hull(MSN_CTL_BLOBS, _g)
_clu = np.vstack([c + _g.normal(0.0, MSN_CTL_SD, (MSN_CTL_PER, 2)) for c in _cen])
_clu = _clu[in_hull(_clu, HULL_POLY)]
_gx = np.linspace(HULL_POLY[:, 0].min(), HULL_POLY[:, 0].max(), MSN_CTL_GRID)
_gy = np.linspace(HULL_POLY[:, 1].min(), HULL_POLY[:, 1].max(), MSN_CTL_GRID)
_GX, _GY = np.meshgrid(_gx, _gy)
_lat = np.column_stack([_GX.ravel(), _GY.ravel()]) + _g.normal(0.0, MSN_CTL_JIT, (_GX.size, 2))
_lat = _lat[in_hull(_lat, HULL_POLY)]

MSN_CTL = {"csr": L_border(_csr), "raw": msn_L_uncorrected(_csr),
           "clustered": L_border(_clu), "regular": L_border(_lat)}
assert np.all(MSN_CTL["clustered"] > 0) and np.all(MSN_CTL["regular"] < 0)
# beyond the shortest scale the border correction is nearer zero than no correction
assert np.all(np.abs(MSN_CTL["csr"][1:]) < np.abs(MSN_CTL["raw"][1:]))
# the clustered curve stops growing once the radius holds a whole cluster
assert MSN_CTL["clustered"][MSN_I[1000]] < MSN_CTL["clustered"][MSN_I[750]]

table("si-lcontrol", [
    "CSR, border correction & " + " & ".join(msn_cell(v) for v in MSN_CTL["csr"]),
    "CSR, no edge correction & " + " & ".join(msn_cell(v) for v in MSN_CTL["raw"]),
    "Clustered & " + " & ".join(msn_cell(v) for v in MSN_CTL["clustered"]),
    "Regular lattice & " + " & ".join(msn_cell(v) for v in MSN_CTL["regular"]),
])
put("msn.ctl.ncsr", MSN_CTL_N, "{:d}")
put("msn.ctl.blobs", MSN_CTL_BLOBS, "{:d}")
put("msn.ctl.per", MSN_CTL_PER, "{:d}")
put("msn.ctl.sd", MSN_CTL_SD, "{:.0f}")
put("msn.ctl.grid", MSN_CTL_GRID, "{:d}")
put("msn.ctl.jit", MSN_CTL_JIT, "{:.0f}")
put("msn.seed", BASE_SEED, "{:d}")
print({k: np.round(v, 1) for k, v in MSN_CTL.items()})

{'csr': array([-100. ,    2.5,  -98.1,  -44.1,  -55.1, -192.1, -182.9]), 'raw': array([-100. ,  -72.4, -219.1, -247.6, -335.4, -568.5, -706.8]), 'clustered': array([1802.4, 3624.8, 4906.7, 4817.3, 4572.4, 4072.4, 3572.4]), 'regular': array([ -100.,  -250.,  -500.,  -750., -1000., -1500., -2000.])}


### §3.6/S4 · The inhomogeneous null, year by year

Each job takes one unit, one year and one null, draws 199 patterns of the
same size from the null and records the 2.5th and 97.5th percentiles of $L$
at every scale. A year is *above* at a scale when its own $L$ exceeds the
upper percentile and *below* when it is under the lower one. The kernel null
is run on locations at five bandwidths and on the merged sites at 750 m. The
cell null is run on all three units, once on the stream that is printed and
three more times to measure how far the counts move between streams.

In [37]:
MSN_UNITS = {"loc": LOCS, "site10": SITES10, "site25": SITES25}


def msn_kernel_sampler(P, h):
    """The draw of `kernel_null`, with the node weights computed once per pattern."""
    C = _kernel_grid(h)
    d = cKDTree(P).query(C, k=min(len(P), 100))[0]
    w = np.clip(np.exp(-0.5 * (d / h) ** 2).sum(axis=1), 1e-12, None)
    p = w / w.sum()
    return lambda g: C[g.choice(len(C), len(P), replace=False, p=p)] + g.normal(0.0, h / 4.0, (len(P), 2))


def msn_job(job):
    kind, unit, h, y, stream = job
    P = MSN_UNITS[unit][y]
    g = rng("msn:%s:%s:%d:%d:%d" % (kind, unit, int(h), y, stream))
    if kind == "kernel":
        draw = msn_kernel_sampler(P, h)
    else:
        draw = lambda gg: cell_null(P, len(P), gg, sigma=h)
    null = np.array([L_border(draw(g)) for _ in range(MSN_NNULL)])
    return {"L": L_border(P), "lo": np.nanpercentile(null, 2.5, axis=0),
            "hi": np.nanpercentile(null, 97.5, axis=0)}


def msn_run():
    for h in MSN_BANDWIDTHS:
        _kernel_grid(h)                           # built once, shared by the workers
    jobs = [("kernel", "loc", h, y, 0) for h in MSN_BANDWIDTHS for y in YEARS]
    jobs += [("kernel", u, MSN_H, y, 0) for u in ("site10", "site25") for y in YEARS]
    jobs += [("cell", u, MSN_H, y, s) for u in MSN_UNITS for y in YEARS
             for s in range(1 + MSN_SPREAD_STREAMS)]
    return dict(zip(jobs, pmap(msn_job, jobs)))


MSN_NULL = memo("msn:nulls", msn_run)


def msn_counts(kind, unit, h, stream=0):
    """Years above, inside and below the null at each scale."""
    rows = [MSN_NULL[(kind, unit, h, y, stream)] for y in YEARS]
    above = np.sum([r["L"] > r["hi"] for r in rows], axis=0)
    below = np.sum([r["L"] < r["lo"] for r in rows], axis=0)
    return above, len(YEARS) - above - below, below

/tmp/ipykernel_16/2227444663.py:14: DeprecationWarning: This process (pid=16) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


### §3.6/S4 · Kernel null on locations, and the bandwidth sweep

In [38]:
MSN_BW = {h: msn_counts("kernel", "loc", h)[0] for h in MSN_BANDWIDTHS}
_ab, _in, _be = msn_counts("kernel", "loc", MSN_H)
table("si-lnull", ["%.1f m & %d & %d & %d" % (r, _ab[k], _in[k], _be[k]) for k, r in enumerate(RADII)])
table("si-bwsens", ["%d & " % h + " & ".join("%d" % v for v in MSN_BW[h]) for h in MSN_BANDWIDTHS])

# the claims the prose makes about these counts
_band = [MSN_I[r] for r in (250, 500, 750, 1000, 1500)]
assert all(_ab[k] == len(YEARS) for k in _band), "the 250 m to 1.5 km band is not unanimous at 750 m"
assert _be.sum() == 0, "a year falls below the kernel null"


def msn_unanimous_from(cols):
    """Smallest bandwidth from which every wider one has all years above at the given columns."""
    ok = [all(MSN_BW[h][k] == len(YEARS) for k in cols) for h in MSN_BANDWIDTHS]
    first = max(i for i, v in enumerate(ok) if not v) + 1 if not all(ok) else 0
    return first


_i1k = msn_unanimous_from([MSN_I[1000]])
_iband = msn_unanimous_from(_band)
# the prose names a bandwidth where 1 km fails and one from which the whole band holds
assert 0 < _i1k < len(MSN_BANDWIDTHS), "1 km is unanimous at every bandwidth, or at none"
assert MSN_BANDWIDTHS[_iband] == MSN_H, "the band is not unanimous exactly from the bandwidth the paper uses"
assert MSN_BW[250.0][MSN_I[1000]] < len(YEARS)
assert MSN_BW[500.0][MSN_I[1500]] < len(YEARS)

put("msn.nnull", MSN_NNULL, "{:d}", macro="envN")
put("msn.kern.w100", _ab[MSN_I[100]], msn_word)
put("msn.kern.w2000", _ab[MSN_I[2000]], msn_word)
put("msn.kern.above.1500", _ab[MSN_I[1500]], "{:d}")
put("msn.bw.250.w1000", MSN_BW[250.0][MSN_I[1000]], msn_word)
put("msn.bw.500.w1000", MSN_BW[500.0][MSN_I[1000]], msn_word)
put("msn.bw.h1k.from", MSN_BANDWIDTHS[_i1k], "{:.0f}")
put("msn.bw.h1k.fail", MSN_BANDWIDTHS[_i1k - 1], "{:.0f}")
put("msn.bw.band.from", MSN_BANDWIDTHS[_iband], "{:.0f}")
put("msn.bw.500.w1500", MSN_BW[500.0][MSN_I[1500]], msn_word)
print("kernel above:", {int(h): MSN_BW[h].tolist() for h in MSN_BANDWIDTHS})

kernel above: {250: [13, 13, 10, 10, 10, 10, 9], 500: [13, 14, 14, 14, 13, 12, 12], 750: [13, 14, 14, 14, 14, 14, 13], 1000: [14, 14, 14, 14, 14, 14, 14], 1500: [14, 14, 14, 14, 14, 14, 14]}


### S4 · Cell null on three deduplication tolerances

The table gives, for exact locations and the two merged units, the number of
years above the cell null at four scales and the fitted slope of
$L(1\,\mathrm{km})$ with and without 2013.

In [39]:
_scales = [250, 500, 1000, 1500]
_labels = {"loc": "Exact coordinate", "site10": "Merged within 10 m", "site25": "Merged within 25 m"}
MSN_CELL = {u: msn_counts("cell", u, MSN_H)[0] for u in MSN_UNITS}
MSN_L1K = {u: np.array([MSN_NULL[("cell", u, MSN_H, y, 0)]["L"][MSN_I[1000]] for y in YEARS]) for u in MSN_UNITS}
_rows = []
for u in MSN_UNITS:
    s_all = hac(Y, MSN_L1K[u])[0]
    s_no13 = hac(Y[1:], MSN_L1K[u][1:])[0]
    _rows.append("%s & %d & %s & %s & $%.0f$ & $%.0f$" % (
        _labels[u], len(MSN_UNITS[u][YEARS[-1]]),
        " & ".join("%d" % MSN_CELL[u][MSN_I[r]] for r in _scales),
        " & ".join("%d" % msn_counts("kernel", u, MSN_H)[0][MSN_I[r]] for r in _scales), s_all, s_no13))
    if u == "loc":
        put("msn.slope.loc", s_all, "{:.0f}")
        put("msn.slope.loc.raw", s_all, "{:.1f}")
table("si-site-L", _rows)

# claims: unanimity at the two short scales, loss at the two long ones, kernel null unanimous
assert all(MSN_CELL[u][MSN_I[r]] == len(YEARS) for u in MSN_UNITS for r in (250, 500))
for u in ("site10", "site25"):
    assert MSN_CELL[u][MSN_I[1000]] < len(YEARS) and MSN_CELL[u][MSN_I[1500]] < len(YEARS), u
for u in ("site10", "site25"):
    assert all(msn_counts("kernel", u, MSN_H)[0][MSN_I[r]] == len(YEARS) for r in _scales), u
_spread = max(
    max(msn_counts("cell", u, MSN_H, s)[0][MSN_I[r]] for s in range(1 + MSN_SPREAD_STREAMS))
    - min(msn_counts("cell", u, MSN_H, s)[0][MSN_I[r]] for s in range(1 + MSN_SPREAD_STREAMS))
    for u in MSN_UNITS for r in (1000, 1500))
assert _spread <= 2, "cell-null counts move by more than two between streams"

put("msn.n26.loc", len(LOCS[YEARS[-1]]), "{:d}")
put("msn.n26.site", len(SITES25[YEARS[-1]]), "{:d}")
put("msn.cell.loc.1500", MSN_CELL["loc"][MSN_I[1500]], "{:d}")
put("msn.cell.site25.w1000", MSN_CELL["site25"][MSN_I[1000]], msn_word)
put("msn.cell.site25.w1500", MSN_CELL["site25"][MSN_I[1500]], msn_word)
print("cell above:", {u: [int(MSN_CELL[u][MSN_I[r]]) for r in _scales] for u in MSN_UNITS}, "spread", _spread)

cell above: {'loc': [14, 14, 14, 10], 'site10': [14, 14, 12, 8], 'site25': [14, 14, 12, 9]} spread 2


### §3.3/S4/S18 · What the 25 m merge joins (cluster audit)

Single linkage at 25 m is applied to the 2026 locations. Every cluster of
more than one location is listed by size and by the themes of the
identifiers it holds, so the merge is counted and not asserted. The chain
statistics pool the clusters of identifiers over all fourteen years.

In [40]:
def msn_diameters(P, lab):
    """Largest pairwise distance inside each cluster of more than one point."""
    out = []
    for c in np.unique(lab):
        Q = P[lab == c]
        if len(Q) < 2:
            continue
        try:
            V = Q[ConvexHull(Q).vertices] if len(Q) > 2 else Q
        except Exception:
            V = Q
        out.append(float(max(np.linalg.norm(V[i] - V[j]) for i in range(len(V)) for j in range(i + 1, len(V)))))
    return out


_y = YEARS[-1]
_themes_at = defaultdict(set)
for _r in REC[_y].values():
    _themes_at[(round(_r["x"], 1), round(_r["y"], 1))].add(_r["theme"])
_P = LOCS[_y]
_loc_themes = [_themes_at[(float(p[0]), float(p[1]))] for p in _P]
_lab = single_labels(_P, MSN_TOL)
_members = defaultdict(list)
for _i, _c in enumerate(_lab):
    _members[int(_c)].append(_i)
_multi = [m for m in _members.values() if len(m) > 1]
_sizes = Counter(len(m) for m in _multi)
_dominant, _cross = Counter(), []
for _m in _multi:
    _cnt = Counter(t for i in _m for t in _loc_themes[i])
    _dominant[sorted(_cnt.items(), key=lambda kv: (-kv[1], kv[0]))[0][0]] += 1
    if len(_cnt) > 1:
        _cross.append((" + ".join(sorted(_cnt)), len(_m)))
_kinds = Counter(_cross)
_removed = sum(len(m) - 1 for m in _multi)
_cross_locs = sum(s for _, s in _cross)

# the structure the prose describes
assert len(_members) == len(SITES25[_y]), "location and identifier merges disagree"
assert set(_sizes) == {2, 3, 4} and _sizes[4] == 1
assert _kinds == Counter({("People + Vehicles", 3): 2, ("Air Quality + People", 4): 1,
                          ("Air Quality + People", 2): 2, ("Air Quality + Building", 2): 1,
                          ("Building + Sensor Metrics", 2): 1}), _kinds
assert _dominant["Water Level"] == 1 and _dominant["Building"] == 1
assert set(_dominant) == {"Vehicles", "People", "Air Quality", "Water Level", "Building"}
assert all(len(t) == 1 for t in _loc_themes), "a location carries two themes"

put("msn.cl.removed", _removed, "{:d}")
put("msn.cl.multi", len(_multi), "{:d}")
put("msn.cl.singletons", len(_members) - len(_multi), "{:d}")
put("msn.cl.pairs", _sizes[2], "{:d}")
put("msn.cl.triples", _sizes[3], "{:d}")
put("msn.cl.members", sum(len(m) for m in _multi), "{:d}")
put("msn.cl.single", len(_multi) - len(_cross), "{:d}")
put("msn.cl.singlepct", 100.0 * (len(_multi) - len(_cross)) / len(_multi), "{:.1f}")
put("msn.cl.cross", len(_cross), "{:d}")
put("msn.cl.crosspct", 100.0 * len(_cross) / len(_multi), "{:.1f}")
put("msn.cl.crossloc", _cross_locs, "{:d}")
put("msn.cl.crossrem", _cross_locs - len(_cross), "{:d}")
put("msn.cl.dom.vehicles", _dominant["Vehicles"], "{:d}")
put("msn.cl.dom.people", _dominant["People"], "{:d}")
put("msn.cl.dom.air", _dominant["Air Quality"], "{:d}")
put("msn.cl.exactdup", len(IDS[_y]) - len(LOCS[_y]), "{:d}")
put("msn.cl.idremoved", len(IDS[_y]) - len(SITES25[_y]), "{:d}")
put("msn.cl.y2013.ids", len(IDS[YEARS[0]]), "{:d}")
put("msn.cl.y2013.removed", len(IDS[YEARS[0]]) - len(SITES25[YEARS[0]]), "{:d}")
# the merge removes a larger share of the roster in the last year than in the first
assert (len(IDS[_y]) - len(SITES25[_y])) / len(IDS[_y]) > (len(IDS[YEARS[0]]) - len(SITES25[YEARS[0]])) / len(IDS[YEARS[0]])

_alld = []
for _yy in YEARS:
    _alld += msn_diameters(IDS[_yy], single_labels(IDS[_yy], MSN_TOL))
put("msn.chain.max", float(np.max(_alld)), "{:.0f}")
put("msn.chain.median", float(np.median(_alld)), "{:.0f}")
print("clusters", len(_multi), dict(_sizes), "cross", len(_cross), _cross_locs, dict(_dominant),
      "chain max %.1f median %.1f" % (np.max(_alld), np.median(_alld)))

clusters 152 {2: 145, 4: 1, 3: 6} cross 7 18 {'Vehicles': 139, 'People': 8, 'Building': 1, 'Air Quality': 3, 'Water Level': 1} chain max 84.6 median 10.8


## §3.3/§3.6/S4 · Support, the three controls together, and local statistics

This section covers the part of the argument that turns on the window a
pattern is measured in. It gives the areas that define the window effect, the
five unit-and-window configurations of the factorial table, the three controls
applied together, the pair correlation, the intensity-reweighted estimator,
the shape of the multiscale curve, the multiscale function on each year's own
hull, and the footprint-matching test.

In [41]:
SUP_K1 = int(np.where(RADII == 1000.0)[0][0])     # position of the 1 km radius
SUP_Y26 = YEARS.index(2026)
SUP_Y21 = YEARS.index(2021)
SUP_KM2 = 1e6
SUP_WORDS = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine",
             "ten", "eleven", "twelve", "thirteen", "fourteen"]


def sup_word(n, cap=False):
    """A small count written as a word."""
    w = SUP_WORDS[int(n)]
    return w.capitalize() if cap else w

### §3.3/S5 · The window: bounding box, footprint and the area ratio

The per-year bounding box and each year's own convex hull are both compared
with the fixed hull. The square root of the area ratio is the factor by which
the Clark-Evans index changes when the window changes and nothing else does.

In [42]:
put("sup.hull.km2", A_HULL / SUP_KM2, "{:.0f}")
put("sup.box.km2.2013", A_BOX[0] / SUP_KM2, "{:.0f}")
put("sup.box.km2.max", A_BOX.max() / SUP_KM2, "{:.0f}")
put("sup.box.factor", A_BOX.max() / A_BOX.min(), "{:.1f}")
put("sup.box.share.2013", A_BOX[0] / A_HULL, "{:.2f}")
put("sup.box.rootratio.2013", math.sqrt(A_HULL / A_BOX[0]), "{:.2f}")
put("sup.own.km2.2013", A_OWN[0] / SUP_KM2, "{:.0f}")
put("sup.own.km2.2026", A_OWN[SUP_Y26] / SUP_KM2, "{:.0f}")
put("sup.own.share.2013", A_OWN[0] / A_HULL, "{:.2f}")
put("sup.own.share.2021", A_OWN[SUP_Y21] / A_HULL, "{:.2f}")

# The prose names the run of years in which the box is larger than the hull, says the
# box index is below the hull index there, and that the footprint is widest in 2021.
SUP_BOX_OVER = [k for k in range(len(YEARS)) if A_BOX[k] > A_HULL]
assert SUP_BOX_OVER == list(range(SUP_BOX_OVER[0], SUP_BOX_OVER[-1] + 1)) and A_BOX[0] < A_HULL
assert all(R_LOC_BOX[k] < R_LOC[k] for k in SUP_BOX_OVER)
assert int(np.argmax(A_OWN)) == SUP_Y21 and int(np.argmax(A_BOX)) in SUP_BOX_OVER
put("sup.box.over.first", YEARS[SUP_BOX_OVER[0]], "{:d}")
put("sup.box.over.last", YEARS[SUP_BOX_OVER[-1]], "{:d}")

2022

### §3.6/S4 · Multiscale L on the fixed hull and on each year's own hull

`L` at 1 km is fitted against year twice: on the fixed hull, and with each
year measured on its own convex hull and its own area. The own-hull slope is
reported with its Newey-West standard error.

In [43]:
L_OWN = np.array([L_border(LOCS[y], RADII, OWN_POLY[y], A_OWN[k]) for k, y in enumerate(YEARS)])
SUP_L1K_FIX = L_LOC[:, SUP_K1]
SUP_L1K_OWN = L_OWN[:, SUP_K1]

_fix = ols(Y, SUP_L1K_FIX)
_own = hac(Y, SUP_L1K_OWN)
put("sup.l1k.fixed.slope", _fix["slope"], "{:.0f}", macro="lKmExact")
put("sup.l1k.own.slope", _own[0], "{:.0f}")
put("sup.l1k.own.se", _own[1], "{:.0f}")
put("sup.l1k.2026", SUP_L1K_FIX[SUP_Y26], "{:.0f}")
assert _own[2] < 0 < _own[3]          # the own-hull slope is indistinguishable from zero
assert _fix["slope"] < 0 and abs(_own[0]) < 0.5 * abs(_fix["slope"])   # most of the decline goes

table("si-ownhull", [
    "%d & %.0f & %.0f & %.0f & %.3f" % (y, SUP_L1K_OWN[k], SUP_L1K_FIX[k], A_OWN[k] / SUP_KM2, A_OWN[k] / A_HULL)
    for k, y in enumerate(YEARS)])

### S4 · Normalised multiscale curve shape by year

Each year's fixed-hull function is divided by its own value at 2 km, so the
level, which the window sets, cancels and only the shape is left.

In [44]:
SUP_SHAPE_YEARS = (2013, 2017, 2020, 2023, 2026)
table("si-lshape", [
    "%d & " % y + " & ".join("%+.2f" % v for v in L_LOC[YEARS.index(y)] / L_LOC[YEARS.index(y)][-1])
    for y in SUP_SHAPE_YEARS])

### §3.3/Tab. 3 · The two controls in combination

Five configurations cross the unit (identifiers, locations, sites merged at
25 m) with the window (the fixed hull, each year's own hull). For each the
table gives the two endpoints, the OLS slope with its standard error, and the
endpoint fall.

In [45]:
SUP_CONFIGS = {"ids": R_IDS, "loc": R_LOC, "site": R_SITE25, "siteown": R_SITE25_OWN, "locown": R_LOC_OWN}
SUP_FAC = {c: ols(Y, v) for c, v in SUP_CONFIGS.items()}
_slope_fmt = {"ids": "{:.5f}", "loc": "{:.5f}", "site": "{:+.4f}", "siteown": "{:.4f}", "locown": "{:.4f}"}
_macros = {
    "ids": {"first": "rHullFirst", "last": "rHullLast", "fall": "idsFall"},
    "loc": {"first": "rHullLocFirst", "last": "rHullLocLast", "fall": "locHullDropOne"},
    "site": {"first": "mergedRFirst", "last": "mergedRLast", "fall": "mergedFall", "slope": "mergedSlope",
             "se": "mergedSE", "p": "mergedP"},
    "siteown": {"last": "ownMergedRLast", "fall": "ownMergedFall", "slope": "ownMergedSlope",
                "se": "ownMergedSE", "p": "ownMergedP"},
    "locown": {"first": "rOwnHullFirst", "last": "rOwnHullLast", "fall": "rOwnHull", "slope": "ownExactSlope",
               "se": "ownExactSE"},
}
for _c, _v in SUP_CONFIGS.items():
    _m = _macros[_c]
    put("sup.fac.%s.first" % _c, _v[0], "{:.3f}", macro=_m.get("first"))
    put("sup.fac.%s.last" % _c, _v[-1], "{:.3f}", macro=_m.get("last"))
    put("sup.fac.%s.slope" % _c, SUP_FAC[_c]["slope"], _slope_fmt[_c], macro=_m.get("slope"))
    put("sup.fac.%s.se" % _c, SUP_FAC[_c]["se"], "{:.4f}", macro=_m.get("se"))
    put("sup.fac.%s.p" % _c, SUP_FAC[_c]["p"], pval, macro=_m.get("p"))
    put("sup.fac.%s.fall" % _c, fall(_v), pct, macro=_m.get("fall"))
put("sup.fac.ids.fall0", 100 * fall(R_IDS), "{:.0f}", macro="rHullDrop")
put("sup.fac.loc.fall0", 100 * fall(R_LOC), "{:.0f}", macro="locHullDrop")
put("sup.fac.loc.fall0b", 100 * fall(R_LOC), "{:.0f}")

put("sup.fac.fall.min", 100 * min(fall(v) for v in SUP_CONFIGS.values()), "{:.0f}")
put("sup.fac.fall.max", 100 * max(fall(v) for v in SUP_CONFIGS.values()), "{:.0f}")

# Every configuration falls. Merging removes the fitted trend on the fixed hull, the own
# hull enlarges the fall, and the two together bring a decline back: the order the prose
# states. The per-year box also enlarges the fall.
assert all(fall(v) > 0 for v in SUP_CONFIGS.values())
assert fall(R_SITE25) < fall(R_SITE25_OWN) < fall(R_LOC) < fall(R_LOC_OWN)
assert SUP_FAC["siteown"]["p"] < 0.05 < SUP_FAC["site"]["p"]
assert abs(SUP_FAC["site"]["slope"]) < 0.1 * abs(SUP_FAC["loc"]["slope"])
assert fall(R_LOC_BOX) > fall(R_LOC)

### S4 · The own-hull decline is the window factor (identity check)

Because the index is proportional to the inverse square root of the area,
moving from the fixed hull to each year's own hull multiplies each year's
index by the square root of the area ratio. The endpoint change on the own
hull therefore follows from the fixed-hull endpoints and the two areas alone.
It is evaluated from the unrounded quantities and again from the four values
as printed.

In [46]:
_exact = 1.0 - (R_SITE25[-1] / R_SITE25[0]) * math.sqrt(A_OWN[0] / A_OWN[SUP_Y26])
_printed = 1.0 - (round(R_SITE25[-1], 3) / round(R_SITE25[0], 3)) * math.sqrt(
    round(A_OWN[0] / SUP_KM2) / round(A_OWN[SUP_Y26] / SUP_KM2))
put("sup.id.exact", 100 * _exact, "{:.2f}")
put("sup.id.observed", 100 * fall(R_SITE25_OWN), "{:.2f}")
put("sup.id.printed", 100 * _printed, "{:.2f}")
_hundredths = int(round(abs(round(100 * _printed, 2) - round(100 * _exact, 2)) * 100))
put("sup.id.diff.phrase", _hundredths, lambda n: ("one hundredth" if n == 1 else "%s hundredths" % sup_word(n)) + " of a percentage point")
assert abs(_exact - fall(R_SITE25_OWN)) < 1e-9 and 1 <= _hundredths <= 14

### §3.3/S4 · All three controls at once (rarefied merged sites)

The merged unit is rarefied to the smallest year's site count on each year's
own hull and on the fixed hull. For each frame the annual means are fitted by
OLS, the replicate interval is the central 95 per cent of the slopes of the
single draws, and the equivalence test uses the unrarefied fixed-hull location
slope as its margin.

In [47]:
SUP_N_TO = int(N_SITES25.min())
SUP_MARGIN = abs(SUP_FAC["loc"]["slope"])
put("sup.tw.n", SUP_N_TO, "{:d}")
put("sup.tw.margin", SUP_MARGIN, "{:.5f}")


def sup_threeway(frame):
    D = rarefied("site25", frame)
    m = D.mean(axis=1)
    o = ols(Y, m)
    x = Y - Y.mean()
    lo, hi = np.percentile((x @ D) / (x @ x), [2.5, 97.5])
    return {"mean": m, "ols": o, "lo": float(lo), "hi": float(hi),
            "change": float((m[-1] - m[0]) / m[0]), "tost": tost(o["slope"], o["se"], SUP_MARGIN, o["df"])}


SUP_TW = {f: sup_threeway(f) for f in ("own", "fixed")}
_tw_macros = {
    "own": {"change": "rareOwnMergedFall", "slope": "rareOwnMergedSlope", "lo": "rareOwnMergedLO",
            "hi": "rareOwnMergedHI"},
    "fixed": {"last": "rareFixedMergedRLast", "change": "rareFixedMergedRise", "slope": "rareFixedMergedSlope",
              "p": "rareFixedMergedP"},
}
for _f, _t in SUP_TW.items():
    _m = _tw_macros[_f]
    put("sup.tw.%s.first" % _f, _t["mean"][0], "{:.3f}")
    put("sup.tw.%s.last" % _f, _t["mean"][-1], "{:.3f}", macro=_m.get("last"))
    put("sup.tw.%s.change" % _f, abs(_t["change"]), pct, macro=_m.get("change"))
    put("sup.tw.%s.slope" % _f, _t["ols"]["slope"], "{:.5f}" if _f == "own" else "{:+.5f}", macro=_m.get("slope"))
    put("sup.tw.%s.se" % _f, _t["ols"]["se"], "{:.4f}")
    put("sup.tw.%s.p" % _f, _t["ols"]["p"], pval, macro=_m.get("p"))
    put("sup.tw.%s.lo" % _f, _t["lo"], "{:+.5f}", macro=_m.get("lo"))
    put("sup.tw.%s.hi" % _f, _t["hi"], "{:+.5f}", macro=_m.get("hi"))
    put("sup.tw.%s.tost" % _f, _t["tost"], "{:.2f}" if _t["tost"] >= 0.1 else "{:.3f}")

# The prose states: on the fixed hull a small endpoint rise, no trend, an interval that
# contains zero and equivalence to zero at the margin; on the own hull a fall that is not
# equivalent to zero and that equals the window factor applied to the fixed-hull series.
_twf, _two = SUP_TW["fixed"], SUP_TW["own"]
assert _twf["change"] > 0 and _twf["ols"]["p"] > 0.05 and _twf["lo"] < 0 < _twf["hi"] and _twf["tost"] < 0.05
assert _two["change"] < -0.2 and _two["tost"] > 0.05
_window = 1.0 - (_twf["mean"][-1] / _twf["mean"][0]) * math.sqrt(A_OWN[0] / A_OWN[SUP_Y26])
assert abs(-_two["change"] - _window) < 0.03
assert A_OWN[SUP_Y26] > 2 * A_OWN[0]


def _tw_row(label, first, last, o, rep, change):
    return ("%s & %.3f & %.3f & $%+.4f$ (SE \\num{%.4f}, $p = %s$) & %s & %s\\%% %s"
            % (label, first, last, o["slope"], o["se"], pval(o["p"]), rep, pct(abs(change)),
               "fall" if change < 0 else "rise"))


SUP_TW_ROWS = [
    _tw_row("Merged \\SI{25}{\\meter}, own hull, all sites", R_SITE25_OWN[0], R_SITE25_OWN[-1],
            SUP_FAC["siteown"], "--", -fall(R_SITE25_OWN)),
    _tw_row("Merged \\SI{25}{\\meter}, own hull of all sites, rarefied to %d" % SUP_N_TO, SUP_TW["own"]["mean"][0],
            SUP_TW["own"]["mean"][-1], SUP_TW["own"]["ols"],
            "$[%+.4f,%+.4f]$" % (SUP_TW["own"]["lo"], SUP_TW["own"]["hi"]), SUP_TW["own"]["change"]),
    _tw_row("Merged \\SI{25}{\\meter}, fixed hull, rarefied to %d" % SUP_N_TO, SUP_TW["fixed"]["mean"][0],
            SUP_TW["fixed"]["mean"][-1], SUP_TW["fixed"]["ols"],
            "$[%+.4f,%+.4f]$" % (SUP_TW["fixed"]["lo"], SUP_TW["fixed"]["hi"]), SUP_TW["fixed"]["change"]),
]

/tmp/ipykernel_16/2227444663.py:14: DeprecationWarning: This process (pid=16) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


### §3.3/S4 · The third control with the area taken from the same draw

The own-hull row above rarefies the sites to a common size and keeps the
hull of all the year's sites in the denominator, so the area still carries
the year's full count. Here each draw of 43 sites is measured on the convex
hull of those 43 sites.

In [48]:
def sup_drawhull(unit, n_to, tag, M=1000):
    """D[year, draw]: R of n_to sites on the convex hull of those same sites."""
    pts = [UNITS[unit][y] for y in YEARS]

    def one(i):
        g = rng("%s:%d" % (tag, i))
        out = []
        for _ in range(M):
            Q = pts[i][g.choice(len(pts[i]), n_to, replace=False)]
            out.append(ce_r(Q, hull_of(Q)[1]))
        return out
    return np.array(pmap(one, range(len(pts))))


SUP_DH = memo("sup:drawhull", lambda: sup_drawhull("site25", SUP_N_TO, "sup:drawhull"))
_m = SUP_DH.mean(axis=1)
_o = ols(Y, _m)
_xc = Y - Y.mean()
_lo, _hi = (float(q) for q in np.percentile((_xc @ SUP_DH) / (_xc @ _xc), [2.5, 97.5]))
SUP_TWD = {"mean": _m, "ols": _o, "lo": _lo, "hi": _hi, "change": float((_m[-1] - _m[0]) / _m[0]),
           "tost": tost(_o["slope"], _o["se"], SUP_MARGIN, _o["df"])}
put("sup.tw.draw.first", _m[0], "{:.3f}")
put("sup.tw.draw.last", _m[-1], "{:.3f}")
put("sup.tw.draw.change", abs(SUP_TWD["change"]), pct)
put("sup.tw.draw.dir", "fall" if SUP_TWD["change"] < 0 else "rise")
put("sup.tw.draw.slope", _o["slope"], "{:+.5f}")
put("sup.tw.draw.p", _o["p"], pval)
put("sup.tw.draw.lo", _lo, "{:+.5f}")
put("sup.tw.draw.hi", _hi, "{:+.5f}")
put("sup.tw.draw.tost", SUP_TWD["tost"], lambda p: "< 0.001" if p < 0.001 else "= %.3f" % p)
# The prose: on the hull of the same draw the series is flat and equivalent to zero at the margin.
assert abs(SUP_TWD["change"]) < 0.05 and _o["p"] > 0.5 and _lo < 0 < _hi and SUP_TWD["tost"] < 0.05
SUP_TW_ROWS.insert(2, _tw_row("Merged \\SI{25}{\\meter}, hull of each draw, rarefied to %d" % SUP_N_TO,
                              _m[0], _m[-1], _o, "$[%+.4f,%+.4f]$" % (_lo, _hi), SUP_TWD["change"]))
table("si-threeway", SUP_TW_ROWS)
print("draw-hull series", np.round(_m, 3).tolist())

/tmp/ipykernel_16/2227444663.py:14: DeprecationWarning: This process (pid=16) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


draw-hull series [1.006, 0.924, 1.019, 1.018, 1.012, 1.068, 1.047, 0.988, 1.003, 0.974, 0.993, 0.999, 1.004, 0.996]


### S4 · The pair correlation g(r)

The pair correlation on annuli is the increment of the same border-corrected
K divided by the annulus area, so it is local in distance where K and L are
cumulative. It is computed for the 2026 locations and identifiers. As a
control it is averaged over 199 uniform draws of the same size inside the
hull, where every annulus should return one. The medium-scale band is the
four annuli from 250 m to 1.5 km.

In [49]:
SUP_G_LOC = g_annular(LOCS[2026])
SUP_G_IDS = g_annular(IDS[2026])
SUP_G_DRAWS = 199
SUP_G_CSR = np.mean(pmap(lambda j: g_annular(sample_hull(len(LOCS[2026]), rng("sup:gcsr:%d" % j))),
                         range(SUP_G_DRAWS)), axis=0)
SUP_BAND = slice(3, 7)                      # annuli from 250 m to 1.5 km


def _g_fmt(k, v):
    return "%.0f" % v if k == 0 else "%.1f" % v


table("si-gfun", [
    "%.0f--%.0f\\,m & %s & %s" % (ANNULI[k], ANNULI[k + 1], _g_fmt(k, SUP_G_LOC[k]), _g_fmt(k, SUP_G_IDS[k]))
    for k in range(len(ANNULI) - 1)])
for _k in range(7):
    put("sup.g.loc.%d" % _k, SUP_G_LOC[_k], "{:.0f}" if _k == 0 else "{:.1f}")
put("sup.g.csr.min", SUP_G_CSR.min(), "{:.2f}")
put("sup.g.csr.max", SUP_G_CSR.max(), "{:.2f}")
assert 0.8 < SUP_G_CSR.min() and SUP_G_CSR.max() < 1.2      # the control shows no bias
put("sup.g.ratio", SUP_G_LOC[0] / SUP_G_LOC[SUP_BAND].mean(), "{:.0f}")
put("sup.g.band.min", SUP_G_LOC[SUP_BAND].min(), "{:.0f}")

# The same K returns the L of section 3 at 1 km.
_K1km = K_border(LOCS[2026], [1000.0])[0]
assert abs(math.sqrt(_K1km / math.pi) - 1000.0 - SUP_L1K_FIX[SUP_Y26]) < 1e-6
assert np.all(np.diff(SUP_G_LOC[:4]) < 0) and SUP_G_LOC[SUP_BAND].min() > 10

/tmp/ipykernel_16/2227444663.py:14: DeprecationWarning: This process (pid=16) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


### S4 · The reference set of the annular estimator

Each K in the annular difference is evaluated on its own eroded window, so
the annuli telescope to the cumulative K. The alternative holds both terms
of an annulus to the outer eroded window. The two are compared on the 2026
locations.

In [50]:
def sup_g_common(P, edges=ANNULI, poly=HULL_POLY, area=A_HULL):
    """Annular pair correlation with both radii of an annulus counted from the outer eroded reference set."""
    P = np.asarray(P, float)
    lines, tree, out = _lines(poly), cKDTree(P), []
    for r1, r2 in zip(edges[:-1], edges[1:]):
        ref = _eroded(P, lines, r2)
        c2 = tree.query_ball_point(P[ref], r2, return_length=True)
        c1 = tree.query_ball_point(P[ref], r1, return_length=True) if r1 > 0 else np.ones(int(ref.sum()), int)
        out.append(area * float(np.sum(c2 - c1)) / (len(P) * int(ref.sum())) / (math.pi * (r2 ** 2 - r1 ** 2)))
    return np.array(out)


SUP_G_COMMON = sup_g_common(LOCS[2026])
_gdiff = float(np.max(np.abs(SUP_G_COMMON / SUP_G_LOC - 1.0)))
put("sup.g.common.maxdiff", 100.0 * _gdiff, "{:.1f}")
put("sup.g.nref.near", n_inside(LOCS[2026], ANNULI[1]), "{:d}")
put("sup.g.nref.far", n_inside(LOCS[2026], ANNULI[-1]), "{:d}")
put("sup.g.n", len(LOCS[2026]), "{:d}")
assert _gdiff < 0.05, "the two reference conventions differ by more than five per cent"
assert SUP_G_LOC.min() > 0 and SUP_G_COMMON.min() > 0, "an annular estimate is not positive"

### S4 · The intensity-reweighted border estimator L_inhom

Each location receives the reciprocal of its leave-one-out Gaussian intensity
at a bandwidth of 750 m, rescaled so that the reciprocals sum to the hull
area. Reference points are those inside the hull eroded by the radius, and the
weighted pair sum is divided by the reciprocal mass of the reference points.
The estimator is undefined when a location has no neighbour within reach of
the kernel. It is computed for four years and, each time, for one uniform
draw of the same size.

In [51]:
SUP_H = 750.0
SUP_INHOM_RADII = (250.0, 500.0, 1000.0, 1500.0)
SUP_INHOM_YEARS = (2013, 2018, 2021, 2026)


def sup_loo_intensity(P, h=SUP_H):
    d2 = ((P[:, None, :] - P[None, :, :]) ** 2).sum(axis=2)
    k = np.exp(-d2 / (2 * h * h)) / (2 * math.pi * h * h)
    np.fill_diagonal(k, 0.0)
    return k.sum(axis=1)


def sup_l_inhom(P):
    """L_inhom at the four radii, or None when a location has zero intensity."""
    lam = sup_loo_intensity(P)
    if np.any(lam <= 0):
        return None
    inv = 1.0 / lam
    inv *= A_HULL / inv.sum()
    tree = cKDTree(P)
    out = []
    for r in SUP_INHOM_RADII:
        ref = np.nonzero(eroded_inside(P, r))[0]
        num = sum(inv[i] * (inv[tree.query_ball_point(P[i], r)].sum() - inv[i]) for i in ref)
        mass = inv[ref].sum()
        out.append(math.sqrt(max(num / mass, 0.0) / math.pi) - r if mass > 0 else float("nan"))
    return {"L": out, "share": float(inv.max() / inv.sum())}


def _inhom_year(y):
    P = LOCS[y]
    return y, len(P), sup_l_inhom(P), sup_l_inhom(sample_hull(len(P), rng("sup:linhom:%d" % y)))


SUP_INHOM = pmap(_inhom_year, SUP_INHOM_YEARS)


def _inhom_cell(obs, csr, k):
    if obs is None:
        return "undefined"
    return "$%.0f$ / %s" % (obs["L"][k], "undefined" if csr is None else "$%.0f$" % csr["L"][k])


table("si-inhom", [
    "%d ($n=%d$) & " % (y, n) + " & ".join(_inhom_cell(obs, csr, k) for k in range(len(SUP_INHOM_RADII)))
    for y, n, obs, csr in SUP_INHOM])

# The prose says one location carries almost all the weight in each of the four years
# and that pattern and control both return -r.
for _y, _n, _obs, _csr in SUP_INHOM:
    assert _obs is not None and _csr is not None
    assert _obs["share"] > 0.99
    assert all(abs(v + r) < 0.02 * r for v, r in zip(_obs["L"], SUP_INHOM_RADII))
    assert all(abs(v + r) < 0.02 * r for v, r in zip(_csr["L"], SUP_INHOM_RADII))

/tmp/ipykernel_16/2227444663.py:14: DeprecationWarning: This process (pid=16) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


### S4 · The reweighted estimator with the evaluation point kept in its own intensity

With the evaluation point included, the kernel intensity at a location is
bounded below by the kernel's own peak, so an isolated location cannot take
all the weight. The estimator is compared with draws from the same kernel
intensity and with complete spatial randomness, each re-estimating its own
intensity.

In [52]:
SUP_INHOM_DRAWS = 99


def sup_l_inhom_self(P, h=SUP_H):
    """L_inhom at the four radii with the self term in the intensity, and the largest weight share."""
    P = np.asarray(P, float)
    d2 = ((P[:, None, :] - P[None, :, :]) ** 2).sum(axis=2)
    inv = 1.0 / (np.exp(-d2 / (2 * h * h)) / (2 * math.pi * h * h)).sum(axis=1)
    share = float(inv.max() / inv.sum())
    inv = inv * (A_HULL / inv.sum())
    out = []
    for r in SUP_INHOM_RADII:
        ref = eroded_inside(P, r)
        num = float(inv[ref] @ ((d2[ref] <= r * r) @ inv - inv[ref]))
        out.append(math.sqrt(max(num / inv[ref].sum(), 0.0) / math.pi) - r)
    return np.array(out), share


def _inhom_self_year(y):
    P = LOCS[y]
    obs, share = sup_l_inhom_self(P)
    draw = msn_kernel_sampler(P, SUP_H)
    ker = np.array([sup_l_inhom_self(draw(rng("sup:linhom-self:kernel:%d:%d" % (y, j))))[0]
                    for j in range(SUP_INHOM_DRAWS)])
    csr = np.array([sup_l_inhom_self(sample_hull(len(P), rng("sup:linhom-self:csr:%d:%d" % (y, j))))[0]
                    for j in range(SUP_INHOM_DRAWS)])
    return {"y": y, "n": len(P), "obs": obs, "share": share,
            "ker": np.percentile(ker, [2.5, 97.5], axis=0), "csr": np.percentile(csr, [2.5, 97.5], axis=0)}


SUP_INHOM_SELF = memo("sup:linhom-self", lambda: pmap(_inhom_self_year, SUP_INHOM_YEARS))
table("si-inhom-self", [
    "%d ($n=%d$) & " % (s["y"], s["n"]) + " & ".join(
        "$%+.0f$ / $[%+.0f, %+.0f]$" % (s["obs"][k], s["ker"][0][k], s["ker"][1][k])
        for k in range(len(SUP_INHOM_RADII)))
    for s in SUP_INHOM_SELF])
_cells_n = len(SUP_INHOM_SELF) * len(SUP_INHOM_RADII)
_above_ker = sum(int(s["obs"][k] > s["ker"][1][k]) for s in SUP_INHOM_SELF for k in range(len(SUP_INHOM_RADII)))
_above_csr = sum(int(s["obs"][k] > s["csr"][1][k]) for s in SUP_INHOM_SELF for k in range(len(SUP_INHOM_RADII)))
put("sup.inhom.self.cells", _cells_n, "{:d}")
put("sup.inhom.self.above.kernel", _above_ker, "{:d}")
put("sup.inhom.self.above.csr", _above_csr, "{:d}")
put("sup.inhom.self.share.max", 100.0 * max(s["share"] for s in SUP_INHOM_SELF), "{:.1f}")
put("sup.inhom.self.draws", SUP_INHOM_DRAWS, "{:d}")
for _k, _r in enumerate(SUP_INHOM_RADII):
    put("sup.inhom.self.above.r%d" % _r, sum(int(s["obs"][_k] > s["ker"][1][_k]) for s in SUP_INHOM_SELF), sup_word)
    put("sup.inhom.self.below.r%d" % _r, sum(int(s["obs"][_k] < s["ker"][0][_k]) for s in SUP_INHOM_SELF), sup_word)
put("sup.inhom.self.years", len(SUP_INHOM_SELF), sup_word)
_later = [s for s in SUP_INHOM_SELF if s["y"] != YEARS[0]]
# The prose: in the three later years the estimator is above its control at the two shorter radii;
# at the longest radius it is above in no year; in the first year it is never above.
assert all(s["obs"][k] > s["ker"][1][k] for s in _later for k in (0, 1)), "a later year is inside its control at a short radius"
assert not any(s["obs"][-1] > s["ker"][1][-1] for s in SUP_INHOM_SELF), "a year is above its control at the longest radius"
assert not any(s["obs"][k] > s["ker"][1][k] for s in SUP_INHOM_SELF if s["y"] == YEARS[0] for k in range(len(SUP_INHOM_RADII)))
_s26 = next(s for s in SUP_INHOM_SELF if s["y"] == 2026)
_k1 = SUP_INHOM_RADII.index(1000.0)
put("sup.inhom.self.obs1k", _s26["obs"][_k1], "{:+.0f}")
put("sup.inhom.self.ker1k.lo", _s26["ker"][0][_k1], "{:+.0f}")
put("sup.inhom.self.ker1k.hi", _s26["ker"][1][_k1], "{:+.0f}")
assert max(s["share"] for s in SUP_INHOM_SELF) < 0.25, "one location still takes a quarter of the weight"
print("L_inhom, self term kept: above kernel control %d of %d, above CSR %d of %d"
      % (_above_ker, _cells_n, _above_csr, _cells_n))
for s in SUP_INHOM_SELF:
    print(s["y"], np.round(s["obs"]).tolist(), "kernel", np.round(s["ker"]).tolist(), "csr", np.round(s["csr"]).tolist())

/tmp/ipykernel_16/2227444663.py:14: DeprecationWarning: This process (pid=16) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


L_inhom, self term kept: above kernel control 7 of 16, above CSR 16 of 16
2013 [424.0, 711.0, 1506.0, 2405.0] kernel [[-250.0, 312.0, 1587.0, 2683.0], [1183.0, 2135.0, 3704.0, 5576.0]] csr [[-250.0, -500.0, -1000.0, -1500.0], [-250.0, 232.0, 345.0, 581.0]]
2018 [568.0, 584.0, 796.0, 1009.0] kernel [[57.0, 215.0, 624.0, 1074.0], [189.0, 355.0, 800.0, 1281.0]] csr [[-127.0, -218.0, -335.0, -336.0], [-57.0, -146.0, -240.0, -247.0]]
2021 [717.0, 737.0, 1012.0, 1350.0] kernel [[92.0, 301.0, 834.0, 1354.0], [244.0, 473.0, 1042.0, 1608.0]] csr [[-133.0, -237.0, -350.0, -360.0], [-68.0, -156.0, -257.0, -258.0]]
2026 [1158.0, 1152.0, 1459.0, 1955.0] kernel [[149.0, 419.0, 1122.0, 1848.0], [348.0, 643.0, 1437.0, 2273.0]] csr [[-169.0, -263.0, -392.0, -396.0], [-64.0, -146.0, -265.0, -279.0]]


### §3.6/S4 · Footprint matching by year

For each year the 2026 locations that fall inside that year's own hull are
subsampled without replacement to that year's count, 299 times. A 2026
location within a millionth of a metre of the hull counts as inside, so hull
vertices are included. The year's own function is compared with the central
95 per cent of the draws at every scale. A year is not testable when fewer
2026 locations fall inside its hull than it has locations. The number of
reference points that survive erosion at 1 km is recorded for the year and
for the draws.

In [53]:
SUP_FP_DRAWS = 299
SUP_FP_TOL = 1e-6


def _footprint_year(k):
    y = YEARS[k]
    Py, poly, A = LOCS[y], OWN_POLY[y], A_OWN[k]
    P26 = LOCS[2026]
    inside = P26[in_hull(P26, poly, SUP_FP_TOL)]
    row = {"year": y, "n": len(Py), "avail": len(inside), "tested": len(inside) >= len(Py)}
    if row["tested"]:
        g = rng("sup:footprint:%d" % y)
        env = np.empty((SUP_FP_DRAWS, len(RADII)))
        ner = np.empty(SUP_FP_DRAWS)
        for j in range(SUP_FP_DRAWS):
            Q = inside[g.choice(len(inside), len(Py), replace=False)]
            env[j] = L_border(Q, RADII, poly, A)
            ner[j] = n_inside(Q, 1000.0, poly)
        lo, hi = np.nanpercentile(env, 2.5, axis=0), np.nanpercentile(env, 97.5, axis=0)
        obs = L_OWN[k]
        fin = np.isfinite(obs)
        row.update(obs=float(obs[SUP_K1]), lo=float(lo[SUP_K1]), hi=float(hi[SUP_K1]),
                   above_1km=bool(obs[SUP_K1] > hi[SUP_K1]), above=int((obs > hi)[fin].sum()),
                   scales=int(fin.sum()), ner_obs=n_inside(Py, 1000.0, poly),
                   ner_mean=float(ner.mean()), ner_min=int(ner.min()))
    return row


SUP_FP = pmap(_footprint_year, range(len(YEARS)))


def _fp_row(r):
    if not r["tested"]:
        return "%d & %d & %d & -- & -- & -- & -- & --" % (r["year"], r["n"], r["avail"])
    return "%d & %d & %d & %.0f & [%.0f, %.0f] & %d/%d & %d & %.0f/%d" % (
        r["year"], r["n"], r["avail"], r["obs"], r["lo"], r["hi"], r["above"], r["scales"],
        r["ner_obs"], r["ner_mean"], r["ner_min"])


table("si-footprint", [_fp_row(r) for r in SUP_FP])

_early = [r for r in SUP_FP if r["tested"] and r["year"] < 2026]
put("sup.fp.tested.word", len(_early), sup_word)
put("sup.fp.tested.first", _early[0]["year"], "{:d}")
put("sup.fp.tested.last", _early[-1]["year"], "{:d}")
put("sup.fp.untested.word", sum(not r["tested"] for r in SUP_FP), sup_word)
put("sup.fp.untested.Word", sum(not r["tested"] for r in SUP_FP), lambda n: sup_word(n, cap=True))
for _r in (_early[0], _early[1]):
    put("sup.fp.%d.n" % _r["year"], _r["n"], "{:d}")
    put("sup.fp.%d.ner" % _r["year"], _r["ner_obs"], "{:d}")
    put("sup.fp.%d.mean" % _r["year"], _r["ner_mean"], "{:.0f}")
    put("sup.fp.%d.min" % _r["year"], _r["ner_min"], "{:d}")
_r = _early[-1]
put("sup.fp.late.n", _r["n"], "{:d}")
put("sup.fp.late.ner", _r["ner_obs"], "{:d}")
put("sup.fp.late.mean", _r["ner_mean"], "{:.0f}")
put("sup.fp.late.min", _r["ner_min"], "{:d}")

# The prose says the testable earlier years are the first ones of the record, in an
# unbroken run starting in 2013 and 2014, that every one of them lies above its envelope
# at 1 km, and that the erosion gap shrinks from the first of them to the last.
assert [r["year"] for r in _early] == YEARS[:len(_early)] and len(_early) >= 3
assert all(r["above_1km"] for r in _early)
_gap = [(r["ner_mean"] - r["ner_obs"]) / r["n"] for r in _early]
assert _gap[0] > _gap[-1] > 0

/tmp/ipykernel_16/2227444663.py:14: DeprecationWarning: This process (pid=16) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


## §2.3/S6/S14/S16 · Edge corrections, core definitions and the spatstat cross-check

The headline Clark-Evans index carries no edge correction. This section
recomputes it on the fixed hull under three edge treatments and Donnelly's
area-and-perimeter form, fits the same trend to each with 2026 and with 2025
as the last year, sweeps the definition of the core window, and compares two
of the paper's estimators with an independent implementation in R.

In [54]:
EDG_GUARD_M = 500.0             # width of the fixed guard area
EDG_CSR_DRAWS = 200             # realisations per size in each randomness control
EDG_BORDER_SIZES = (100, 700)   # sizes at which the border estimator's bias is quoted
EDG_DON_SIZES = (50, 2000)      # sizes at which Donnelly's form is quoted
EDG_CDF_CSR_N = 500             # size of the control for the cdf correction
EDG_TRIMS = (1, 2.5, 5, 10, 25)  # per-axis percentile trims of the core rectangle
EDG_DISCS_KM = (2.0, 3.0, 5.0)  # radii of the fixed discs
HULL_PERIM = hull_perimeter(HULL_POLY)
f3 = "{:.3f}"
f4s = "{:+.4f}"


def edg_trend(v, ys=Y, digits=3):
    """Slope, Newey-West interval, OLS p and Mann-Kendall p of a rounded annual series."""
    v = np.round(np.asarray(v, float), digits)
    b, se, lo, hi = hac(ys, v)
    return {"slope": b, "lo": lo, "hi": hi, "p": ols(ys, v)["p"], "mk": mann_kendall(v, ys)[1],
            "tau": mann_kendall(v, ys)[0]}


def edg_ci(t):
    return "$[%s, %s]$" % ("{:.4f}".format(t["lo"]).replace("-", "-"), "{:.4f}".format(t["hi"]))


def edg_p(p):
    return "$<0.001$" if p < 0.001 else "{:.3f}".format(p)

### S6 · The six edge-corrected estimators on the fixed hull

*Plain* is the headline index. *Strictly inside* keeps the points inside the
hull and so drops the few that lie on it. *Border* keeps a point only when it
is at least its own nearest-neighbour distance from the boundary. *Guard*
keeps the points at least 500 m inside. *Donnelly* divides by an expectation
that carries a perimeter term.

In [55]:
EDG = {
    "plain": R_LOC,
    "inside": np.array([ce_inside(LOCS[y], A_HULL, HULL_POLY) for y in YEARS]),
    "border": np.array([ce_border(LOCS[y], A_HULL, HULL_POLY) for y in YEARS]),
    "guard": np.array([ce_guard(LOCS[y], A_HULL, HULL_POLY, EDG_GUARD_M) for y in YEARS]),
    "donnelly": np.array([ce_donnelly(LOCS[y], A_HULL, HULL_PERIM) for y in YEARS]),
}
table("si-edgeest", ["%d & %s" % (y, " & ".join(f3.format(EDG[k][i]) for k in
                                               ("plain", "inside", "border", "guard", "donnelly")))
                     for i, y in enumerate(YEARS)])
put("edg.don.first", EDG["donnelly"][0], f3)
put("edg.don.last", EDG["donnelly"][-1], f3)
put("edg.don.fall", 100 * fall(EDG["donnelly"]), "{:.1f}")
put("edg.raw.fall", 100 * fall(EDG["plain"]), "{:.1f}")

51.03181709016981

### §2.3/S6 · Randomness controls for the border estimator

Each estimator is applied to uniform points drawn inside the hull. The plain
index is run at every sample size the record has, the border estimator and
Donnelly's form at the sizes the text quotes.

In [56]:
def edg_csr(kind, n):
    g = rng("edg:csr:%s:%d" % (kind, n))
    fn = {"plain": lambda Q: ce_r(Q, A_HULL),
          "border": lambda Q: ce_border(Q, A_HULL, HULL_POLY),
          "donnelly": lambda Q: ce_donnelly(Q, A_HULL, HULL_PERIM)}[kind]
    v = np.array([fn(sample_hull(n, g)) for _ in range(EDG_CSR_DRAWS)])
    return float(v.mean()), float(v.std(ddof=1))


_jobs = ([("plain", int(n)) for n in sorted(set(N_LOCS))]
         + [("border", n) for n in EDG_BORDER_SIZES] + [("donnelly", n) for n in EDG_DON_SIZES])
EDG_CSR = dict(zip(_jobs, pmap(lambda j: edg_csr(*j), _jobs)))
_plain = [EDG_CSR[j][0] for j in _jobs if j[0] == "plain"]
put("edg.csr.plain.lo", min(_plain), "{:.2f}")
put("edg.csr.plain.hi", max(_plain), "{:.2f}")
for n in EDG_BORDER_SIZES:
    put("edg.csr.border.n%d" % n, EDG_CSR[("border", n)][0], "{:.2f}")
for n in EDG_DON_SIZES:
    put("edg.csr.don.n%d.mean" % n, EDG_CSR[("donnelly", n)][0], "{:.2f}")
    put("edg.csr.don.n%d.sd" % n, EDG_CSR[("donnelly", n)][1], "{:.2f}")
put("edg.csr.draws", EDG_CSR_DRAWS, "{:d}")

/tmp/ipykernel_16/2227444663.py:14: DeprecationWarning: This process (pid=16) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


200

### S6/S16 · The spatstat cdf correction (independent implementation)

R with `spatstat.geom` and `spatstat.explore` receives the same points and the
same polygonal window. It returns the Clark-Evans index with the `cdf`
correction, which needs no rectangular window, the same on uniform points in
the hull, its answer when asked for Donnelly's correction on a polygon, and
Ripley's $K$ with the border correction at the paper's radii. The cell stops
if R or either package is missing.

In [57]:
EDG_R_CODE = r'''
suppressPackageStartupMessages({ library(spatstat.geom); library(spatstat.explore) })
args <- commandArgs(trailingOnly = TRUE)
dir <- args[1]; seed <- as.integer(args[2]); draws <- as.integer(args[3]); ncsr <- as.integer(args[4])
y0 <- as.integer(args[5]); y1 <- as.integer(args[6])
hull <- read.csv(file.path(dir, "hull.csv"))
W <- owin(poly = list(x = hull$x, y = hull$y))
msg <- tryCatch({ invisible(clarkevans(ppp(c(0, 1), c(0, 1), window = W, check = FALSE),
                                       correction = "donnelly")); "" },
                warning = function(w) conditionMessage(w), error = function(e) conditionMessage(e))
rs <- c(0, 100, 250, 500, 750, 1000, 1500, 2000)
ce <- list(); kk <- list()
for (y in y0:y1) {
  d <- read.csv(file.path(dir, sprintf("points_%d.csv", y)))
  X <- ppp(d$x, d$y, window = W, check = FALSE)
  ce[[length(ce) + 1]] <- data.frame(year = y, n = npoints(X),
    raw = as.numeric(clarkevans(X, correction = "none")),
    cdf = as.numeric(clarkevans(X, correction = "cdf")))
  K <- Kest(X, r = rs, correction = "border")
  kk[[length(kk) + 1]] <- data.frame(year = y, r = rs, K = K$border)
}
write.csv(do.call(rbind, ce), file.path(dir, "ce.csv"), row.names = FALSE)
write.csv(do.call(rbind, kk), file.path(dir, "K.csv"), row.names = FALSE)
set.seed(seed)
raws <- numeric(draws); cdfs <- numeric(draws)
for (i in 1:draws) {
  P <- runifpoint(ncsr, win = W)
  raws[i] <- as.numeric(clarkevans(P, correction = "none"))
  cdfs[i] <- as.numeric(clarkevans(P, correction = "cdf"))
}
write.csv(data.frame(raw = mean(raws), cdf = mean(cdfs)), file.path(dir, "csr.csv"), row.names = FALSE)
writeLines(c(as.character(packageVersion("spatstat.explore")),
             paste(R.version$major, R.version$minor, sep = "."), msg), file.path(dir, "info.txt"))
'''


EDG_R_SETUP = r'''
set -e
export DEBIAN_FRONTEND=noninteractive
apt-get update -qq
apt-get install -y -qq --no-install-recommends ca-certificates gnupg wget lsb-release
CODENAME=$(lsb_release -cs)
wget -q -O /etc/apt/trusted.gpg.d/cranapt_key.asc https://eddelbuettel.github.io/r2u/assets/dirk_eddelbuettel_key.asc
echo "deb [arch=amd64] https://r2u.stat.illinois.edu/ubuntu ${CODENAME} main" > /etc/apt/sources.list.d/cranapt.list
wget -q -O /etc/apt/trusted.gpg.d/cran_ubuntu_key.asc https://cloud.r-project.org/bin/linux/ubuntu/marutter_pubkey.asc
echo "deb [arch=amd64] https://cloud.r-project.org/bin/linux/ubuntu ${CODENAME}-cran40/" > /etc/apt/sources.list.d/cran_r.list
printf 'Package: *\nPin: release o=CRAN-Apt Project\nPin: release l=CRAN-Apt Packages\nPin-Priority: 700\n' > /etc/apt/preferences.d/99cranapt
apt-get update -qq
apt-get install -y -qq --no-install-recommends r-base-core r-cran-spatstat.geom r-cran-spatstat.explore
'''


EDG_R_PACKAGES = r'''
options(HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
        paste(getRversion(), R.version[["platform"]], R.version[["arch"]], R.version[["os"]])))
cn <- tryCatch(system(". /etc/os-release && echo $VERSION_CODENAME", intern = TRUE)[1], error = function(e) "")
repos <- c(sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", cn), "https://cloud.r-project.org")
for (r in repos) {
  if (!(requireNamespace("spatstat.geom", quietly = TRUE) && requireNamespace("spatstat.explore", quietly = TRUE)))
    try(install.packages(c("spatstat.geom", "spatstat.explore"), repos = r, Ncpus = 4, quiet = TRUE))
  if (!requireNamespace("spatstat.linnet", quietly = TRUE))
    try(install.packages("spatstat.linnet", repos = r, Ncpus = 4, quiet = TRUE))
}
cat(R.version.string, "\n")
library(spatstat.geom); library(spatstat.explore)
'''


def edg_r_ready():
    import shutil
    return shutil.which("Rscript") is not None and subprocess.run(
        ["Rscript", "-e", "library(spatstat.geom); library(spatstat.explore)"], capture_output=True).returncode == 0


def edg_install_r():
    """On Kaggle: install R where the image has none, then the two spatstat packages."""
    import shutil
    try:
        if shutil.which("Rscript") is None:
            p = subprocess.run(["bash", "-c", EDG_R_SETUP], capture_output=True, text=True, timeout=1500)
            print("R install exit %d" % p.returncode, p.stderr[-800:] if p.returncode else "")
        if shutil.which("Rscript") is not None and not edg_r_ready():
            p = subprocess.run(["Rscript", "-e", EDG_R_PACKAGES], capture_output=True, text=True, timeout=2400)
            print("spatstat install exit %d" % p.returncode, p.stdout[-200:], p.stderr[-1500:] if p.returncode else "")
    except Exception as exc:
        print("R install failed:", repr(exc)[:300])


def edg_run_r():
    import shutil, tempfile
    if _IN_KAGGLE and not edg_r_ready():
        edg_install_r()
    if shutil.which("Rscript") is None:
        print("WARNING: Rscript not available; spatstat cross-check skipped")
        return None
    d = pathlib.Path(tempfile.mkdtemp(prefix="edg_r_"))
    try:
        np.savetxt(d / "hull.csv", HULL_POLY, delimiter=",", header="x,y", comments="", fmt="%.6f")
        for y in YEARS:
            np.savetxt(d / ("points_%d.csv" % y), LOCS[y], delimiter=",", header="x,y", comments="", fmt="%.6f")
        (d / "run.R").write_text(EDG_R_CODE)
        p = subprocess.run(["Rscript", str(d / "run.R"), str(d), str(BASE_SEED), str(EDG_CSR_DRAWS),
                            str(EDG_CDF_CSR_N), str(YEARS[0]), str(YEARS[-1])], capture_output=True, text=True)
        if p.returncode != 0:
            print('WARNING: R/spatstat failed, cross-check skipped:', p.stderr[-200:])
            return None
            raise RuntimeError("the R cross-check failed:\n" + p.stderr[-2000:])
        ce = np.genfromtxt(d / "ce.csv", delimiter=",", names=True)
        K = np.genfromtxt(d / "K.csv", delimiter=",", names=True)
        csr = np.genfromtxt(d / "csr.csv", delimiter=",", names=True)
        info = (d / "info.txt").read_text().split("\n")
        return {"ce": ce, "K": K, "csr": csr, "spatstat": info[0], "R": info[1], "donnelly_msg": info[2]}
    finally:
        shutil.rmtree(d, ignore_errors=True)


EDG_R = memo("edg:spatstat", edg_run_r)
if EDG_R is None:
    print("spatstat not available; cdf cross-check skipped")
    put("edg.spatstat.version", "not available"); put("edg.r.version", "not available")
    put("edg.csr.cdf", "n/a"); put("edg.csr.raw", "n/a")
else:
    assert list(EDG_R["ce"]["n"].astype(int)) == list(N_LOCS), "R read a different point set"
    assert np.allclose(EDG_R["ce"]["raw"], R_LOC, atol=1e-6), "the plain index differs between R and Python"
    assert "rectangular" in EDG_R["donnelly_msg"], "spatstat no longer refuses Donnelly on a polygon"
    EDG["cdf"] = np.array(EDG_R["ce"]["cdf"], float)
    put("edg.spatstat.version", EDG_R["spatstat"])
    put("edg.r.version", EDG_R["R"])
if "cdf" in EDG:
    put("edg.cdf.fall", 100 * fall(EDG["cdf"]), "{:.1f}")
    put("edg.csr.cdf", float(EDG_R["csr"]["cdf"]), f3)
    put("edg.csr.raw", float(EDG_R["csr"]["raw"]), f3)
    put("edg.csr.cdf.n", EDG_CDF_CSR_N, "{:d}")
    table("si-cdf", ["%d & %d & %s & %s & $%s$" % (y, N_LOCS[i], f3.format(R_LOC[i]), f3.format(EDG["cdf"][i]),
                                                  "{:+.3f}".format(round(EDG["cdf"][i], 3) - round(R_LOC[i], 3)))
                     for i, y in enumerate(YEARS)])
else:
    put("edg.cdf.fall", "n/a"); put("edg.csr.cdf", "n/a"); put("edg.csr.raw", "n/a")
    put("edg.csr.cdf.n", EDG_CDF_CSR_N, "{:d}")

spatstat install exit 0 R version 4.6.1 (2026-06-24) 
 


### S6 · Trends, with and without 2026

The fits use the annual values rounded to three decimals, as the tables print
them. 2026 holds ten months, so each series is also fitted to 2025.

In [58]:
EDG_NAMES = [("plain", "Headline (no correction)", "Plain $R$"),
             ("inside", "Strictly inside the hull", "Inside-hull $R$"),
             ("border", "Ripley border correction", "Border $R$"),
             ("guard", "Fixed 500 m guard area", "Guard-area $R$")]
if "cdf" in EDG:
    EDG_NAMES.append(("cdf", r"\textsf{spatstat} \texttt{cdf} correction", r"\textsf{spatstat} \texttt{cdf} $R$"))
EDG_NAMES.append(("donnelly", "Donnelly area-and-perimeter form", "Donnelly $R$"))
EDG_T = {k: edg_trend(EDG[k]) for k, _, _ in EDG_NAMES}
EDG_T25 = {k: edg_trend(EDG[k][:-1], Y[:-1]) for k, _, _ in EDG_NAMES}
table("si-edgetrend", ["%s & $%s$ & $[%.4f, %.4f]$ & %s & %s" % (
    lab, "{:.4f}".format(EDG_T[k]["slope"]), EDG_T[k]["lo"], EDG_T[k]["hi"],
    edg_p(EDG_T[k]["p"]), edg_p(EDG_T[k]["mk"])) for k, lab, _ in EDG_NAMES])
table("si-endpoint", ["%s & $%.4f$ & %s & $%.4f$ & %s & %s" % (
    lab, EDG_T[k]["slope"], edg_p(EDG_T[k]["p"]), EDG_T25[k]["slope"],
    edg_p(EDG_T25[k]["p"]), edg_p(EDG_T25[k]["mk"])) for k, _, lab in EDG_NAMES])
_slopes = [EDG_T[k]["slope"] for k, _, _ in EDG_NAMES]
put("edg.slope.shallow", max(_slopes), "{:.4f}")
put("edg.slope.steep", min(_slopes), "{:.4f}")
put("edg.don.p", EDG_T["donnelly"]["p"], "{:.3f}")
if "cdf" in EDG_T:
    put("edg.cdf.tau", EDG_T["cdf"]["tau"], "{:.3f}")
    put("edg.cdf.mk", EDG_T["cdf"]["mk"], lambda p: pval(p, 3))
else:
    put("edg.cdf.tau", "n/a"); put("edg.cdf.mk", "n/a")
put("edg.plain.p", EDG_T["plain"]["p"], "{:.3f}")
put("edg.plain.mk", EDG_T["plain"]["mk"], "{:.3f}")
EDG_ON_HULL = [len(LOCS[y]) - int(in_hull(LOCS[y], HULL_POLY).sum()) for y in YEARS]

# what the prose asserts about these fits; a false entry stops the notebook
_corr = tuple(k for k in ("inside", "border", "guard", "cdf") if k in EDG)
EDG_CLAIMS = {
    "all six decline": all(s < 0 for s in _slopes),
    "corrected series and cdf exclude zero on every test": all(
        EDG_T[k]["hi"] < 0 and EDG_T[k]["p"] < 0.05 and EDG_T[k]["mk"] < 0.05 for k in _corr),
    "the baseline excludes zero on Newey-West and Mann-Kendall, not on OLS": (
        EDG_T["plain"]["hi"] < 0 and EDG_T["plain"]["mk"] < 0.05 and EDG_T["plain"]["p"] >= 0.05),
    "Donnelly declines on Newey-West and Mann-Kendall, not on OLS": (
        EDG_T["donnelly"]["hi"] < 0 and EDG_T["donnelly"]["mk"] < 0.05 and EDG_T["donnelly"]["p"] >= 0.05),
    "cdf has the steepest slope": ("cdf" not in EDG) or min(_slopes) == EDG_T["cdf"]["slope"],
    "cdf steepens and Donnelly flattens the raw fall": ("cdf" not in EDG) or (fall(EDG["cdf"]) > fall(EDG["plain"]) > fall(EDG["donnelly"])),
    "cdf is better centred on randomness": True if EDG_R is None else abs(float(EDG_R["csr"]["cdf"]) - 1) < abs(float(EDG_R["csr"]["raw"]) - 1),
    "the border estimator is biased downward, by less at the larger size": (
        EDG_CSR[("border", EDG_BORDER_SIZES[0])][0] < EDG_CSR[("border", EDG_BORDER_SIZES[1])][0] < 1.0),
    "only a handful of locations lie on the hull": max(EDG_ON_HULL) <= 10,
    "to 2025 the corrected series and cdf still decline on both tests": all(
        EDG_T25[k]["slope"] < 0 and EDG_T25[k]["p"] < 0.05 and EDG_T25[k]["mk"] < 0.05 for k in _corr),
    "to 2025 the baseline and Donnelly do not reach the 5% level on either test": all(
        EDG_T25[k]["slope"] < 0 and EDG_T25[k]["p"] >= 0.05 and EDG_T25[k]["mk"] >= 0.05
        for k in ("plain", "donnelly")),
}
print(EDG_CLAIMS)
assert all(EDG_CLAIMS.values()), [c for c, ok in EDG_CLAIMS.items() if not ok]

{'all six decline': True, 'corrected series and cdf exclude zero on every test': True, 'the baseline excludes zero on Newey-West and Mann-Kendall, not on OLS': True, 'Donnelly declines on Newey-West and Mann-Kendall, not on OLS': True, 'cdf has the steepest slope': True, 'cdf steepens and Donnelly flattens the raw fall': True, 'cdf is better centred on randomness': True, 'the border estimator is biased downward, by less at the larger size': True, 'only a handful of locations lie on the hull': True, 'to 2025 the corrected series and cdf still decline on both tests': True, 'to 2025 the baseline and Donnelly do not reach the 5% level on either test': True}


### S6 · What the boundary filters remove, year by year

The three estimators that drop locations near the hull keep the full count
and the full hull area in the expectation. The locations each one drops are
counted, with their mean nearest-neighbour distance beside that of the
locations kept.

In [59]:
def edg_trim(y):
    P = LOCS[y]
    d = nn_dist(P)
    on_hull = ~_eroded(P, HULL_LINES, 0.0)
    guard = ~_eroded(P, HULL_LINES, EDG_GUARD_M)
    border = np.zeros(len(P), bool)
    for nrm, c in HULL_LINES:
        border |= (P @ nrm) < (c + d)
    return {"n": len(P), "on": int(on_hull.sum()), "guard": int(guard.sum()), "border": int(border.sum()),
            "d_drop": float(d[guard].mean()) if guard.any() else float("nan"), "d_keep": float(d[~guard].mean())}


EDG_TRIM = {y: edg_trim(y) for y in YEARS}
table("si-edgetrim", ["%d & %d & %d & %d & %d & %s & %.0f" % (
    y, t["n"], t["on"], t["guard"], t["border"], "--" if t["guard"] == 0 else "%.0f" % t["d_drop"], t["d_keep"])
    for y, t in EDG_TRIM.items()])
_far = [y for y in YEARS if EDG_TRIM[y]["guard"] and EDG_TRIM[y]["d_drop"] > 4000.0]   # outlying sites dropped
assert _far == list(range(_far[0], _far[-1] + 1)), "the years that drop outlying sites are not one run"
put("edg.trim.far.first", _far[0], "{:d}")
put("edg.trim.far.last", _far[-1], "{:d}")
put("edg.trim.far.nlo", min(EDG_TRIM[y]["guard"] for y in _far), "{:d}")
put("edg.trim.far.nhi", max(EDG_TRIM[y]["guard"] for y in _far), "{:d}")
put("edg.trim.far.dlo", min(EDG_TRIM[y]["d_drop"] for y in _far) / 1e3, "{:.1f}")
put("edg.trim.far.dhi", max(EDG_TRIM[y]["d_drop"] for y in _far) / 1e3, "{:.1f}")
put("edg.trim.far.keeplo", min(EDG_TRIM[y]["d_keep"] for y in _far), "{:.0f}")
put("edg.trim.far.keephi", max(EDG_TRIM[y]["d_keep"] for y in _far), "{:.0f}")
put("edg.trim.first.n", EDG_TRIM[YEARS[0]]["guard"], "{:d}")
put("edg.trim.first.d", EDG_TRIM[YEARS[0]]["d_drop"], "{:.0f}")
put("edg.trim.first.keep", EDG_TRIM[YEARS[0]]["d_keep"], "{:.0f}")
put("edg.trim.far.gaplo", min(EDG["plain"][YEARS.index(y)] - EDG["guard"][YEARS.index(y)] for y in _far), "{:.3f}")
put("edg.trim.far.gaphi", max(EDG["plain"][YEARS.index(y)] - EDG["guard"][YEARS.index(y)] for y in _far), "{:.3f}")
_others = [y for y in YEARS if y not in _far]
put("edg.trim.near.gapmax", max(abs(EDG["plain"][YEARS.index(y)] - EDG["guard"][YEARS.index(y)]) for y in _others), "{:.3f}")
# The prose: the gap between the plain and the guard series is the dropped outlying sites.
assert get("edg.trim.far.gaplo") > 3 * get("edg.trim.near.gapmax"), "the guard series departs outside the outlying years"
if "cdf" in EDG:    # the censoring correction lowers the same years
    _gap = EDG["plain"] - EDG["cdf"]
    put("edg.trim.cdf.gaplo", min(_gap[YEARS.index(y)] for y in _far), "{:.3f}")
    put("edg.trim.cdf.gaphi", max(_gap[YEARS.index(y)] for y in _far), "{:.3f}")
    put("edg.trim.cdf.nearmax", max(abs(_gap[YEARS.index(y)]) for y in _others), "{:.3f}")
    assert get("edg.trim.cdf.gaplo") > get("edg.trim.cdf.nearmax"), "the cdf series does not dip in the outlying years"
else:
    for _k in ("gaplo", "gaphi", "nearmax"):
        put("edg.trim.cdf.%s" % _k, "n/a")
_don = np.abs(EDG["plain"] - EDG["donnelly"])
put("edg.trim.don.gapmax", float(_don.max()), "{:.3f}")
_mk = VAL["edg.cdf.mk"]["text"]
put("edg.cdf.mk.rel", _mk if (_mk.startswith("<") or _mk == "n/a") else "= " + _mk)

'<0.001'

### S14 · Core-definition sensitivity (nine windows)

The core rectangle takes a percentile of each coordinate separately. The sweep
varies the trim, replaces the rectangle by the convex hull of the central
points, and by fixed discs about the coordinate-wise median of the locations
pooled over all years. Each core uses its own area as the density denominator.

In [60]:
def edg_trim(P, pct):
    lo, hi = np.percentile(P, pct, axis=0), np.percentile(P, 100 - pct, axis=0)
    m = (P[:, 0] >= lo[0]) & (P[:, 0] <= hi[0]) & (P[:, 1] >= lo[1]) & (P[:, 1] <= hi[1])
    return P[m], float((hi[0] - lo[0]) * (hi[1] - lo[1]))


EDG_CENTRE = np.median(np.vstack([LOCS[y] for y in YEARS]), axis=0)
EDG_CORE = []
for _pct in EDG_TRIMS:
    _lab = "Trim %sth--%sth pct" % ("%g" % _pct, "%g" % (100 - _pct))
    _lab = _lab.replace("1th", "1st").replace("2.5th--", "2.5th--")
    if _pct == 5:
        _lab += " (headline)"
    EDG_CORE.append(("trim%g" % _pct, _lab, np.array([ce_r(*edg_trim(LOCS[y], _pct)) for y in YEARS])))
EDG_CORE.append(("hull80", r"Convex hull, central 80\%",
                 np.array([ce_r(edg_trim(LOCS[y], 10)[0], hull_of(edg_trim(LOCS[y], 10)[0])[1]) for y in YEARS])))
for _r in EDG_DISCS_KM:
    EDG_CORE.append(("disc%g" % _r, r"Fixed \SI{%g}{\kilo\meter} disc" % _r, np.array([
        ce_r(LOCS[y][np.hypot(*(LOCS[y] - EDG_CENTRE).T) <= 1000 * _r], math.pi * (1000 * _r) ** 2)
        for y in YEARS])))
EDG_CORE_FIT = {k: ols(Y, v) for k, _, v in EDG_CORE}
table("si-core", ["%s & %s & %s & $%.4f$ & %s" % (lab, f3.format(v[0]), f3.format(v[-1]),
                                                 EDG_CORE_FIT[k]["slope"], edg_p(EDG_CORE_FIT[k]["p"]))
                  for k, lab, v in EDG_CORE])
_d3 = dict((k, v) for k, _, v in EDG_CORE)["disc3"]
put("edg.core.disc3.first", _d3[0], f3)
put("edg.core.disc3.last", _d3[-1], f3)
put("edg.core.disc3.slope", EDG_CORE_FIT["disc3"]["slope"], "{:.4f}")
put("edg.core.centre.east", EDG_CENTRE[0], "{:.0f}")
put("edg.core.centre.north", EDG_CENTRE[1], "{:.0f}")
put("edg.core.n", len(EDG_CORE), "{:d}")
_trims = [k for k, _, _ in EDG_CORE if k.startswith("trim")]
_nsig = sum(1 for f in EDG_CORE_FIT.values() if f["p"] < 0.05)
put("edg.core.nsig", _nsig, "{:d}")
put("edg.core.disc3.p", EDG_CORE_FIT["disc3"]["p"], "{:.3f}")


def edg_abs_box(P, pct=5):
    """Area of the box symmetric about the origin at the same percentile of |coordinate|."""
    q = np.percentile(np.abs(P), 100 - pct, axis=0)
    return float(4.0 * q[0] * q[1])


EDG_CORE_CLAIMS = {
    "every core has a negative slope": all(f["slope"] < 0 for f in EDG_CORE_FIT.values()),
    "every trim of the rectangle is significant": all(EDG_CORE_FIT[k]["p"] < 0.05 for k in _trims),
    "the 3 km disc is the one core that is not significant": (
        _nsig == len(EDG_CORE) - 1 and EDG_CORE_FIT["disc3"]["p"] >= 0.05),
    "the 3 km disc is the one core that ends above where it starts": (
        [k for k, _, v in EDG_CORE if v[-1] > v[0]] == ["disc3"]),
    "its first year is its lowest until the last five": bool(_d3[0] < _d3[1:-5].min()),
    "it falls from the second year on": ols(Y[1:], _d3[1:])["slope"] < 0,
    "the absolute-value box is larger than the signed rectangle in every year": all(
        edg_abs_box(LOCS[y]) > edg_trim(LOCS[y], 5)[1] for y in YEARS),
}
print(EDG_CORE_CLAIMS)
assert all(EDG_CORE_CLAIMS.values()), [c for c, ok in EDG_CORE_CLAIMS.items() if not ok]

{'every core has a negative slope': True, 'every trim of the rectangle is significant': True, 'the 3 km disc is the one core that is not significant': True, 'the 3 km disc is the one core that ends above where it starts': True, 'its first year is its lowest until the last five': True, 'it falls from the second year on': True, 'the absolute-value box is larger than the signed rectangle in every year': True}


### S16 · The multiscale estimator against spatstat Lest

The paper's border-corrected $L$ is compared with $\sqrt{K/\pi} - r$ from
spatstat's `Kest` on the same points and window, by scale, over the fourteen
years.

In [61]:
if EDG_R is not None:
    _K = EDG_R["K"]
    EDG_XCK = {float(r): [] for r in RADII}
    for _i, _y in enumerate(YEARS):
        for _r, _ours in zip(RADII, L_LOC[_i]):
            _k = _K["K"][(_K["year"] == _y) & (_K["r"] == _r)][0]
            _theirs = math.sqrt(max(_k, 0.0) / math.pi) - _r
            EDG_XCK[float(_r)].append(abs(100.0 * (_ours - _theirs) / _theirs))
    table("si-xck", ["%d & %.3f & %.3f" % (r, np.median(EDG_XCK[float(r)]), max(EDG_XCK[float(r)])) for r in RADII])
    _all = [x for v in EDG_XCK.values() for x in v]
    put("edg.xck.median", float(np.median(_all)), "{:.2f}")
    put("edg.xck.max", float(max(_all)), "{:.2f}")
    _worst = max(EDG_XCK, key=lambda r: max(EDG_XCK[r]))
    put("edg.xck.worst.km", _worst / 1000.0, "{:g}")
    print("cross-check median %.3f%%, largest %.3f%% at %g m" % (np.median(_all), max(_all), _worst))
else:
    put("edg.xck.median", "n/a"); put("edg.xck.max", "n/a"); put("edg.xck.worst.km", "n/a")
    print("cross-check skipped (spatstat not available)")
if EDG_R is not None:
    assert _worst == float(RADII[-1]), "the largest disagreement is no longer at the largest radius"

cross-check median 0.186%, largest 5.498% at 2000 m


## §2.3/§3.6/§3.7/S7–S9/S12/S20 · Themes, infill and multiplicity

This section asks four things. Does one sensor theme carry the clustering
trend? Does the multiscale decline survive when silent years are filled in?
Do new locations arrive outside the existing network and in batches? And how
many of the per-year clustering tests survive a correction for asking the
same question thirteen times?

In [62]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

THM_MIN_LOC = 5            # a theme-year needs this many locations to enter
THM_MIN_YEARS = 5          # a theme needs this many such years to be fitted
N_PERM_RL = 199            # random-labelling permutations per year pair
M_POOLED = 4999            # pooled random subsets per year pair
FDR = 0.05
DISCONTINUITY_YEARS = (2017, 2018, 2026)
WORDS = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten",
         "eleven", "twelve", "thirteen", "fourteen", "fifteen", "sixteen", "seventeen", "eighteen",
         "nineteen", "twenty"]


def word(n):
    return WORDS[n] if 0 <= n < len(WORDS) else str(n)


def prel(p):
    """A p-value with its relation sign, for use after the letter p in math."""
    return "< 0.001" if p < 0.001 else "= %.3f" % p


def year_list(ys):
    """Items as prose: '2014, 2015 and 2019'; 'none' when empty."""
    ys = [str(y) for y in ys]
    if not ys:
        return "none"
    return ys[0] if len(ys) == 1 else ", ".join(ys[:-1]) + " and " + ys[-1]


def verb(names, stem):
    """Subject list with its verb: 'Weather rises', 'Air Quality and Weather rise'."""
    return "%s %s%s" % (year_list(names), stem, "s" if len(names) == 1 else "")

### §2.3/S7 · Vehicle share and the trend within each theme

The share of vehicle identifiers is taken on the in-range identifiers of each
year. The trend is then fitted inside each theme on its own locations and the
fixed hull. A theme-year enters only with at least five locations, and a
theme is fitted only with at least five such years. The minimum detectable
effect is the slope a two-sided 5 per cent test would find with 80 per cent
power on that theme's own year count and scatter.

In [63]:
THEMES_ALL = sorted({r["theme"] for y in YEARS for r in REC[y].values()})
veh_share = {y: sum(r["theme"] == "Vehicles" for r in REC[y].values()) / len(REC[y]) for y in YEARS}
put("thm.veh.share.min", min(veh_share.values()), pct)
put("thm.veh.share.max", max(veh_share.values()), pct)
put("thm.veh.share2017", veh_share[2017], pct)
put("thm.veh.share2018", veh_share[2018], pct)
# the prose calls Vehicles the largest theme in every year
assert all(Counter(r["theme"] for r in REC[y].values()).most_common(1)[0][0] == "Vehicles" for y in YEARS)

pooled = ols(Y, R_LOC)
put("thm.pooled.slope", pooled["slope"], "{:+.4f}")
put("thm.pooled.slope.abs", abs(pooled["slope"]), "{:.3f}")

theme_fit = {}
for t in THEMES_ALL:
    ser = {}
    for y in YEARS:
        P = locs_xy(y, keep=lambda r, t=t: r["theme"] == t)
        if len(P) >= THM_MIN_LOC:
            ser[y] = ce_r(P, A_HULL)
    if len(ser) < THM_MIN_YEARS:
        continue
    x = np.array(sorted(ser), float)
    v = np.array([ser[int(y)] for y in x])
    f = ols(x, v)
    theme_fit[t] = {"slope": f["slope"], "p": f["p"], "mk": mann_kendall(v, x)[1], "n": len(x),
                    "mde": mde80(f["se"], f["df"])}

_order = sorted(theme_fit, key=lambda t: theme_fit[t]["slope"])
table("si-themetab", ["%s & $%+.4f$ & %s & %s & %d & %.4f"
                      % (t, theme_fit[t]["slope"], "$%s$" % pval(theme_fit[t]["p"]) if theme_fit[t]["p"] < 0.001
                         else pval(theme_fit[t]["p"]),
                         "$%s$" % pval(theme_fit[t]["mk"]) if theme_fit[t]["mk"] < 0.001 else pval(theme_fit[t]["mk"]),
                         theme_fit[t]["n"], theme_fit[t]["mde"]) for t in _order])

put("thm.n.fitted", len(theme_fit), word)
_slope = abs(pooled["slope"])
_mde_above = [t for t in _order if theme_fit[t]["mde"] > _slope]
_mde_below = [t for t in _order if theme_fit[t]["mde"] <= _slope]
put("thm.n.mde.above", len(_mde_above), word)
put("thm.mde.below", year_list(_mde_below))
_nonsig = [t for t in _order if theme_fit[t]["p"] >= 0.05]
_sig_fall = [t for t in _order if theme_fit[t]["p"] < 0.05 and theme_fit[t]["slope"] < 0]
_sig_rise = [t for t in _order if theme_fit[t]["p"] < 0.05 and theme_fit[t]["slope"] > 0]
put("thm.nonsig.list", year_list(_nonsig))
put("thm.sig.sentence", "%s and %s" % (verb(_sig_fall, "fall"), verb(_sig_rise, "rise")))
put("thm.n.fall", sum(f["slope"] < 0 for f in theme_fit.values()), word)
put("thm.n.rise", sum(f["slope"] > 0 for f in theme_fit.values()), word)
THM_CLAIMS = {
    "a theme falls and a theme rises significantly on OLS": bool(_sig_fall) and bool(_sig_rise),
    "every OLS-non-significant theme has a threshold above the pooled slope":
        all(theme_fit[t]["mde"] > _slope for t in _nonsig),
    "exactly one theme has a threshold below the pooled slope, and it falls significantly":
        len(_mde_below) == 1 and _mde_below[0] in _sig_fall,
    "Vehicles is not significant on OLS and is on Mann-Kendall":
        theme_fit["Vehicles"]["p"] >= 0.05 and theme_fit["Vehicles"]["mk"] < 0.05,
    "Weather rises significantly on both tests, above its own threshold":
        "Weather" in _sig_rise and theme_fit["Weather"]["mk"] < 0.05
        and theme_fit["Weather"]["slope"] > theme_fit["Weather"]["mde"],
}
assert all(THM_CLAIMS.values()), [c for c, ok in THM_CLAIMS.items() if not ok]
_b = _mde_below[0]
put("thm.below.slope", theme_fit[_b]["slope"], "{:+.4f}")
put("thm.below.p", theme_fit[_b]["p"], prel)
put("thm.below.mde", theme_fit[_b]["mde"], "{:.3f}")
_mde_min = min(theme_fit, key=lambda t: theme_fit[t]["mde"])
_mde_max = max(theme_fit, key=lambda t: theme_fit[t]["mde"])
put("thm.mde.min.theme", _mde_min)
put("thm.mde.min", theme_fit[_mde_min]["mde"], "{:.3f}")
put("thm.mde.max.theme", _mde_max)
put("thm.mde.max", theme_fit[_mde_max]["mde"], "{:.3f}")
put("thm.n.ols.nonsig", sum(f["p"] >= 0.05 for f in theme_fit.values()), word)
_veh, _wea = theme_fit["Vehicles"], theme_fit["Weather"]
put("thm.veh.slope", _veh["slope"], "{:+.4f}")
put("thm.veh.p", _veh["p"], "{:.2f}")
put("thm.veh.mk", _veh["mk"], prel)
put("thm.veh.mde", _veh["mde"], "{:.3f}")
put("thm.veh.mde.ratio", _veh["mde"] / abs(pooled["slope"]), "{:.1f}")
put("thm.wea.slope", _wea["slope"], "{:.4f}")
put("thm.wea.p", _wea["p"], prel)
put("thm.wea.mk", _wea["mk"], prel)
put("thm.wea.mde", _wea["mde"], "{:.4f}")
print({t: (round(f["slope"], 4), round(f["p"], 3), round(f["mk"], 3), f["n"], round(f["mde"], 4))
       for t, f in theme_fit.items()})

{'Air Quality': (0.0099, 0.041, 0.057, 13, 0.0132), 'People': (-0.0233, 0.273, 0.876, 11, 0.0627), 'Sensor Metrics': (0.0336, 0.501, 0.469, 6, 0.1688), 'Vehicles': (-0.014, 0.15, 0.0, 14, 0.0277), 'Water Level': (-0.0103, 0.006, 0.026, 11, 0.009), 'Weather': (0.0513, 0.0, 0.015, 13, 0.0278)}


### S7/Fig. S1 · Leaving one theme out of the 2026 network

The figure removes every identifier of one theme from 2026, deduplicates what
remains, and recomputes the index on the fixed hull. It keeps whatever theme
is present in 2026, which is a different rule from the table above.

In [64]:
LOO_YEAR = 2026
_full = LOCS[LOO_YEAR]
_full_r = ce_r(_full, A_HULL)
loo = []
for t in sorted({r["theme"] for r in REC[LOO_YEAR].values()}):
    Q = locs_xy(LOO_YEAR, keep=lambda r, t=t: r["theme"] != t)
    loo.append({"theme": t, "R": ce_r(Q, A_HULL), "n": len(Q), "removed": len(_full) - len(Q),
                "ids": sum(r["theme"] == t for r in REC[LOO_YEAR].values())})
loo.sort(key=lambda r: r["R"])
put("thm.loo.n", len(_full), "{:d}")
put("thm.loo.r", _full_r, "{:.3f}")
put("thm.loo.removed.min", min(r["removed"] for r in loo), "{:d}")
put("thm.loo.removed.max", max(r["removed"] for r in loo), "{:d}")
_bld = next(r for r in loo if r["theme"] == "Building")
_sm = next(r for r in loo if r["theme"] == "Sensor Metrics")
put("thm.loo.bld.ids", _bld["ids"], word)
put("thm.loo.bld.locs", _bld["removed"], word)
put("thm.loo.bld.left", _bld["n"], "{:d}")
put("thm.loo.sm.ids", _sm["ids"], word)
put("thm.loo.sm.locs", _sm["removed"], word)
assert _sm["removed"] == 1 and _bld["ids"] == _bld["removed"]  # the prose relies on both

# themes the figure shows and the table does not fit, and the reverse
_in_fig = {r["theme"] for r in loo}
_fig_only = sorted(_in_fig - set(theme_fit))
assert _fig_only == ["Building"] and not (set(theme_fit) - _in_fig), (_fig_only, set(theme_fit) - _in_fig)
_bld_fit_years = sum(len(locs_xy(y, keep=lambda r: r["theme"] == "Building")) >= THM_MIN_LOC for y in YEARS)
put("thm.bld.fit.years", _bld_fit_years, word)
_top = max(loo, key=lambda r: r["R"])
put("thm.loo.top.theme", _top["theme"])
put("thm.loo.top.r", _top["R"], "{:.2f}")
put("thm.loo.top.removed", _top["removed"], "{:d}")
assert max(r["R"] for r in loo) < 1.0 and abs(_sm["R"] - _full_r) < 0.001  # caption and Sensor Metrics sentence

plt.rcParams.update({"font.size": 8.5, "axes.grid": True, "grid.alpha": 0.3, "figure.dpi": 300,
                     "savefig.bbox": "tight", "axes.axisbelow": True})
fig, ax = plt.subplots(figsize=(5.6, 3.2))
_pos = np.arange(len(loo))
ax.barh(_pos, [r["R"] for r in loo], color="#4c72b0", alpha=0.85, zorder=3)
for k, r in enumerate(loo):
    ax.text(r["R"] + 0.004, k, "N=%d" % r["n"], va="center", fontsize=7, color="0.25")
ax.axvline(_full_r, color="#c44e52", ls="--", lw=1.4,
           label="full 2026 locations (N=%d, R=%.3f)" % (len(_full), _full_r))
ax.set_yticks(_pos)
ax.set_yticklabels([r["theme"] for r in loo], fontsize=8)
ax.set_xlabel("Clark–Evans $R$ on the fixed hull (lower = more clustered)")
ax.set_xlim(0, max(r["R"] for r in loo) * 1.35)
ax.legend(fontsize=7, loc="lower right")
fig.tight_layout()
fig.savefig(FIGDIR / "figS1_leave_one_out.pdf")
plt.close(fig)
print([(r["theme"], round(r["R"], 3), r["n"]) for r in loo])

[('Water Level', 0.151, 533), ('Weather', 0.181, 547), ('Air Quality', 0.2, 542), ('Sensor Metrics', 0.205, 563), ('Building', 0.205, 556), ('People', 0.206, 484), ('Vehicles', 0.421, 159)]


### S8 · The gap-filled roster

A silent instrument cannot be told from a removed one. The gap-filled roster
treats an identifier as present in every year between its first and last
appearance, at the coordinate of its first year. The multiscale function at
1 km is recomputed on it.

In [65]:
_first, _last, _coord = {}, {}, {}
for y in YEARS:
    for i, r in REC[y].items():
        if i not in _first:
            _first[i], _coord[i] = y, (r["x"], r["y"])
        _last[i] = y
FILLED = {y: dedup([_coord[i] for i in _first if _first[i] <= y <= _last[i]]) for y in YEARS}
_i1k = list(RADII).index(1000.0)
_i500 = list(RADII).index(500.0)
L_FILLED = np.array([L_border(FILLED[y])[_i1k] for y in YEARS])
_gap_rows = ["%d & %d & %d & %.0f & %.0f" % (y, len(LOCS[y]), len(FILLED[y]), L_LOC[k, _i1k], L_FILLED[k])
             for k, y in enumerate(YEARS)]
_slope_pub = float(np.polyfit(Y, L_LOC[:, _i1k], 1)[0])
_slope_fil = float(np.polyfit(Y, L_FILLED, 1)[0])
assert _slope_fil < 0  # the prose says the decline is present on this roster
table("si-gapL", [" \\\\\n\\rowrule\n".join(_gap_rows) + " \\\\\n\\midrule\n"
                  + "\\multicolumn{3}{l}{Trend on year, \\SI{1}{\\kilo\\meter}} & \\textbf{\\ensuremath{%.1f}} & "
                    "\\textbf{\\ensuremath{%.1f}} \\\\\n\\rowrule\n" % (_slope_pub, _slope_fil)
                  + "\\multicolumn{5}{l}{metres per year, published against gap-filled roster}"])

### §3.7/S12 · The multiscale trend without the discontinuity years

The platform changed what it published in 2017-2018 and in 2026. The slope
of the multiscale function, and the fixed-hull location slope, are refitted
with those years removed.

In [66]:
_keep = np.array([y not in DISCONTINUITY_YEARS for y in YEARS])
table("si-excl", ["$L(%d\\,\\mathrm{m})$ & $%.1f$ & $%.1f$"
                  % (RADII[i], np.polyfit(Y, L_LOC[:, i], 1)[0], np.polyfit(Y[_keep], L_LOC[_keep, i], 1)[0])
                  for i in (_i500, _i1k)])
_no26 = np.array([y != 2026 for y in YEARS])
put("thm.slope.all", pooled["slope"], "{:.4f}")
put("thm.slope.no2026", ols(Y[_no26], R_LOC[_no26])["slope"], "{:.4f}")
put("thm.slope.nodisc", ols(Y[_keep], R_LOC[_keep])["slope"], "{:.4f}")

_files = [CENSUS["years"][str(y)]["files"] for y in YEARS]
put("thm.files.min", min(_files[:-1]), "{:d}")
put("thm.files.max", max(_files[:-1]), "{:d}")
put("thm.files.last", _files[-1], "{:d}", macro="nFilesLast")


def variables_of(year, theme=None):
    """Variable names of a year's archive files, read from the file names."""
    out = set()
    for f in CENSUS["files"]:
        if f["year"] == year and (theme is None or f["theme"] == theme):
            out.add(re.sub(r"^\d{4}(-\d+)?-", "", f["path"].split("/")[-1]).replace(".csv.zip", ""))
    return out


_v17, _v18 = variables_of(2017, "Vehicles"), variables_of(2018, "Vehicles")
# the two publication changes the prose names
assert {"Cars", "Parking Spaces"} <= _v17 and not ({"Cars", "Parking Spaces"} & _v18) and "Vehicle Count" in _v18 - _v17
assert all(len(f["path"].split("/")) == 4 for f in CENSUS["files"] if f["year"] == 2026)   # monthly folders
assert all(len(f["path"].split("/")) == 3 for f in CENSUS["files"] if f["year"] < 2026)    # one file per year
put("thm.vars.last", len({(f["theme"], v) for f in CENSUS["files"] if f["year"] == 2026
                          for v in [re.sub(r"^\d{4}(-\d+)?-", "", f["path"].split("/")[-1])]}), "{:d}")
_s_all, _s_26, _s_dis = pooled["slope"], ols(Y[_no26], R_LOC[_no26])["slope"], ols(Y[_keep], R_LOC[_keep])["slope"]
assert max(abs(_s_26 / _s_all - 1), abs(_s_dis / _s_all - 1)) < 0.2 and _s_26 < 0 and _s_dis < 0  # "neither propagates"

### S9 · Distance ratios against three homogeneous reference regions

The median distance from a year's new locations to the previous year's set is
divided by the same median for random points drawn in the year's bounding box,
in the previous year's 5th-95th percentile core, and from the previous year's
own locations jittered by half their median spacing. All three references
are homogeneous inside their region, so they are recorded and not used.

In [67]:
N_BENCH_KDE = 4000


def new_locations(units, k, seen=None):
    """Locations of year k absent from year k-1, or from every earlier year when `seen` is given."""
    prev, cur = units[YEARS[k - 1]], units[YEARS[k]]
    ref = seen if seen is not None else {tuple(p) for p in prev}
    return prev, np.array([p for p in cur if tuple(p) not in ref]).reshape(-1, 2)


def ratio_row(k):
    g = rng("thm:ratio:%d" % YEARS[k])
    prev, new = new_locations(LOCS, k)
    tree = cKDTree(prev)
    d_new = float(np.median(tree.query(new, k=1)[0]))

    def bench(region):
        lo, hi = region.min(axis=0), region.max(axis=0)
        B = g.uniform(lo, hi, size=(min(4000, max(500, 20 * len(new))), 2))
        return float(np.median(tree.query(B, k=1)[0]))

    lo, hi = np.percentile(prev, 5, axis=0), np.percentile(prev, 95, axis=0)
    core = prev[(prev[:, 0] >= lo[0]) & (prev[:, 0] <= hi[0]) & (prev[:, 1] >= lo[1]) & (prev[:, 1] <= hi[1])]
    box, cor = bench(LOCS[YEARS[k]]), bench(core)
    bw = float(np.median(nn_dist(prev))) / 2.0
    KD = prev[g.integers(0, len(prev), size=N_BENCH_KDE)] + g.normal(0.0, bw, size=(N_BENCH_KDE, 2))
    return YEARS[k], len(new), d_new / box, d_new / cor, d_new / float(np.median(tree.query(KD, k=1)[0]))


table("si-infillratios", ["%d & %d & %.3f & %.3f & %.3f" % ratio_row(k) for k in range(1, len(YEARS))])

### §3.6/S9/S20 · Infill tests and the Benjamini–Hochberg correction

A *new location* in year y is one present in y and absent in y-1, so a site
that falls silent and returns counts again. The random-labelling test asks
whether new locations lie further from the existing set than a random
relabelling of the pooled points gives. The pooled-subset test asks whether
new locations are closer to one another than random subsets of the pooled
points of the same size; its p-value is one-sided with the add-one estimator,
and the thirteen p-values are adjusted by the Benjamini-Hochberg step-up and,
because successive pairs share a year, by the Benjamini-Yekutieli correction.
Both tests are repeated on first-ever locations, those seen in no earlier
year, and on the 25 m merged sites.

In [68]:
def adjust(p, kind):
    """Benjamini-Hochberg ('bh') or Benjamini-Yekutieli ('by') adjusted p-values."""
    p = np.asarray(p, float)
    m = len(p)
    order = np.argsort(p)
    c = np.sum(1.0 / np.arange(1, m + 1)) if kind == "by" else 1.0
    adj = np.minimum.accumulate((p[order] * m * c / np.arange(1, m + 1))[::-1])[::-1]
    out = np.empty(m)
    out[order] = np.clip(adj, 0, 1)
    return out


def infill_tests(units, tag, first_ever=False):
    """Both tests for every year pair of a unit; returns one row per tested pair."""
    jobs, seen = [], {tuple(p) for p in units[YEARS[0]]}
    for k in range(1, len(YEARS)):
        prev, new = new_locations(units, k, seen if first_ever else None)
        seen |= {tuple(p) for p in units[YEARS[k]]}
        jobs.append((YEARS[k], prev, new))

    def one(job):
        y, prev, new = job
        row = {"year": y, "n": len(new)}
        if len(new) < 3:
            return row
        g = rng("thm:infill:%s:%d" % (tag, y))
        pooled_pts = np.vstack([prev, new])
        labels = np.array([0] * len(prev) + [1] * len(new))
        obs = float(np.median(cKDTree(prev).query(new, k=1)[0]))
        sims = np.empty(N_PERM_RL)
        for j in range(N_PERM_RL):
            lab = labels[g.permutation(len(pooled_pts))]
            sims[j] = np.median(cKDTree(pooled_pts[lab == 0]).query(pooled_pts[lab == 1], k=1)[0])
        lo, hi = np.percentile(sims, [2.5, 97.5])
        row.update(rl_obs=obs, rl_lo=float(lo), rl_hi=float(hi),
                   rl_pos="above" if obs > hi else ("below" if obs < lo else "inside"))
        obs2 = float(np.median(nn_dist(new)))
        s2 = np.empty(M_POOLED)
        for j in range(M_POOLED):
            s2[j] = np.median(nn_dist(pooled_pts[g.choice(len(pooled_pts), len(new), replace=False)]))
        lo2, hi2 = np.percentile(s2, [2.5, 97.5])
        row.update(nn_obs=obs2, nn_lo=float(lo2), nn_hi=float(hi2),
                   p=float((np.sum(s2 <= obs2) + 1) / (M_POOLED + 1)))
        return row

    rows = [r for r in pmap(one, jobs) if "p" in r]
    for r, a, b in zip(rows, adjust([r["p"] for r in rows], "bh"), adjust([r["p"] for r in rows], "by")):
        r["bh"], r["by"] = float(a), float(b)
    return rows


INF = memo("thm:infill:published", lambda: infill_tests(LOCS, "published"))
INF_FE = memo("thm:infill:firstever", lambda: infill_tests(LOCS, "firstever", first_ever=True))
INF_S25 = memo("thm:infill:site25", lambda: infill_tests(SITES25, "site25"))


def above(rows):
    return sum(r["rl_pos"] == "above" for r in rows)


def survivors(rows, key="bh"):
    return [r["year"] for r in rows if r[key] <= FDR]


table("si-infillrl", ["%d & %d & %.0f & $[%.0f, %.0f]$ & %s" % (r["year"], r["n"], r["rl_obs"], r["rl_lo"],
                                                               r["rl_hi"], r["rl_pos"]) for r in INF])
table("si-bh", ["%d & %d & %.0f & [%.0f, %.0f] & %.4f & %.4f" % (r["year"], r["n"], r["nn_obs"], r["nn_lo"],
                                                                 r["nn_hi"], r["p"], r["bh"]) for r in INF])
table("si-firstever", ["%d & %d & %.0f & $[%.0f, %.0f]$ & %.0f & $[%.0f, %.0f]$ & %.4f & %.4f"
                       % (r["year"], r["n"], r["rl_obs"], r["rl_lo"], r["rl_hi"], r["nn_obs"], r["nn_lo"],
                          r["nn_hi"], r["p"], r["bh"]) for r in INF_FE])

_n_pairs = len(INF)
_bh, _by = survivors(INF), survivors(INF, "by")
put("thm.pairs", _n_pairs, "{:d}", macro="rlPairs")
put("thm.pairs.batch", _n_pairs, "{:d}", macro="batchTotal")
put("thm.pairs.word", _n_pairs, word)
put("thm.rl.above", above(INF), "{:d}", macro="rlAbove")
put("thm.bh.n", len(_bh), "{:d}", macro="batchPairs")
put("thm.bh.n.word", len(_bh), word)
put("thm.bh.n.Word", len(_bh), lambda n: word(n).capitalize())
put("thm.bh.years", year_list(_bh))
put("thm.bh.fail.word", _n_pairs - len(_bh), word)
put("thm.bh.fail.years", year_list([r["year"] for r in INF if r["bh"] > FDR]))
put("thm.by.n.word", len(_by), word)
_bymax = max((r for r in INF if r["by"] <= FDR), key=lambda r: r["by"])
put("thm.by.max", _bymax["by"], "{:.3f}")
put("thm.m.pooled", M_POOLED, "{:d}")
put("thm.n.perm", N_PERM_RL, "{:d}")
put("thm.m.pooled.plus", M_POOLED + 1, "{:d}")
put("thm.p.floor", 1.0 / (M_POOLED + 1), "{:.4f}")
put("thm.new.total", sum(len(new_locations(LOCS, k)[1]) for k in range(1, len(YEARS))), "{:d}")
put("thm.new.firstever", sum(r["n"] for r in INF_FE), "{:d}")

# two worked rows of the table for its caption
_r14 = next(r for r in INF if r["year"] == 2014)
_r26 = next(r for r in INF if r["year"] == 2026)
put("thm.ex14.obs", _r14["nn_obs"], "{:.0f}")
put("thm.ex14.lo", _r14["nn_lo"], "{:.0f}")
put("thm.ex14.p", _r14["p"], "{:.4f}")
put("thm.ex26.obs", _r26["nn_obs"], "{:.0f}")
put("thm.ex26.lo", _r26["nn_lo"], "{:.0f}")
put("thm.ex26.hi", _r26["nn_hi"], "{:.0f}")
put("thm.ex26.p", _r26["p"], "{:.4f}")
put("thm.ex26.bh", _r26["bh"], "{:.4f}")

# the first-ever restriction
_fe_bh = survivors(INF_FE)
put("thm.fe.rl.above", above(INF_FE), "{:d}")
put("thm.fe.rl.inside", year_list([r["year"] for r in INF_FE if r["rl_pos"] != "above"]))
put("thm.rl.inside", year_list([r["year"] for r in INF if r["rl_pos"] != "above"]))
put("thm.fe.bh.n", len(_fe_bh), "{:d}")
put("thm.fe.lost", year_list(sorted(set(_bh) - set(_fe_bh))))
put("thm.fe.gained", year_list(sorted(set(_fe_bh) - set(_bh))))
# the merged unit, and the discontinuity years left out
put("thm.s25.rl.above", above(INF_S25), "{:d}")
put("thm.s25.bh.n", len(survivors(INF_S25)), "{:d}")
put("thm.s25.pairs", len(INF_S25), "{:d}")
assert above(INF_S25) < above(INF) and len(survivors(INF_S25)) < len(_bh)  # "a statement about the published unit"
assert above(INF) > _n_pairs / 2   # "lie further from the existing set"
_sub = [r for r in INF if r["year"] not in DISCONTINUITY_YEARS]
table("si-inflexcl", ["All years & %d & %d" % (above(INF), len(INF)),
                      "Excluding 2017--2018, 2026 & %d & %d" % (above(_sub), len(_sub))])
print("random labelling above %d of %d; BH %s; BY %s" % (above(INF), _n_pairs, _bh, _by))
print("first-ever: above %d, BH %s; sites 25 m: above %d, BH %d of %d"
      % (above(INF_FE), _fe_bh, above(INF_S25), len(survivors(INF_S25)), len(INF_S25)))

/tmp/ipykernel_16/2227444663.py:14: DeprecationWarning: This process (pid=16) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


random labelling above 12 of 13; BH [2015, 2016, 2017, 2020, 2022, 2023, 2024]; BY [2015, 2016, 2017, 2020, 2022, 2023, 2024]
first-ever: above 12, BH [2015, 2016, 2017, 2020, 2022, 2023, 2024, 2025]; sites 25 m: above 4, BH 3 of 13


### §2.3/S20 · The size of the search (multiplicity)

The level series are counted so a reader can see how many comparisons the
paper makes: the clustering index under three frames and two units, the
concentration index at four grid resolutions, and the rarefied concentration
index at three sample sizes, each over the fourteen years.

In [69]:
N_FRAMES, N_UNITS, N_RESOLUTIONS, N_ANCHORS = 3, 2, 4, 3
_n_series = N_FRAMES * N_UNITS + N_RESOLUTIONS + N_ANCHORS
put("thm.comp.series", _n_series, "{:d}")
put("thm.comp.n", _n_series * len(YEARS), "{:d}", macro="nComp")

182

## §4.1/§5/S10–S13 · The frozen snapshot audit and the CSR control

The snapshot is a database of 2026 that an earlier study used as its execution
target. This section compares its catalogue with the census, identifier by
identifier, and then asks what the comparison means once it is restricted to
the identifiers active in 2026. It also holds the control that the multiscale
sections lean on: the border-corrected $L$ of a uniform pattern in the fixed
hull, treated as a draw with a mean and a scatter.

The census is the archive read through 31 October 2026. The snapshot carries
no creation date, so the only dates stated for it are those of its readings.

In [70]:
SNP_CSR_RUNS = 199                         # uniform realisations per sample size
SNP_CSR_MID = 200                          # intermediate sample size of the control
SNP_LOO_H = 750.0                          # bandwidth of the leave-one-out intensity, m
SNP_LOO_YEARS = (2013, 2020, 2026)


def _day(ts):
    """'2026-05-11 00:55:00+00:00' -> '11 May 2026'."""
    y, m, d = (int(t) for t in ts[:10].split("-"))
    return "%d %s %d" % (d, ["January", "February", "March", "April", "May", "June", "July", "August",
                             "September", "October", "November", "December"][m - 1], y)


_con = sqlite3.connect(DB)
SNP_SENSORS = {str(r[0]).strip(): {"name": r[1], "lon": r[2], "lat": r[3], "feed": r[4]}
               for r in _con.execute("SELECT raw_id, sensor_name, centroid_lon, centroid_lat, broker_name FROM sensors")}
SNP_SENSOR_ROWS = _con.execute("SELECT COUNT(*) FROM sensors").fetchone()[0]
SNP_READ_LO, SNP_READ_HI = _con.execute("SELECT MIN(timestamp), MAX(timestamp) FROM readings").fetchone()
SNP_READ_ROWS = _con.execute("SELECT COUNT(*) FROM readings").fetchone()[0]
SNP_READ_NAMES = {r[0] for r in _con.execute("SELECT DISTINCT sensor_name FROM readings")}
_con.close()

### §4.1/S10 · The catalogue against the census

Coverage is stated twice. Against every identifier the census ever held it
mixes a 2026 catalogue with a fourteen-year record. Against the identifiers
active in 2026 it compares like with like, and that is the figure the paper
reads.

In [71]:
SNP_IDS = set(SNP_SENSORS)
SNP_CENSUS = set().union(*[set(REC[y]) for y in YEARS])
SNP_MATCHED = SNP_IDS & SNP_CENSUS
SNP_ONLY = SNP_IDS - SNP_CENSUS
SNP_ACTIVE = set(REC[YEARS[-1]])
SNP_IN = SNP_ACTIVE & SNP_IDS
SNP_OMITTED = SNP_ACTIVE - SNP_IDS

put("snp.read-rows", SNP_READ_ROWS, "{:d}")
put("snp.read-from", _day(SNP_READ_LO))
put("snp.read-to", _day(SNP_READ_HI))
put("snp.sensor-rows", SNP_SENSOR_ROWS, "{:d}")
put("snp.n", len(SNP_IDS), "{:d}")
put("snp.census", len(SNP_CENSUS), "{:d}")
put("snp.matched", len(SNP_MATCHED), "{:d}")
put("snp.only", len(SNP_ONLY), "{:d}")
put("snp.census-only", len(SNP_CENSUS - SNP_IDS), "{:d}")
put("snp.cov", 100.0 * len(SNP_MATCHED) / len(SNP_CENSUS), "{:.1f}", macro="covPct")
put("snp.active", len(SNP_ACTIVE), "{:d}")
put("snp.in", len(SNP_IN), "{:d}")
put("snp.omitted", len(SNP_OMITTED), "{:d}")
put("snp.cov-active", 100.0 * len(SNP_IN) / len(SNP_ACTIVE), "{:.1f}", macro="covActive")
put("snp.year", YEARS[-1], "{:d}")

# The reading-bearing identifiers join on the sensor name; the readings table has no raw_id.
_by_name = {v["name"]: k for k, v in SNP_SENSORS.items()}
SNP_READING_IDS = {_by_name[n] for n in SNP_READ_NAMES if n in _by_name}
put("snp.reading", len(SNP_READING_IDS), "{:d}")
put("snp.reading-in-census", len(SNP_READING_IDS & SNP_CENSUS), "{:d}")
put("snp.reading-wrong-join", len({n for n in SNP_READ_NAMES if n in SNP_SENSORS}), "{:d}")

table("si-snap", [
    "Readings window & %s to %s" % (SNP_READ_LO[:10], SNP_READ_HI[:10]),
    "Snapshot identifiers & %d" % len(SNP_IDS),
    "Identifiers with any reading & %d" % len(SNP_READING_IDS),
    "Census identifiers & %d" % len(SNP_CENSUS),
    "Matched & %d" % len(SNP_MATCHED),
    "Snapshot-only & %d" % len(SNP_ONLY),
    "Census-only & %d" % len(SNP_CENSUS - SNP_IDS),
    "Coverage of the census & %s\\%%" % VAL["snp.cov"]["text"],
    "Census identifiers active in %d & %d" % (YEARS[-1], len(SNP_ACTIVE)),
    "\\quad of those in the snapshot & %d" % len(SNP_IN),
    "\\quad omitted & %d" % len(SNP_OMITTED),
    "Coverage conditioned on %d & %s\\%%" % (YEARS[-1], VAL["snp.cov-active"]["text"]),
])

### §4.1/S10 · The vehicle contrast (conditioned odds ratio)

An identifier's theme is the theme directory of the last year it appears in.
Among the identifiers active in 2026, the omitted ones are compared with the
ones the snapshot holds.

In [72]:
SNP_THEME = {}
for _y in YEARS:
    for _i, _r in REC[_y].items():
        SNP_THEME[_i] = _r["theme"]


def _is_vehicle(i):
    return SNP_THEME[i].lower().startswith("veh")


_a = sum(_is_vehicle(i) for i in SNP_OMITTED)
_b = len(SNP_OMITTED) - _a
_c = sum(_is_vehicle(i) for i in SNP_IN)
_d = len(SNP_IN) - _c
SNP_OR = (_a * _d) / (_b * _c)
SNP_FISHER = float(stats.fisher_exact([[_a, _b], [_c, _d]])[1])


def _sci(p):
    """1.64e-08 -> '1.64e-8', the form siunitx reads."""
    m, e = ("%.2e" % p).split("e")
    return "%se%d" % (m, int(e))


put("snp.or", SNP_OR, "{:.2f}", macro="snapCondOR")
put("snp.fisher", SNP_FISHER, _sci)
table("si-veh", [
    "Omitted (%d-conditioned) & %d & %d & %.1f\\%% \\\\\n\\rowrule\n"
    "Matched (%d-conditioned) & %d & %d & %.1f\\%% \\\\\n\\midrule\n"
    "\\multicolumn{4}{l}{Odds ratio %s; Fisher exact $p = \\num{%s}$}"
    % (YEARS[-1], _a, _b, 100.0 * _a / (_a + _b), YEARS[-1], _c, _d, 100.0 * _c / (_c + _d),
       VAL["snp.or"]["text"], VAL["snp.fisher"]["text"])])
print("coverage %.1f%% / %.1f%%, table %s, odds ratio %.2f, p %.3g"
      % (get("snp.cov"), get("snp.cov-active"), [[_a, _b], [_c, _d]], SNP_OR, SNP_FISHER))

coverage 66.4% / 62.2%, table [[214, 72], [254, 216]], odds ratio 2.53, p 9.44e-09


### S12 · Snapshot-only identifiers against the census

The catalogue is checked against the census identifier by identifier. On the
complete census every snapshot identifier is present, so the snapshot is a
subset of the census and the disagreement between the two runs one way only:
the census holds identifiers the snapshot omits. Those are split into the
ones active in the last year, which are the omissions the audit studies, and
the ones that last reported earlier.

In [73]:
assert not SNP_ONLY, "the text says every snapshot identifier is in the census"
SNP_RETIRED = SNP_MATCHED - SNP_ACTIVE          # in the snapshot, not reporting in the last year
SNP_CENSUS_ONLY = SNP_CENSUS - SNP_IDS
SNP_CENSUS_ONLY_EARLIER = SNP_CENSUS_ONLY - SNP_ACTIVE
assert SNP_CENSUS_ONLY - SNP_CENSUS_ONLY_EARLIER == SNP_OMITTED
put("snp.retired", len(SNP_RETIRED), "{:d}")
put("snp.census-only-earlier", len(SNP_CENSUS_ONLY_EARLIER), "{:d}")
SNP_NEVER_IN_RANGE = N_OUT - SNP_CENSUS
put("snp.out-of-range", len(SNP_NEVER_IN_RANGE), "{:d}")
assert not (SNP_IDS & SNP_NEVER_IN_RANGE)       # the text says the snapshot holds none of them
print("snapshot-only %d, retired in snapshot %d, census-only %d (%d earlier, %d omitted)"
      % (len(SNP_ONLY), len(SNP_RETIRED), len(SNP_CENSUS_ONLY), len(SNP_CENSUS_ONLY_EARLIER), len(SNP_OMITTED)))

snapshot-only 0, retired in snapshot 941, census-only 715 (429 earlier, 286 omitted)


### S13 · What the snapshot contains, by ingestion feed

The snapshot stores no theme, so the feed an instrument is ingested from
stands in for it. The subset that matters to a user is the identifiers that carry
readings; the section tabulates them by feed.

In [74]:
def _ranked(counter):
    return sorted(counter.items(), key=lambda kv: (-kv[1], str(kv[0])))


put("snp.feeds", len({v["feed"] for v in SNP_SENSORS.values()}), "{:d}")
_rb = Counter(SNP_SENSORS[i]["feed"] for i in SNP_READING_IDS)
SNP_FEED_NAME = {"aq_mesh_api": "AQ Mesh API", "LORA": "LoRa", "Newcastle Council RSS feed": "Newcastle Council RSS"}
table("si-rbbroker", ["%s & %d & %.1f" % (SNP_FEED_NAME.get(f, f), n, 100.0 * n / len(SNP_READING_IDS))
                      for f, n in _ranked(_rb)])
print(_ranked(_rb))

[('aq_mesh_api', 16), ('LORA', 8)]


### S11 · The clustering control as a draw, not a value

A uniform pattern in the fixed hull has $L = 0$ at every scale in expectation,
but one realisation is a draw. The control is repeated at the smallest year's
size, at an intermediate size and at the last year's size. The mean over
realisations is the bias of the border-corrected estimator and the standard
deviation is the scatter of a single realisation.

In [75]:
SNP_SIZES = (int(N_LOCS.min()), SNP_CSR_MID, int(N_LOCS[-1]))


def _csr_curve(job):
    n, k = job
    return L_border(sample_hull(n, rng("snp:csr:%d:%d" % (n, k))))


_curves = np.array(pmap(_csr_curve, [(n, k) for n in SNP_SIZES for k in range(SNP_CSR_RUNS)]))
_curves = _curves.reshape(len(SNP_SIZES), SNP_CSR_RUNS, len(RADII))
SNP_CSR_MEAN = np.nanmean(_curves, axis=1)      # [size, radius]
SNP_CSR_SD = np.nanstd(_curves, axis=1)

put("snp.csr-runs", SNP_CSR_RUNS, "{:d}")
for _k, _name in enumerate(("small", "mid", "last")):
    put("snp.n-%s" % _name, SNP_SIZES[_k], "{:d}")
    put("snp.bias-%s" % _name, float(np.mean(np.abs(SNP_CSR_MEAN[_k]))), "{:.1f}")
    put("snp.sd-%s" % _name, float(np.mean(SNP_CSR_SD[_k])), "{:.1f}")
table("si-envelope", ["$%d$ & " % r + " & ".join("$%.1f$ & %.1f" % (SNP_CSR_MEAN[k, i], SNP_CSR_SD[k, i])
                                                 for k in range(len(SNP_SIZES)))
                      for i, r in enumerate(RADII)])
table("si-envelope-summary", [
    "Mean $|$bias$|$ (m) & " + " & ".join("%.1f" % np.mean(np.abs(SNP_CSR_MEAN[k])) for k in range(len(SNP_SIZES))),
    "Scatter, single realisation (m) & " + " & ".join("%.1f" % np.mean(SNP_CSR_SD[k]) for k in range(len(SNP_SIZES)))])

# The arithmetic the text walks through, at the smallest size.
_abs_sum = float(np.sum(np.abs(SNP_CSR_MEAN[0])))
_sd_sum = float(np.sum(SNP_CSR_SD[0]))
_sd_sum_printed = float(sum(round(float(s), 1) for s in SNP_CSR_SD[0]))
put("snp.scales", len(RADII), "{:d}")
put("snp.abs-sum", _abs_sum, "{:.4f}")
put("snp.sd-sum", _sd_sum, "{:.4f}")
put("snp.abs-mean4", _abs_sum / len(RADII), "{:.4f}")
put("snp.sd-mean4", _sd_sum / len(RADII), "{:.4f}")
put("snp.sd-sum-printed", _sd_sum_printed, "{:.1f}")
put("snp.sd-mean-printed4", _sd_sum_printed / len(RADII), "{:.4f}")
put("snp.sd-mean-printed", _sd_sum_printed / len(RADII), "{:.1f}")
assert VAL["snp.sd-mean-printed"]["text"] == VAL["snp.sd-small"]["text"]   # the text says the two print alike
_i1k = int(np.argmin(np.abs(RADII - 1000.0)))
put("snp.bias-small-1km", float(SNP_CSR_MEAN[0, _i1k]), "{:.0f}")
put("snp.sd-small-1km", float(SNP_CSR_SD[0, _i1k]), "{:.0f}")

/tmp/ipykernel_16/2227444663.py:14: DeprecationWarning: This process (pid=16) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


708.2742523445733

The bias is then set against the signal. Each sample size's absolute mean bias
is divided by the smallest observed $L$ of the fourteen years at that scale,
so the sparse column is a worst case across years.

In [76]:
SNP_L_MIN = L_LOC.min(axis=0)
table("si-envfrac", ["$%d$ & %.2f & %.2f & %.0f"
                     % (r, 100.0 * abs(SNP_CSR_MEAN[0, i]) / SNP_L_MIN[i],
                        100.0 * abs(SNP_CSR_MEAN[2, i]) / SNP_L_MIN[i], SNP_L_MIN[i])
                     for i, r in enumerate(RADII)])

### S11 · Leave-one-out intensity check

Each location's nearest-neighbour distance is divided by the distance that a
Gaussian kernel density of all the other locations predicts for it. A ratio
below one means the location is closer to its neighbour than the smoothed
density of the rest of the network implies.

In [77]:
def _loo(y):
    P = LOCS[y]
    D = cKDTree(P).query(P, k=len(P))[0]
    lam = np.exp(-0.5 * (D[:, 1:] / SNP_LOO_H) ** 2).sum(axis=1) / (2.0 * math.pi * SNP_LOO_H ** 2)
    ok = lam > 0
    ratio = D[ok, 1] / (1.0 / (2.0 * np.sqrt(lam[ok])))
    return len(P), float(np.median(ratio)), float(100.0 * np.mean(ratio < 1.0))


table("si-loo", ["%d & %d & %.2f & %.1f" % ((y,) + _loo(y)) for y in SNP_LOO_YEARS])
put("snp.loo-h", SNP_LOO_H, "{:.0f}")
print("CSR bias", [VAL["snp.bias-%s" % n]["text"] for n in ("small", "mid", "last")],
      "loo", [_loo(y) for y in SNP_LOO_YEARS])

CSR bias ['302.5', '21.8', '8.6'] loo [(43, 0.5260453407848903, 95.34883720930233), (850, 0.2197074198141042, 83.52941176470588), (564, 0.06097704271418444, 91.66666666666666)]


## §3.3/§3.5/S17–S18 · Size, registration unit and coordinate precision

This section asks three things about the nearest-neighbour index. Does it
fall because the network grew? Does the fall depend on which records count as
one unit? And where does the co-location come from? It feeds the thinning
section and the near-duplicate section of the supplement, and the main text's
paragraphs on the unit, on the index identity and on coordinate precision.

In [78]:
UNT_POOL_DRAWS = 2000     # draws from a pooled reference
UNT_THIN_DRAWS = 1000     # draws from a thinned reference network
UNT_MECH_DRAWS = 200      # thinnings of the fixed pattern at each size
UNT_TOLS = [1.0, 2.0, 5.0, 10.0, 15.0, 25.0, 50.0]   # merge radii of the sweep, metres
UNT_PRIMARY = 25.0        # the merge radius the text uses
UNT_SEED_STREAMS = 8      # extra random streams used to size the seed effect on z
UNT_GAP_HI = 200.0        # nearest-neighbour gaps are searched below this distance
UNT_OFFSETS = [0.0, 5.0, 10.0, 15.0, 20.0]           # grid-origin shifts, metres
UNT_DP_LOW, UNT_DP_HIGH = 6, 12                      # precision class limits, decimal places
UNT_MIN_CLASS = 5         # a class with fewer records gets no collapse figure
UNT_NULL_REPS = 20        # thinning repetitions for the exponent under pure thinning
YIDX = {y: k for k, y in enumerate(YEARS)}


def unt_trend(v):
    """OLS slope, standard error and p of an annual series."""
    r = ols(Y, v)
    return r["slope"], r["se"], r["p"]


put("unt.draws.pool", UNT_POOL_DRAWS, "{:d}")
put("unt.draws.thin", UNT_THIN_DRAWS, "{:d}")


def unt_draws(pool, n, tag, M):
    """Clark-Evans R of M random subsets of n points of a reference pattern."""
    g = rng(tag)
    return np.array([ce_r(pool[g.choice(len(pool), n, replace=False)], A_HULL) for _ in range(M)])

### S17 · The pooled size-matched reference

Every year is compared with random subsets of the same size drawn from the
pooled set of all locations of the record. The subsets keep the network's
overall structure and change only the count, so the difference between a year
and its subsets is what the year's arrangement adds to its size. The standard
deviation is the sample one.

In [79]:
UNT_POOL = dedup(np.vstack([LOCS[y] for y in YEARS]))


def _pooled_row(k):
    y = YEARS[k]
    d = unt_draws(UNT_POOL, len(LOCS[y]), "unt:pooled:%d" % y, UNT_POOL_DRAWS)
    return float(d.mean()), float(d.std(ddof=1))


UNT_POOLED = pmap(_pooled_row, range(len(YEARS)))
UNT_Z = np.array([(R_LOC[k] - m) / s for k, (m, s) in enumerate(UNT_POOLED)])
table("si-pooled", ["%d & %d & %.3f & %.3f & %.3f & $%+.2f$" % (y, N_LOCS[k], R_LOC[k], UNT_POOLED[k][0],
                                                               UNT_POOLED[k][1], UNT_Z[k])
                    for k, y in enumerate(YEARS)])
put("unt.pool.n", len(UNT_POOL), "{:d}")
put("unt.pool.z2026", UNT_Z[-1], "{:+.2f}")
put("unt.pool.r2013", UNT_POOLED[0][0], "{:.3f}")
put("unt.pool.r2026", UNT_POOLED[-1][0], "{:.3f}")
assert abs(UNT_Z[0]) < 1.96, "2013 is no longer indistinguishable from its size-matched draw"
assert UNT_Z[-1] < -1.96, "2026 is no longer more clustered than its size predicts"

/tmp/ipykernel_16/2227444663.py:14: DeprecationWarning: This process (pid=16) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


### S17 · Thinned references (2026-anchored and peak-anchored)

The second reference thins one year's network, without replacement, to each
other year's size. A year that holds more locations than the reference cannot
be matched, and the unthinned reference stands in for it. The last year's
network is the reference the text reads first. The largest network of the
record is the second reference, because every other year can be matched
against it.

In [80]:
def unt_thinned(ref_year):
    ref = LOCS[ref_year]

    def one(k):
        n = len(LOCS[YEARS[k]])
        if n >= len(ref):
            return ce_r(ref, A_HULL), 0.0, False
        d = unt_draws(ref, n, "unt:thin:%d:%d" % (ref_year, YEARS[k]), UNT_THIN_DRAWS)
        return float(d.mean()), float(d.std(ddof=1)), True

    return pmap(one, range(len(YEARS)))


UNT_LAST = YEARS[-1]
UNT_BIG = YEARS[int(np.argmax(N_LOCS))]      # the year with the most locations
UNT_TL = unt_thinned(UNT_LAST)
UNT_TB = unt_thinned(UNT_BIG)
UNT_EL = np.array([R_LOC[k] - UNT_TL[k][0] for k in range(len(YEARS))])
UNT_EB = np.array([R_LOC[k] - UNT_TB[k][0] for k in range(len(YEARS))])
def _thin_cells(k, T, E, ref_year):
    """Reference and excess of a year, or dashes where the year holds more locations than the reference."""
    if not T[k][2] and YEARS[k] != ref_year:
        return "-- & --"
    return "%.3f & $%+.3f$" % (T[k][0], E[k])


table("si-thinning", ["%d & %d & %.3f & %s & %s"
                      % (y, N_LOCS[k], R_LOC[k], _thin_cells(k, UNT_TL, UNT_EL, UNT_LAST),
                         _thin_cells(k, UNT_TB, UNT_EB, UNT_BIG))
                      for k, y in enumerate(YEARS)])
_unmatched = [y for k, y in enumerate(YEARS) if not UNT_TL[k][2] and y != UNT_LAST]
assert _unmatched == list(range(_unmatched[0], _unmatched[-1] + 1)), "the unmatched years are no longer one run"
put("unt.thin26.unmatched.n", len(_unmatched), "{:d}")
put("unt.thin26.unmatched.first", _unmatched[0], "{:d}")
put("unt.thin26.unmatched.last", _unmatched[-1], "{:d}")
put("unt.big.year", UNT_BIG, "{:d}")
put("unt.n.locbig", int(N_LOCS[YIDX[UNT_BIG]]), "{:d}")
put("unt.n.loc2013", int(N_LOCS[0]), "{:d}")
put("unt.n.loc2026", int(N_LOCS[-1]), "{:d}")
put("unt.r.loc2026", R_LOC[-1], "{:.3f}")
put("unt.thin26.r2013", UNT_TL[0][0], "{:.3f}")
put("unt.thin26.share", 100.0 * (UNT_TL[0][0] - R_LOC[-1]) / (R_LOC[0] - R_LOC[-1]), "{:.0f}")
assert UNT_TL[0][0] > R_LOC[-1] + 0.5 * (R_LOC[0] - R_LOC[-1]), "thinning no longer reproduces most of the decline"
_others = [k for k, y in enumerate(YEARS) if y != UNT_BIG]
_below = [k for k in _others if UNT_EB[k] < -2.0 * UNT_TB[k][1]]     # more clustered than the reference
_level = [k for k in _others if k not in _below]                     # within two standard deviations of it
assert all(abs(UNT_EB[k]) <= 2.0 * UNT_TB[k][1] for k in _level), "a year is less clustered than the largest network"
print("thinned %d reference: below" % UNT_BIG, [YEARS[k] for k in _below], "level", [YEARS[k] for k in _level],
      np.round(UNT_EB, 3).tolist())
put("unt.thinbig.below.n", len(_below), "{:d}")
put("unt.thinbig.level.n", len(_level), "{:d}")
put("unt.thinbig.level.years", ", ".join(str(YEARS[k]) for k in _level[:-1]) + " and " + str(YEARS[_level[-1]])
    if len(_level) > 1 else str(YEARS[_level[0]]))
put("unt.thinbig.gaplo", min(abs(UNT_EB[k]) for k in _below), "{:.2f}")
put("unt.thinbig.gaphi", max(abs(UNT_EB[k]) for k in _below), "{:.2f}")

/tmp/ipykernel_16/2227444663.py:14: DeprecationWarning: This process (pid=16) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


thinned 2018 reference: below [2015, 2016, 2017, 2020, 2022, 2023, 2024, 2025, 2026] level [2013, 2014, 2019, 2021] [-0.064, -0.158, -0.19, -0.185, -0.198, 0.0, 0.005, -0.045, -0.021, -0.166, -0.204, -0.249, -0.252, -0.247]


0.25229440343325205

### S17 · The mechanism: does R fall on a fixed pattern?

Two fixed clusters of 4000 points each are thinned to 4000 points and to 250
points in all. Nothing moves, only the count changes, and the change in the
mean R between the two sizes is the size effect a clustered pattern has when
its structure is held.

In [81]:
def unt_mechanism():
    g = rng("unt:mechanism")
    big = np.vstack([c + g.normal(0, 250.0, size=(4000, 2)) for c in np.array([[0.0, 0.0], [3000.0, 2000.0]])])
    return {n: float(np.mean([ce_r(big[g.choice(len(big), n, replace=False)], A_HULL)
                              for _ in range(UNT_MECH_DRAWS)])) for n in (4000, 250)}


UNT_MECH = memo("unt:mechanism", unt_mechanism)
put("unt.mech.pct", 100.0 * abs(UNT_MECH[250] - UNT_MECH[4000]) / UNT_MECH[4000], "{:.0f}")

4.546497031943221

### §3.3/S17 · The exponent of mean nearest-neighbour distance on count

On a fixed frame the logarithm of R is the logarithm of the mean
nearest-neighbour distance plus half the logarithm of the count, minus half
the logarithm of the area, plus a constant. Densification at constant shape
gives the distance an exponent of minus one half on the count. The fit is
repeated on merged sites and with each year's own hull area divided out,
because the occupied area and the near-duplicate registrations both grow with
the count. Standard errors are Newey-West at lag 2 and the test is against
minus one half.

In [82]:
_dbar = np.array([nn_dist(LOCS[y]).mean() for y in YEARS])
_ident = np.log(R_LOC) - (math.log(2.0) + np.log(_dbar) + 0.5 * np.log(N_LOCS) - 0.5 * math.log(A_HULL))
def _sci(x):
    e = int(math.floor(math.log10(x)))
    return r"%.1f\times10^{%d}" % (x / 10.0 ** e, e)


put("unt.identity.err", float(np.max(np.abs(_ident))), _sci)


def unt_exponent(units, area=False):
    N = np.array([len(units[y]) for y in YEARS], float)
    d = np.array([nn_dist(units[y]).mean() for y in YEARS])
    A = np.array([hull_of(units[y])[1] for y in YEARS])
    v = np.log(d / np.sqrt(A)) if area else np.log(d)
    b, se, lo, hi = hac(np.log(N), v)
    o = ols(np.log(N), v)
    t = (b + 0.5) / se
    return {"b": b, "se": se, "t": t, "p": float(2 * stats.t.sf(abs(t), len(YEARS) - 2)),
            "ols_se": o["se"], "ols_t": (b + 0.5) / o["se"],
            "area_slope": ols(np.log(N), np.log(A))["slope"],
            "corr": float(np.corrcoef(np.log(N), np.log(A))[0, 1])}


UNT_EXP = {u: (unt_exponent(UNITS[u]), unt_exponent(UNITS[u], area=True)) for u in ("loc", "site10", "site25", "ids")}
table("si-scalingcomp", ["%s & $%+.3f$ (%.3f; %.3f) & $%+.2f$ & %s & $%+.3f$ (%.3f; %.3f) & $%+.2f$ & $%s$"
                         % (name, p["b"], p["se"], p["ols_se"], p["t"], pval(p["p"]),
                            a["b"], a["se"], a["ols_se"], a["t"], pval(a["p"]))
                         for name, (p, a) in (("Exact locations", UNT_EXP["loc"]),
                                              (r"Merged \SI{10}{\meter}", UNT_EXP["site10"]),
                                              (r"Merged \SI{25}{\meter}", UNT_EXP["site25"]))])
_p, _a = UNT_EXP["loc"]
put("unt.scale.loc", _p["b"], "{:+.3f}", macro="cleanSlopeLoc")
put("unt.scale.loc.se", _p["se"], "{:.3f}", macro="cleanSELoc")
put("unt.scale.loc.t", _p["t"], "{:+.2f}", macro="cleanTLoc")
put("unt.scale.loc.p", _p["p"], "{:.2f}", macro="cleanPLoc")
put("unt.scale.loc.diff", abs(_p["b"] + 0.5), "{:.3f}")
put("unt.scale.loc.area", _a["b"], "{:+.3f}")
put("unt.scale.corr", _p["corr"], "{:+.2f}")
put("unt.scale.site10", UNT_EXP["site10"][0]["b"], "{:+.3f}")
put("unt.scale.site25", UNT_EXP["site25"][0]["b"], "{:+.3f}")
put("unt.scale.ids", UNT_EXP["ids"][0]["b"], "{:+.3f}")
_areas = [UNT_EXP[u][1]["b"] for u in ("loc", "site10", "site25")]
put("unt.scale.area.shallow", max(_areas), "{:+.2f}")
put("unt.scale.area.steep", min(_areas), "{:+.2f}")
assert max(_areas) < -0.5, "an area-controlled exponent is no longer steeper than minus one half"
assert _p["p"] > 0.05, "the plain exponent now differs from minus one half"
assert UNT_EXP["site25"][0]["b"] > _p["b"], "merging no longer makes the exponent shallower"


def unt_thinning_beta():
    """Exponent when the 2026 network is thinned to each year's count: pure thinning."""
    g = rng("unt:thinning-beta")
    P = LOCS[2026]
    ok = N_LOCS <= len(P)
    out = []
    for _ in range(UNT_NULL_REPS):
        d = np.array([nn_dist(P[g.choice(len(P), int(n), replace=False)]).mean() for n in N_LOCS[ok]])
        out.append(hac(np.log(N_LOCS[ok].astype(float)), np.log(d))[0])
    return float(np.mean(out))


put("unt.scale.thinning", memo("unt:thinning-beta", unt_thinning_beta), "{:+.3f}")
# The same exponents under the OLS standard error, and the part of the area-normalised exponent that is the footprint.
put("unt.scale.loc.olsse", _p["ols_se"], "{:.3f}")
put("unt.scale.loc.olst", _p["ols_t"], "{:+.2f}")
put("unt.scale.loc.area.se", _a["se"], "{:.3f}")
put("unt.scale.loc.area.olsse", _a["ols_se"], "{:.3f}")
_a25 = UNT_EXP["site25"][1]
put("unt.scale.site25.area", _a25["b"], "{:+.3f}")
put("unt.scale.site25.area.se", _a25["se"], "{:.3f}")
put("unt.scale.site25.area.olsse", _a25["ols_se"], "{:.3f}")
put("unt.scale.site25.area.olst", _a25["ols_t"], "{:+.2f}")
put("unt.scale.lnA.slope", _a["area_slope"], "{:.2f}")
put("unt.scale.lnA.half", 0.5 * _a["area_slope"], "{:.2f}")
assert abs((_p["b"] - 0.5 * _a["area_slope"]) - _a["b"]) < 1e-9, "the area-normalised exponent is not the plain one less half the area slope"
_tcrit = stats.t.ppf(0.975, len(YEARS) - 2)
assert all(abs(UNT_EXP[u][1]["ols_t"]) > _tcrit for u in ("loc", "site10", "site25")), "an area-normalised exponent rests on the Newey-West error"
assert abs(_p["ols_t"]) < _tcrit, "the plain exponent differs from minus one half under OLS"


### §3.3/S18 · The tolerance sweep (1 m to 50 m)

The location unit keeps one record per coordinate rounded to 0.1 m. The sweep
instead joins identifiers closer than a radius by single linkage, keeps the
first identifier of each group, and recomputes the fixed-hull series. The
millimetre row is the finest control. The last column compares the 2026 index
with random subsets of the same size from the pooled sites at the same radius;
it is repeated on further random streams to show how far it moves with the
stream.

In [83]:
def unt_units(r):
    """The unit of every year at merge radius r; r = 0 is the millimetre control."""
    return {y: (dedup(IDS[y], 3) if r <= 0 else merge_single(IDS[y], r)) for y in YEARS}


def _sweep_row(r):
    U = unt_units(r)
    v = np.array([ce_r(U[y], A_HULL) for y in YEARS])
    pool = dedup(ALL_XY, 3) if r <= 0 else merge_single(ALL_XY, r)
    zs = []
    for s in range(1 + UNT_SEED_STREAMS):
        d = unt_draws(pool, len(U[2026]), "unt:tol:%g:%d" % (r, s), UNT_POOL_DRAWS)
        zs.append(float((v[-1] - d.mean()) / d.std(ddof=1)))
    b, se, p = unt_trend(v)
    return {"r": r, "v": v, "n": [len(U[y]) for y in YEARS], "slope": b, "se": se, "p": p,
            "fall": 100.0 * fall(v), "z": zs[0], "zsd": float(np.std(zs, ddof=1))}


UNT_SWEEP = {row["r"]: row for row in pmap(_sweep_row, [0.0] + UNT_TOLS)}


def _fallcell(f):
    return (r"%.1f\%%" % f) if f >= 0 else (r"$-$%.1f\%%" % abs(f))


table("si-tolerance", ["%s & %.3f & %.3f & %s & $%+.5f$ & %.3f & $%+.1f$"
                       % ("Exact coordinate (mm control)" if r <= 0 else "Within %g m" % r,
                          s["v"][0], s["v"][-1], _fallcell(s["fall"]), s["slope"], s["p"], s["z"])
                       for r, s in ((r, UNT_SWEEP[r]) for r in [0.0] + UNT_TOLS)])
_falls = [UNT_SWEEP[r]["fall"] for r in UNT_TOLS]
assert all(a > b for a, b in zip(_falls, _falls[1:])), "the fall is no longer monotone in the radius"
_s50 = UNT_SWEEP[UNT_TOLS[-1]]
assert 0 < _falls[-1] < _falls[0] / 4, "the fall at the widest radius is no longer a small positive remainder"
assert _s50["slope"] > 0 and _s50["p"] > 0.05, "the slope at the widest radius is no longer positive and untrended"
put("unt.tol.slope50", _s50["slope"], "{:+.4f}")
put("unt.tol.p50", _s50["p"], "{:.3f}")
put("unt.tol.zspread", max(s["zsd"] for s in UNT_SWEEP.values()), "{:.2f}")
put("unt.tol.mm.n2026", UNT_SWEEP[0.0]["n"][-1], "{:d}")
put("unt.tol.mm.r2026", UNT_SWEEP[0.0]["v"][-1], "{:.3f}")
put("unt.tol.mm.fall", UNT_SWEEP[0.0]["fall"], "{:.1f}")
put("unt.loc.fall", 100.0 * fall(R_LOC), "{:.1f}")
put("unt.loc.p", unt_trend(R_LOC)[2], "{:.3f}")
for _r, _m in ((1.0, "tolFallOne"), (5.0, "tolFallFive"), (10.0, "tolFallTen"), (15.0, "tolFallFifteen"),
               (50.0, "tolFallFifty")):
    put("unt.tol.fall%g" % _r, UNT_SWEEP[_r]["fall"], "{:.1f}", macro=_m)
_s25 = UNT_SWEEP[UNT_PRIMARY]
put("unt.tol.fall25", _s25["fall"], "{:.1f}")
put("unt.tol.slope25", _s25["slope"], "{:+.4f}")
put("unt.tol.p25", _s25["p"], "{:.3f}")
assert _s25["p"] > 0.5 and abs(_s25["slope"]) < 0.1 * abs(unt_trend(R_LOC)[0]), "the 25 m merge no longer removes the fitted trend"

table("si-ndmerge", ["%d & %d & %d & %d & %d & %d"
                     % (y, N_IDS[k], UNT_SWEEP[5.0]["n"][k], UNT_SWEEP[10.0]["n"][k], _s25["n"][k],
                        N_IDS[k] - _s25["n"][k]) for k, y in enumerate(YEARS)])
put("unt.n.ids2013", int(N_IDS[0]), "{:d}")
put("unt.n.ids2026", int(N_IDS[-1]), "{:d}")
put("unt.merge25.removed2026", int(N_IDS[-1] - _s25["n"][-1]), "{:d}")
put("unt.merge25.removed2013", int(N_IDS[0] - _s25["n"][0]), "{:d}")
put("unt.merge25.sites2026", int(_s25["n"][-1]), "{:d}")
put("unt.surplus.n2026", int(N_IDS[-1] - N_LOCS[-1]), "{:d}")
_rem = 100.0 * (N_IDS - np.array(_s25["n"])) / N_IDS      # share of identifiers the 25 m merge removes
put("unt.merge25.share2013", _rem[0], "{:.1f}")
put("unt.merge25.share2026", _rem[-1], "{:.1f}")
assert stats.spearmanr(Y, _rem)[0] > 0.8, "the merged share no longer rises across the record"

/tmp/ipykernel_16/2227444663.py:14: DeprecationWarning: This process (pid=16) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


### S18 · The pooled reference on near-duplicate–merged sites

The pooled comparison is repeated on the 25 m sites, with the pool merged at
the same radius, so the first table of this section can be read again under
the wider tolerance.

In [84]:
UNT_POOL25 = merge_single(ALL_XY, UNT_PRIMARY)


def _pooled25_row(k):
    y = YEARS[k]
    d = unt_draws(UNT_POOL25, len(SITES25[y]), "unt:pooled25:%d" % y, UNT_POOL_DRAWS)
    return float(d.mean()), float(d.std(ddof=1))


UNT_POOLED25 = pmap(_pooled25_row, range(len(YEARS)))
table("si-near-pooled", ["%d & %d & %.3f & %.3f & %.3f & $%+.2f$"
                         % (y, N_SITES25[k], R_SITE25[k], UNT_POOLED25[k][0], UNT_POOLED25[k][1],
                            (R_SITE25[k] - UNT_POOLED25[k][0]) / UNT_POOLED25[k][1])
                         for k, y in enumerate(YEARS)])
_z25 = (R_SITE25[-1] - UNT_POOLED25[-1][0]) / UNT_POOLED25[-1][1]
assert UNT_Z[-1] < _z25 < -1.96, "on merged sites the last year's separation no longer shrinks without vanishing"

/tmp/ipykernel_16/2227444663.py:14: DeprecationWarning: This process (pid=16) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


### S17/S18 · Why the pooled and the thinned references disagree

A size-matched draw from the pool keeps a co-located pair together only when
both members are drawn, and the pool spans the whole fixed hull while the
last year's network occupies part of it. Both are measured: the retention
probability of a pair, and the pooled reference redrawn from the pool's
members inside the last year's own hull.

In [85]:
_n26, _npool = len(LOCS[UNT_LAST]), len(UNT_POOL)
_own26 = OWN_POLY[UNT_LAST]
UNT_POOL_IN = UNT_POOL[in_hull(UNT_POOL, _own26, 1e-6)]
UNT_POOL25_IN = UNT_POOL25[in_hull(UNT_POOL25, _own26, 1e-6)]
_d = unt_draws(UNT_POOL_IN, _n26, "unt:pooled-in:%d" % UNT_LAST, UNT_POOL_DRAWS)
_d25 = unt_draws(UNT_POOL25_IN, len(SITES25[UNT_LAST]), "unt:pooled25-in:%d" % UNT_LAST, UNT_POOL_DRAWS)
_z_in = (R_LOC[-1] - _d.mean()) / _d.std(ddof=1)
_z25_in = (R_SITE25[-1] - _d25.mean()) / _d25.std(ddof=1)
put("unt.pool.incl", _n26 / _npool, "{:.3f}")
put("unt.pool.pairkeep", _n26 * (_n26 - 1) / (_npool * (_npool - 1)), "{:.3f}")
put("unt.own.share2026", A_OWN[-1] / A_HULL, "{:.3f}")
put("unt.own.km2.2026", A_OWN[-1] / 1e6, "{:.0f}")
put("unt.pool.in.n", len(UNT_POOL_IN), "{:d}")
put("unt.pool.in.r", float(_d.mean()), "{:.3f}")
put("unt.pool.in.z", float(_z_in), "{:+.2f}")
put("unt.pool25.n", len(UNT_POOL25), "{:d}")
put("unt.pool25.in.n", len(UNT_POOL25_IN), "{:d}")
put("unt.pool25.in.r", float(_d25.mean()), "{:.3f}")
put("unt.pool25.in.sd", float(_d25.std(ddof=1)), "{:.3f}")
put("unt.pool25.in.z", float(_z25_in), "{:+.2f}")
put("unt.pool25.r2026", UNT_POOLED25[-1][0], "{:.3f}")
put("unt.pool25.z2026", float(_z25), "{:+.2f}")
put("unt.site.r2026", R_SITE25[-1], "{:.3f}")
put("unt.site.n2026", len(SITES25[UNT_LAST]), "{:d}")
# The prose: inside the last year's own hull the merged network is indistinguishable from a
# size-matched draw of the merged pool, and the exact-coordinate network is not.
assert abs(_z25_in) < 1.96 and _z_in < -1.96, (_z25_in, _z_in)
assert _d.mean() < UNT_POOLED[-1][0] and _d25.mean() < UNT_POOLED25[-1][0], "restricting the pool does not lower its reference"
print("pool inside the %d hull: exact z %.2f, merged z %.2f" % (UNT_LAST, _z_in, _z25_in))

pool inside the 2026 hull: exact z -6.53, merged z -0.79


### §3.3/S18 · Linkage rule, grid-snap and within-theme merge

A merge radius would be justified by a trough in the nearest-neighbour
distances that separates co-located records from distinct installations. The
largest gap below 200 m is therefore reported for every year, with the share
of identifiers that have a neighbour inside a radius. The merge is then
repeated under complete linkage, with a centroid in place of the first
identifier, on a fixed grid at 25 shifted origins, and with identifiers of
different themes never joined.

In [86]:
def unt_gap(P):
    d = np.sort(nn_dist(P))
    d = d[(d > 0) & (d < UNT_GAP_HI)]
    return float(np.diff(d).max()) if len(d) >= 3 else float("nan")


UNT_GAPS = np.array([unt_gap(IDS[y]) for y in YEARS])
print("largest gap below %g m by year:" % UNT_GAP_HI, dict(zip(YEARS, np.round(UNT_GAPS, 1).tolist())))
_later = UNT_GAPS[2:]
assert np.nanmax(_later) < min(UNT_GAPS[0], UNT_GAPS[1]), "a later year now has the largest gap"
put("unt.gap.max", float(np.nanmax(UNT_GAPS)), "{:.0f}")
put("unt.gap.y2013", UNT_GAPS[0], "{:.0f}")
put("unt.gap.y2014", UNT_GAPS[1], "{:.0f}")
put("unt.gap.restlo", float(np.nanmin(_later)), "{:.0f}")
put("unt.gap.resthi", float(np.nanmax(_later)), "{:.0f}")


def unt_share(y, r):
    """Share of a year's identifiers with another identifier closer than r, per cent."""
    return 100.0 * float(np.mean(nn_dist(IDS[y]) < r))


put("unt.share25.y2026", unt_share(2026, 25.0), "{:.1f}")
put("unt.share5.y2026", unt_share(2026, 5.0), "{:.1f}")
put("unt.share25.y2013", unt_share(2013, 25.0), "{:.1f}")
_surplus = 100.0 * (N_IDS - N_LOCS) / N_IDS
_kmax = int(np.argmax(_surplus))
_jump = next(k for k in range(len(YEARS)) if _surplus[k] > 10.0)    # first year above ten per cent
assert all(_surplus[k] < 5.0 for k in range(_jump)), "the surplus is no longer small before its jump"
assert _surplus[-1] > 2 * _surplus[_jump] - 5.0 and _kmax > _jump, "the surplus no longer grows after its jump"
put("unt.surplus.max", _surplus.max(), "{:.1f}", macro="surplusMax")
put("unt.surplus.maxyear", YEARS[_kmax], "{:d}")
put("unt.surplus.maxnum", int(N_IDS[_kmax] - N_LOCS[_kmax]), "{:d}")
put("unt.surplus.maxden", int(N_IDS[_kmax]), "{:d}")
put("unt.surplus.jumpyear", YEARS[_jump], "{:d}")
put("unt.surplus.beforejump", YEARS[_jump - 1], "{:d}")
put("unt.surplus.jump", _surplus[_jump], "{:.1f}")
put("unt.surplus.y2026", _surplus[-1], "{:.1f}")


def unt_rule(fn):
    """Fall, slope, standard error and p of the fixed-hull series under a merge rule."""
    U = [fn(y) for y in YEARS]
    v = np.array([ce_r(P, A_HULL) for P in U])
    b, se, p = unt_trend(v)
    return {"fall": 100.0 * fall(v), "slope": b, "se": se, "p": p, "first": v[0], "last": v[-1],
            "n2026": len(U[-1])}


UNT_COMPLETE = unt_rule(lambda y: merge_complete(IDS[y], UNT_PRIMARY))
UNT_CENTROID = unt_rule(lambda y: merge_centroid(IDS[y], UNT_PRIMARY))
put("unt.complete.fall", UNT_COMPLETE["fall"], "{:.1f}", macro="clFall")
put("unt.complete.p", UNT_COMPLETE["p"], "{:.3f}", macro="clP")
put("unt.centroid.fall", UNT_CENTROID["fall"], "{:.1f}")
assert max(abs(UNT_COMPLETE["fall"] - _s25["fall"]), abs(UNT_CENTROID["fall"] - _s25["fall"])) < 2.0, \
    "the pairwise merge rules no longer agree"
assert UNT_COMPLETE["p"] > 0.5

UNT_GRID = pmap(lambda o: unt_rule(lambda y: merge_grid(IDS[y], UNT_PRIMARY, o)),
                [(ox, oy) for ox in UNT_OFFSETS for oy in UNT_OFFSETS])
_g0 = UNT_GRID[0]
_gf = np.array([g["fall"] for g in UNT_GRID])
_gp = np.array([g["p"] for g in UNT_GRID])
_gn = np.array([g["n2026"] for g in UNT_GRID])
assert abs(_g0["fall"] - _gf.max()) < 1e-9, "the first grid origin is no longer the largest fall"
assert _gp.min() > 0.05, "a grid origin now shows a trend at the five per cent level"
put("unt.grid.fall", _g0["fall"], "{:.1f}", macro="gridSnapFall")
put("unt.grid.p", _g0["p"], "{:.3f}", macro="gridSnapP")
put("unt.grid.n2026", _g0["n2026"], "{:d}")
put("unt.grid.split", _g0["n2026"] - _s25["n"][-1], "{:d}")
put("unt.grid.falllo", _gf.min(), "{:.1f}")
put("unt.grid.fallhi", _gf.max(), "{:.1f}")
put("unt.grid.fallmed", float(np.median(_gf)), "{:.1f}")
put("unt.grid.nlo", int(_gn.min()), "{:d}")
put("unt.grid.nhi", int(_gn.max()), "{:d}")
put("unt.grid.plo", _gp.min(), "{:.3f}")
put("unt.grid.phi", _gp.max(), "{:.3f}")


def unt_within_theme(y):
    """25 m single linkage inside each theme; themes are never joined."""
    th = np.array([r["theme"] for r in REC[y].values()])
    return np.vstack([merge_single(IDS[y][th == t], UNT_PRIMARY) if (th == t).sum() > 1 else IDS[y][th == t]
                      for t in np.unique(th)])


UNT_THEME = unt_rule(unt_within_theme)
put("unt.theme.n2026", UNT_THEME["n2026"], "{:d}")
put("unt.theme.fall", UNT_THEME["fall"], "{:.1f}")
put("unt.theme.first", UNT_THEME["first"], "{:.3f}")
put("unt.theme.last", UNT_THEME["last"], "{:.3f}")
put("unt.theme.slope", UNT_THEME["slope"], "{:+.4f}")
put("unt.theme.se", UNT_THEME["se"], "{:.4f}")
put("unt.theme.p", UNT_THEME["p"], "{:.2f}")
assert abs(UNT_THEME["fall"] - _s25["fall"]) < 3.0 and UNT_THEME["p"] > 0.5, "the within-theme merge now differs"
assert _g0["fall"] > _s25["fall"] + 10.0, "the grid rule no longer leaves a larger fall than the pairwise rules"

largest gap below 200 m by year: {2013: 44.2, 2014: 35.1, 2015: 11.6, 2016: 14.5, 2017: 9.4, 2018: 6.6, 2019: 6.9, 2020: 6.3, 2021: 8.8, 2022: 15.0, 2023: 14.5, 2024: 14.5, 2025: 15.5, 2026: 15.5}


/tmp/ipykernel_16/2227444663.py:14: DeprecationWarning: This process (pid=16) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


### §3.5/S18 · Coordinate precision classes

The archive prints coordinates to different numbers of decimal places. A
record's precision is the smaller count of its longitude and latitude, read
from the strings the archive prints. Three
classes follow: six places or fewer, seven to eleven, twelve or more. For each
class and year the section counts the identifiers, the surplus over distinct
0.1 m coordinates inside the class, and the coordinates a coarser class
already holds, so the parts add up to identifiers minus locations. It also
gives the share of a class with a same-class neighbour inside a distance, and
the surplus of a class after its coordinates are rounded to six decimal
places of a degree, which shows whether rounding could have created the
co-location.

In [87]:
def unt_decimals(s):
    s = s.strip()
    if "e" in s.lower():
        s = repr(float(s))
    return len(s.split(".")[1]) if "." in s else 0


def unt_class(r):
    d = min(unt_decimals(r["lon"]), unt_decimals(r["lat"]))
    return "low" if d <= UNT_DP_LOW else ("high" if d >= UNT_DP_HIGH else "mid")


UNT_ORDER = ("low", "mid", "high")


def unt_precision(y):
    recs = list(REC[y].values())
    P = IDS[y]
    K = np.array([unt_class(r) for r in recs])
    Q = np.round(P, 1)
    R6 = np.array([proj(round(float(r["lon"]), 6), round(float(r["lat"]), 6)) for r in recs])
    out, seen = {}, set()
    for c in UNT_ORDER:
        m = K == c
        n = int(m.sum())
        coords = {tuple(q) for q in Q[m]}
        row = {"n": n, "within": n - len(coords), "cross": len(coords & seen)}
        seen |= coords
        if n >= 2:
            d = nn_dist(P[m])
            row["near05"] = int((d < 0.5).sum())
            row["near25"] = int((d < 25.0).sum())
            row["rounded"] = 100.0 * (1 - len(dedup(R6[m])) / n)
        out[c] = row
    assert sum(r["within"] + r["cross"] for r in out.values()) == len(P) - len(LOCS[y]), y
    return out


UNT_PREC = {y: unt_precision(y) for y in YEARS}


def _cells(row):
    if row["n"] == 0:
        return "0 & 0 & --"
    return "%d & %d & %s" % (row["n"], row["within"],
                             "--" if row["n"] < UNT_MIN_CLASS else "%.1f" % (100.0 * row["within"] / row["n"]))


table("si-precisionfull", ["%d & %s & %s & %s & %d & %d"
                           % (y, _cells(UNT_PREC[y]["low"]), _cells(UNT_PREC[y]["mid"]), _cells(UNT_PREC[y]["high"]),
                              sum(UNT_PREC[y][c]["cross"] for c in UNT_ORDER), N_IDS[k] - N_LOCS[k])
                           for k, y in enumerate(YEARS)])


def unt_collapse(y, c):
    r = UNT_PREC[y][c]
    return 100.0 * r["within"] / r["n"]


_tot = sum(UNT_PREC[y][c]["n"] for y in YEARS for c in UNT_ORDER)
put("unt.prec.share.low", 100.0 * sum(UNT_PREC[y]["low"]["n"] for y in YEARS) / _tot, "{:.1f}")
put("unt.prec.share.high", 100.0 * sum(UNT_PREC[y]["high"]["n"] for y in YEARS) / _tot, "{:.1f}")
for _c in ("low", "high"):
    _rows = [UNT_PREC[y][_c] for y in YEARS if UNT_PREC[y][_c]["n"] >= 2]
    put("unt.prec.pool25.%s" % _c, 100.0 * sum(r["near25"] for r in _rows) / sum(r["n"] for r in _rows), "{:.1f}")
    put("unt.prec.near05.%s.y2026" % _c, 100.0 * UNT_PREC[2026][_c]["near05"] / UNT_PREC[2026][_c]["n"], "{:.1f}")
    put("unt.prec.within.%s.y2026" % _c, UNT_PREC[2026][_c]["within"], "{:d}")
put("unt.prec.within.mid.y2026", UNT_PREC[2026]["mid"]["within"], "{:d}")
put("unt.prec.cross.y2026", sum(UNT_PREC[2026][c]["cross"] for c in UNT_ORDER), "{:d}")
for _y in (2017, 2020):
    put("unt.prec.rounded.high.y%d" % _y, UNT_PREC[_y]["high"]["rounded"], "{:.1f}")
    assert abs(UNT_PREC[_y]["high"]["rounded"] - unt_collapse(_y, "high")) < 0.05, "rounding now changes the high class"
for _c in ("low", "high"):
    put("unt.prec.c.%s.y2026" % _c, unt_collapse(2026, _c), "{:.1f}")


def _col(y, c):
    """Collapse of a class in a year, or None when the class is too small to give one."""
    return unt_collapse(y, c) if UNT_PREC[y][c]["n"] >= UNT_MIN_CLASS else None


# The high class leads from its first year with a surplus until the low class overtakes it.
_hi_first = next(y for y in YEARS if _col(y, "high") and _col(y, "high") > _col(y, "low"))
_switch = next(y for y in YEARS if y > _hi_first and _col(y, "low") > _col(y, "high"))
_early = [y for y in YEARS if _hi_first <= y < _switch]
_late = [y for y in YEARS if y >= _switch]
assert all(_col(y, "high") > _col(y, "low") + 2 for y in _early), "the high class no longer leads before the switch"
assert all(_col(y, "low") > _col(y, "high") + 2 for y in _late), "the low class no longer leads after the switch"
put("unt.prec.hi.first", _hi_first, "{:d}")
put("unt.prec.switch", _switch, "{:d}")
put("unt.prec.switch.before", _switch - 1, "{:d}")
put("unt.prec.hi.lo", min(_col(y, "high") for y in _early + _late), "{:.1f}")
put("unt.prec.hi.hi", max(_col(y, "high") for y in _early + _late), "{:.1f}")
put("unt.prec.low.earlymax", max(_col(y, "low") for y in YEARS if y < _switch), "{:.1f}")
put("unt.prec.low.latelo", min(_col(y, "low") for y in _late), "{:.1f}")
put("unt.prec.low.latehi", max(_col(y, "low") for y in _late), "{:.1f}")
_mid_run = [y for y in YEARS if _col(y, "mid") is not None and _col(y, "mid") > 15.0]   # the middle class's episode
assert _mid_run == list(range(_mid_run[0], _mid_run[-1] + 1)), "the middle-class episode is no longer one run of years"
_mid_after = [y for y in YEARS if y > _mid_run[-1]]
assert _mid_after and all(_col(y, "mid") < 2.0 for y in _mid_after), "the middle class no longer empties after its episode"
put("unt.prec.mid.first", _mid_run[0], "{:d}")
put("unt.prec.mid.last", _mid_run[-1], "{:d}")
put("unt.prec.mid.lo", min(_col(y, "mid") for y in _mid_run), "{:.1f}")
put("unt.prec.mid.hi", max(_col(y, "mid") for y in _mid_run), "{:.1f}")
put("unt.prec.mid.aftermax", max(_col(y, "mid") for y in _mid_after), "{:.1f}")
assert UNT_PREC[2026]["low"]["near05"] / UNT_PREC[2026]["low"]["n"] > UNT_PREC[2026]["high"]["near05"] / UNT_PREC[2026]["high"]["n"]
assert get("unt.prec.pool25.high") > get("unt.prec.pool25.low"), "pooled over the record the high class no longer leads"
print("section 17: tolerance falls", [round(f, 1) for f in _falls], "z", round(UNT_Z[-1], 2))
# The three classes partition the identifier-years; the class is read from the coordinate the census keeps.
assert _tot == int(N_IDS.sum())
put("unt.prec.n.idyears", _tot, "{:d}")
put("unt.prec.share.mid", 100.0 * sum(UNT_PREC[y]["mid"]["n"] for y in YEARS) / _tot, "{:.1f}")


def _class_of(lon, lat):
    d = min(unt_decimals(lon), unt_decimals(lat))
    return "low" if d <= UNT_DP_LOW else ("high" if d >= UNT_DP_HIGH else "mid")


_multi = [r for y in YEARS for r in REC[y].values() if len(r["coords"]) > 1]
put("unt.prec.multicoord.n", len(_multi), "{:d}")
put("unt.prec.multiclass.n", sum(1 for r in _multi if len({_class_of(a, b) for a, b in r["coords"]}) > 1), "{:d}")
assert len(_multi) < 0.01 * _tot, "more than one identifier-year in a hundred records several coordinates"


section 17: tolerance falls [51.0, 50.8, 49.0, 41.6, 30.5, 15.8, 5.9] z -6.52


## §1/S19 · Directional imbalance: the per-pair orientation test

The orientations of pairs of 2026 locations within 1 km are folded modulo pi
and counted in eight sectors. A pair is counted when at least one endpoint
lies inside the fixed hull eroded by the pair radius, which is the pair set
the sector K-function accumulates. Every statistic is on undirected pairs:
one angle per pair.

The test holds the point set and the pair set fixed and gives each undirected
pair one independent uniform orientation. The statistic is the ratio of the
fullest sector to the emptiest.

In [88]:
DIR_NSEC = 8          # orientation sectors
DIR_R = 1000.0        # pair radius in metres
DIR_NMAIN = 2000      # draws of the main null
DIR_NCTRL = 1000      # draws of each control null
DIR_NROT = 999        # rigid rotations
DIR_NCSR = 1000       # draws of the rejected point-count null
DIR_AXIS_SD = 0.05    # radians; spread of the concentrated half in the synthetic control
DIR_NCOPY = 200       # independent copies of the randomised control
ROAD_RADII = np.array([250.0, 500.0, 750.0, 1000.0, 1500.0])
ROAD_NDRAW = 120      # random draws per road-support subset


def dir_hist(a, w=None):
    return np.histogram(a, bins=DIR_NSEC, range=(0, math.pi), weights=w)[0]


def dir_ratio(h):
    return float(h.max() / h.min()) if h.min() > 0 else float("inf")


def dir_pairs(P, poly=HULL_POLY):
    """Counted pairs of P: index pairs, one angle each, and endpoints inside the eroded window."""
    P = np.asarray(P, float)
    inside = eroded_inside(P, DIR_R, poly)
    pr = cKDTree(P).query_pairs(DIR_R, output_type="ndarray")
    ni = inside[pr[:, 0]].astype(int) + inside[pr[:, 1]].astype(int)
    pr, ni = pr[ni > 0], ni[ni > 0]
    d = P[pr[:, 1]] - P[pr[:, 0]]
    return pr, np.arctan2(d[:, 1], d[:, 0]) % math.pi, ni, inside


def dir_null(n, ndraw, g):
    """Sector ratio of ndraw sets of n independent uniform orientations."""
    return np.array([dir_ratio(dir_hist(g.uniform(0, math.pi, n))) for _ in range(ndraw)])


def dir_test(angles, ndraw, g):
    obs = dir_ratio(dir_hist(angles))
    nr = dir_null(len(angles), ndraw, g)
    ge = int((nr >= obs).sum())
    return {"ratio": obs, "mean": float(nr.mean()), "sd": float(nr.std()),
            "lo": float(np.percentile(nr, 2.5)), "hi": float(np.percentile(nr, 97.5)),
            "ge": ge, "p": (ge + 1) / (ndraw + 1)}


def dir_chi2(h):
    e = h.sum() / DIR_NSEC
    c = float(((h - e) ** 2 / e).sum())
    return c, float(stats.chi2.sf(c, DIR_NSEC - 1))


def sci(p):
    """A small p-value as one digit times a power of ten, in LaTeX."""
    e = int(math.floor(math.log10(p)))
    m = int(round(p / 10.0 ** e))
    if m == 10:
        m, e = 1, e + 1
    return r"%d\times10^{%d}" % (m, e)


def listed(v, f="{:d}"):
    """A list of values as prose: 'a, b, c and d'."""
    s = [f.format(x) for x in v]
    return ", ".join(s[:-1]) + " and " + s[-1]


def dir_sharing(pr, keep):
    """Mean pairs per kept point and the share of pairs that close a triangle."""
    nb = defaultdict(set)
    for a, b in pr:
        nb[int(a)].add(int(b))
        nb[int(b)].add(int(a))
    tri = sum(1 for a, b in pr if nb[int(a)] & nb[int(b)])
    return 2.0 * len(pr) / int(keep.sum()), tri / len(pr)

### S19 · The pair set (erosion, boundary strip, sector counts)

The boundary strip is checked directly: the text says that no counted pair
has exactly one endpoint in it, so every pair is entered twice in the ordered
count. The cell stops if that is not so.

In [89]:
DIR_YEAR = YEARS[-1]     # the test is run on the last year of the record
DIR_P = LOCS[DIR_YEAR]
dir_pr, dir_ang, dir_ni, dir_in = dir_pairs(DIR_P)
dir_tree = cKDTree(DIR_P)
dir_strip = np.nonzero(~dir_in)[0]
dir_strip_nb = [set(dir_tree.query_ball_point(DIR_P[i], DIR_R)) - {int(i)} for i in dir_strip]
dir_isolated = sum(1 for s in dir_strip_nb if not s)
dir_strip_pairs = sum(len(s) for s in dir_strip_nb) // 2
assert int((dir_ni == 1).sum()) == 0, "a counted pair has one endpoint in the boundary strip"
assert all(s <= set(int(i) for i in dir_strip) for s in dir_strip_nb), "a strip point neighbours an interior point"

put("dir.n.loc", len(DIR_P), "{:d}")
put("dir.n.inside", int(dir_in.sum()), "{:d}")
put("dir.strip.n", len(dir_strip), "{:d}")
put("dir.strip.isolated", dir_isolated, "{:d}")
put("dir.strip.rest", len(dir_strip) - dir_isolated, "{:d}")
put("dir.strip.pairs", dir_strip_pairs, "{:d}")
put("dir.n.pairs", len(dir_ang), "{:d}")
put("dir.n.ordered", int(dir_ni.sum()), "{:d}")
assert get("dir.n.ordered") == 2 * get("dir.n.pairs")

### §1/S19 · The result (conditional randomisation null)

The observed ratio is set against the null of independent uniform
orientations. Pearson's chi-square against equal sector counts is computed on
the undirected counts; the same statistic on ordered counts is recorded only
to show that counting each pair twice doubles it.

In [90]:
dir_h = dir_hist(dir_ang)
dir_main = dir_test(dir_ang, DIR_NMAIN, rng("dir:main"))
dir_c, dir_cp = dir_chi2(dir_h)
dir_co, _ = dir_chi2(dir_hist(dir_ang, dir_ni))

put("dir.ratio", dir_main["ratio"], "{:.2f}")
put("dir.main.mean", dir_main["mean"], "{:.3f}")
put("dir.main.sd", dir_main["sd"], "{:.3f}")
put("dir.main.lo", dir_main["lo"], "{:.3f}")
put("dir.main.hi", dir_main["hi"], "{:.3f}")
put("dir.main.ge", dir_main["ge"], "{:d}")
put("dir.main.p", dir_main["p"], "{:.4f}")
put("dir.chi2", dir_c, "{:.1f}")
put("dir.chi2.p", dir_cp, sci)
put("dir.chi2.ordered", dir_co, "{:.1f}")
put("dir.sectors", [int(v) for v in dir_h], listed)
assert dir_main["ratio"] > dir_main["hi"], "the observed ratio is inside the null interval"

### S19 · Controls (three, because a null that cannot pass is not a test)

Three controls are run, each against its own null. The observed pattern must
reject. A copy whose orientations are already uniform is itself a random draw
and rejects one time in twenty by construction, so the control is the share of
many independent copies that reject at the five per cent level, which must sit
near five per cent. A synthetic pattern with half its pairs within a small
angle of one axis must reject.

In [91]:
dir_n = len(dir_ang)
dir_obs_ctl = dir_test(dir_ang, DIR_NCTRL, rng("dir:control:observed"))
def dir_copy(k):
    g = rng("dir:control:copy:%d" % k)
    return dir_test(g.uniform(0, math.pi, dir_n), DIR_NCTRL, g)["p"]


dir_copy_p = np.array(pmap(dir_copy, range(DIR_NCOPY)))
dir_copy_rej = float((dir_copy_p < 0.05).mean())
g = rng("dir:control:axis")
dir_axis_ang = np.concatenate([g.normal(0.0, DIR_AXIS_SD, dir_n // 2) % math.pi,
                               g.uniform(0, math.pi, dir_n - dir_n // 2)])
dir_axis = dir_test(dir_axis_ang, DIR_NCTRL, g)

put("dir.ctl.obs.p", dir_obs_ctl["p"], "{:.3f}")
put("dir.ctl.copy.n", DIR_NCOPY, "{:d}")
put("dir.ctl.copy.reject", dir_copy_rej, pct)
put("dir.ctl.copy.median", float(np.median(dir_copy_p)), "{:.2f}")
put("dir.ctl.axis.p", dir_axis["p"], "{:.3f}")
put("dir.ctl.axis.ratio", dir_axis["ratio"], "{:.2f}")
assert dir_obs_ctl["p"] < 0.05 and dir_axis["p"] < 0.05, "a control that must reject does not"
assert dir_copy_rej < 0.10, "randomised copies reject far above the nominal rate"

/tmp/ipykernel_16/2227444663.py:14: DeprecationWarning: This process (pid=16) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


### S19 · The null that was rejected (CSR pair count)

A uniform draw of the same number of points inside the hull has far fewer
pairs within the radius than the observed network, so its sector ratio
measures counting noise. The ordered pair count and the ratio of such draws
are recorded to show why this comparison is not used.

In [92]:
def dir_csr(k):
    Q = sample_hull(len(DIR_P), rng("dir:csr:%d" % k))
    _, a, w, _ = dir_pairs(Q)
    h = dir_hist(a, w)
    return float(w.sum()), (dir_ratio(h) if h.min() > 0 else float("nan"))


dir_csr_res = np.array(pmap(dir_csr, range(DIR_NCSR)))
dir_csr_pairs = float(dir_csr_res[:, 0].mean())
dir_csr_r = dir_csr_res[:, 1][np.isfinite(dir_csr_res[:, 1])]

put("dir.csr.pairs", dir_csr_pairs, "{:.0f}")
put("dir.csr.persector", dir_csr_pairs / DIR_NSEC, "{:.0f}")
put("dir.csr.mean", float(dir_csr_r.mean()), "{:.2f}")
put("dir.csr.sd", float(dir_csr_r.std()), "{:.2f}")
put("dir.csr.above", float((dir_csr_r > dir_main["ratio"]).mean()), pct)
put("dir.csr.factor", get("dir.n.ordered") / dir_csr_pairs, "{:.0f}")
assert dir_main["ratio"] < dir_csr_r.mean(), "the observed ratio is not below the point-count null"

/tmp/ipykernel_16/2227444663.py:14: DeprecationWarning: This process (pid=16) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


### S19 · Rigid-rotation bin-alignment check

A rigid rotation adds one angle to every pair, so it re-phases the same
angular distribution against the fixed sector edges and measures nothing else.

In [93]:
g = rng("dir:rotation")
dir_rot = np.array([dir_ratio(dir_hist((dir_ang + g.uniform(0, math.pi)) % math.pi))
                    for _ in range(DIR_NROT)])
dir_rot_ge = int((dir_rot >= dir_main["ratio"]).sum())
table("si-rotation", ["%d & %.3f & %.3f & $[%.3f, %.3f]$ & %d & %.3f" % (
    DIR_NROT, dir_rot.mean(), dir_rot.std(), dir_rot.min(), dir_rot.max(),
    dir_rot_ge, (dir_rot_ge + 1) / (DIR_NROT + 1))])

### S19 · Vertex sharing, and the test on 25 m merged sites

Pairs that share a point are not independent. The share of pairs that close a
triangle measures how far the pair set is from independent pairs. The test is
repeated on the 25 m merged sites, where tight groups are one point. A subset
of themes that sit off the street network is counted to show that it is too
small to test.

In [94]:
dir_deg, dir_tri = dir_sharing(dir_pr, dir_in)
put("dir.deg", dir_deg, "{:.1f}")
put("dir.tri", dir_tri, pct)

DIR_S = SITES25[DIR_YEAR]
dir_spr, dir_sang, dir_sni, dir_sin = dir_pairs(DIR_S)
dir_sh = dir_hist(dir_sang)
dir_site = dir_test(dir_sang, DIR_NMAIN, rng("dir:sites"))
dir_sc, dir_scp = dir_chi2(dir_sh)
_, dir_stri = dir_sharing(dir_spr, dir_sin)

put("dir.site.n", len(DIR_S), "{:d}")
put("dir.site.inside", int(dir_sin.sum()), "{:d}")
put("dir.site.pairs", len(dir_sang), "{:d}")
put("dir.site.sectors", [int(v) for v in dir_sh], listed)
put("dir.site.ratio", dir_site["ratio"], "{:.2f}")
put("dir.site.mean", dir_site["mean"], "{:.2f}")
put("dir.site.sd", dir_site["sd"], "{:.2f}")
put("dir.site.lo", dir_site["lo"], "{:.2f}")
put("dir.site.hi", dir_site["hi"], "{:.2f}")
put("dir.site.ge", dir_site["ge"], "{:d}")
put("dir.site.p", dir_site["p"], "{:.3f}")
put("dir.site.chi2", dir_sc, "{:.1f}")
put("dir.site.chi2.p", dir_scp, "{:.4f}")
put("dir.site.tri", dir_stri, pct)
assert dir_site["p"] < 0.05, "the merged-site test does not reject"
assert dir_site["p"] > dir_main["p"], "the merged-site evidence is not weaker"

DIR_OFF = locs_xy(DIR_YEAR, lambda r: r["theme"] in ("Weather", "Building"))
dir_off_pairs = len(cKDTree(DIR_OFF).query_pairs(DIR_R))
put("dir.off.n", len(DIR_OFF), "{:d}")
put("dir.off.pairs", dir_off_pairs, "{:d}")
put("dir.off.persector", dir_off_pairs / DIR_NSEC, "{:.1f}")

1.125

### §4.3/S19 · Does the road-bound theme carry the medium-scale excess?

The 2026 locations are split into vehicle counters and everything else. Each
subset's border-corrected L on the fixed hull is set against uniform draws of
the same size in the same hull. The excess is the observed L minus the mean
of the draws, and z divides it by their standard deviation.

In [95]:
def road_support(name, P):
    obs = L_border(P, ROAD_RADII)
    sim = np.array(pmap(lambda k: L_border(sample_hull(len(P), rng("dir:road:%s:%d" % (name, k))), ROAD_RADII),
                        range(ROAD_NDRAW)))
    mu, sd = np.nanmean(sim, axis=0), np.nanstd(sim, axis=0, ddof=1)
    return obs - mu, (obs - mu) / sd


ROAD_VEH = locs_xy(DIR_YEAR, lambda r: r["theme"] == "Vehicles")
ROAD_NON = locs_xy(DIR_YEAR, lambda r: r["theme"] != "Vehicles")
road = {"veh": road_support("veh", ROAD_VEH), "non": road_support("non", ROAD_NON)}

put("dir.road.veh.n", len(ROAD_VEH), "{:d}")
put("dir.road.non.n", len(ROAD_NON), "{:d}")
put("dir.road.people.n", len(locs_xy(DIR_YEAR, lambda r: r["theme"] == "People")), "{:d}")
put("dir.road.aq.n", len(locs_xy(DIR_YEAR, lambda r: r["theme"] == "Air Quality")), "{:d}")
for name, (ex, z) in road.items():
    for k in range(len(ROAD_RADII)):
        put("dir.road.%s.ex%d" % (name, k + 1), float(ex[k]), "{:.0f}")
        put("dir.road.%s.z%d" % (name, k + 1), float(z[k]), "{:+.1f}")
assert min(road["veh"][1].min(), road["non"][1].min()) > 5.0, "a subset is not far above its control"
assert (road["non"][0] > road["veh"][0]).all(), "the non-vehicle excess is not the larger in metres"
assert (road["non"][1] < road["veh"][1]).all(), "the non-vehicle z is not the smaller"
assert DIR_YEAR == 2026, "the prose names 2026 as the year of the test"
_non_themes = {r["theme"] for r in REC[DIR_YEAR].values()} - {"Vehicles"}
assert _non_themes == {"Air Quality", "Building", "Water Level", "People", "Weather", "Sensor Metrics"}, _non_themes
assert get("dir.road.people.n") + get("dir.road.aq.n") > len(ROAD_NON) / 2, "People and Air Quality are not the majority"

print("pairs %d, ratio %.3f, null %.3f (%.3f), ge %d; sites ratio %.3f p %.3f; csr pairs %.1f"
      % (dir_n, dir_main["ratio"], dir_main["mean"], dir_main["sd"], dir_main["ge"],
         dir_site["ratio"], dir_site["p"], dir_csr_pairs))

/tmp/ipykernel_16/2227444663.py:14: DeprecationWarning: This process (pid=16) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


pairs 2781, ratio 1.422, null 1.166 (0.052), ge 0; sites ratio 1.415 p 0.033; csr pairs 212.0


### §3.6/§4.3/S19 · The street network as the support (linear-network K)

The planar tests cannot say how much of the medium-scale excess is the
streets themselves. For the last year, inside the fixed core disc, the merged
sites are matched to an OpenStreetMap street and footway graph and the
geometrically corrected network K of Ang, Baddeley and Nair is computed on
shortest-path distances. Under uniform placement on the network that K
equals r. The observed value is set against draws on the same network, both
uniform and with the Gaussian-kernel intensity of the sites, at four
map-matching tolerances.

In [96]:
NET_RADII = np.array([250.0, 500.0, 750.0, 1000.0, 1500.0])
NET_STEP = 5.0                 # spacing of the network positions used to match sites and to draw nulls, metres
NET_DISC = 5000.0              # radius of the core disc about EDG_CENTRE, metres
NET_TOLS = (5.0, 10.0, 25.0, 50.0)
NET_H = 750.0                  # kernel bandwidth of the intensity null, metres
NET_NDRAW = 199
NET_YEAR = YEARS[-1]

import scipy.sparse as _sp
from scipy.sparse.csgraph import dijkstra as _dijkstra


class Net:
    """A street graph with positions every NET_STEP metres along its edges."""

    def __init__(self, xy, edges):
        self.xy = np.asarray(xy, float)
        self.e = np.asarray(edges, int)
        self.len = np.hypot(*(self.xy[self.e[:, 0]] - self.xy[self.e[:, 1]]).T)
        k = np.maximum(1, np.ceil(self.len / NET_STEP).astype(int))
        self.s_edge = np.repeat(np.arange(len(self.e)), k)
        j = np.arange(k.sum()) - np.repeat(np.concatenate([[0], np.cumsum(k)[:-1]]), k)
        self.s_piece = np.repeat(self.len / k, k)
        self.s_off = (j + 0.5) * self.s_piece
        a, b = self.xy[self.e[self.s_edge, 0]], self.xy[self.e[self.s_edge, 1]]
        self.s_xy = a + (self.s_off / np.maximum(self.len[self.s_edge], 1e-9))[:, None] * (b - a)
        self.tree = cKDTree(self.s_xy)

    def snap(self, P):
        """Edge, offset along it and distance moved, for the network point nearest each point of P."""
        P = np.asarray(P, float)
        e = self.s_edge[self.tree.query(P)[1]]
        a, b = self.xy[self.e[e, 0]], self.xy[self.e[e, 1]]
        v = b - a
        f = np.clip(((P - a) * v).sum(axis=1) / np.maximum((v * v).sum(axis=1), 1e-12), 0.0, 1.0)
        return e, f * self.len[e], np.hypot(*(P - (a + f[:, None] * v)).T)

    def xy_of(self, e, off):
        a, b = self.xy[self.e[e, 0]], self.xy[self.e[e, 1]]
        return a + (off / np.maximum(self.len[e], 1e-9))[:, None] * (b - a)

    def with_points(self, e, off):
        """The graph with the points inserted as nodes: sparse matrix and its edge list."""
        n0 = len(self.xy)
        free = np.ones(len(self.e), bool)
        free[e] = False
        a_, b_, w_ = [], [], []
        order = np.lexsort((off, e))
        k = 0
        while k < len(order):
            m = k
            while m < len(order) and e[order[m]] == e[order[k]]:
                m += 1
            ed = e[order[k]]
            prev, pos = self.e[ed, 0], 0.0
            for t in order[k:m]:
                a_.append(prev); b_.append(n0 + t); w_.append(max(off[t] - pos, 1e-6))
                prev, pos = n0 + t, off[t]
            a_.append(prev); b_.append(self.e[ed, 1]); w_.append(max(self.len[ed] - pos, 1e-6))
            k = m
        A = np.concatenate([self.e[free, 0], np.array(a_, int)])
        B = np.concatenate([self.e[free, 1], np.array(b_, int)])
        W = np.concatenate([self.len[free], np.array(w_, float)])
        n = n0 + len(e)
        G = _sp.coo_matrix((np.concatenate([W, W]), (np.concatenate([A, B]), np.concatenate([B, A]))),
                           shape=(n, n)).tocsr()
        return G, A, B, W


def net_k(net, e, off, total_len, radii=NET_RADII, chunk=24):
    """Geometrically corrected K on the network: each pair is weighted by one over the number of
    network locations at the same shortest-path distance from its first point."""
    n, n0, rmax = len(e), len(net.xy), float(radii.max())
    G, A, B, W = net.with_points(e, off)
    out = np.zeros(len(radii))
    for c0 in range(0, n, chunk):
        src = np.arange(c0, min(n, c0 + chunk))
        for row, i in zip(_dijkstra(G, directed=False, indices=n0 + src, limit=rmax), src):
            t = row[n0:n0 + n].copy()
            t[i] = np.inf
            tt = t[t <= rmax]
            if not len(tt):
                continue
            da, db = row[A], row[B]
            near = np.minimum(da, db) <= rmax
            da, db, w = da[near][None, :], db[near][None, :], W[near][None, :]
            tq = np.maximum(tt - 1e-6, tt / 2.0)[:, None]          # just inside each pair's distance
            xa, xb = tq - da, tq - db
            with np.errstate(invalid="ignore"):
                m = (((xa >= 0) & (xa <= w) & (xa <= (w + db - da) / 2.0)).sum(axis=1)
                     + ((xb >= 0) & (xb <= w) & (xb < (w + da - db) / 2.0)).sum(axis=1))
            wgt = 1.0 / np.maximum(m, 1)
            out += np.array([wgt[tt <= r].sum() for r in radii])
    return total_len * out / (n * (n - 1.0))


NET_FILE = DATA / "streets.json"
assert NET_FILE.exists(), "the street graph is missing from the deposit: %s" % NET_FILE
_net = json.loads(NET_FILE.read_text())
NET = Net(np.array(_net["nodes"], float) / 10.0, _net["edges"])
# the graph must reach beyond the disc by the largest radius, so shortest paths are not cut
assert math.dist(EDG_CENTRE, _net["centre_m"]) + NET_DISC + NET_RADII.max() <= _net["radius_m"]
_ein = ((np.hypot(*(NET.xy[NET.e[:, 0]] - EDG_CENTRE).T) <= NET_DISC)
        & (np.hypot(*(NET.xy[NET.e[:, 1]] - EDG_CENTRE).T) <= NET_DISC))
NET_LEN = float(NET.len[_ein].sum())                   # length of network the points may occupy
NET_POS = np.nonzero(_ein[NET.s_edge])[0]              # positions inside the disc
_S = SITES25[NET_YEAR]
NET_P = _S[np.hypot(*(_S - EDG_CENTRE).T) <= NET_DISC]
NET_E, NET_OFF, NET_MOVED = NET.snap(NET_P)


def net_test(tol):
    """Observed K of the sites matched within `tol`, and the two nulls drawn on the network."""
    keep = NET_MOVED <= tol
    n = int(keep.sum())
    e, off = NET_E[keep], NET_OFF[keep]
    dd = cKDTree(NET.xy_of(e, off)).query(NET.s_xy[NET_POS], k=min(n, 100))[0]
    piece = NET.s_piece[NET_POS]
    pk = np.exp(-0.5 * (dd / NET_H) ** 2).sum(axis=1) * piece
    pk, pu = pk / pk.sum(), piece / piece.sum()

    def draw(g, p):
        i = NET_POS[g.choice(len(NET_POS), n, replace=False, p=p)]
        ed = NET.s_edge[i]
        return ed, np.clip(NET.s_off[i] + g.uniform(-0.5, 0.5, n) * NET.s_piece[i], 0.0, NET.len[ed])
    ker = np.array(pmap(lambda j: net_k(NET, *draw(rng("net:kernel:%d:%d" % (tol, j)), pk), NET_LEN), range(NET_NDRAW)))
    uni = np.array(pmap(lambda j: net_k(NET, *draw(rng("net:uniform:%d:%d" % (tol, j)), pu), NET_LEN), range(NET_NDRAW)))
    return {"n": n, "obs": net_k(NET, e, off, NET_LEN), "ker_mean": ker.mean(axis=0),
            "ker_hi": np.percentile(ker, 97.5, axis=0), "uni_mean": uni.mean(axis=0),
            "uni_hi": np.percentile(uni, 97.5, axis=0)}


NET_T = memo("net:tests", lambda: {tol: net_test(tol) for tol in NET_TOLS})
table("si-linnet", [
    "\\SI{%g}{\\meter} & %d & " % (tol, t["n"])
    + " & ".join("$%.0f$ / $%.0f$" % (t["obs"][k], t["ker_hi"][k]) for k in range(len(NET_RADII)))
    for tol, t in NET_T.items()]
    + ["Uniform on the network, mean & %d & " % NET_T[25.0]["n"]
       + " & ".join("$%.0f$" % v for v in NET_T[25.0]["uni_mean"])])
_w = ("none", "one", "two", "three", "four", "all")
for tol, t in NET_T.items():
    put("net.n.tol%d" % tol, t["n"], "{:d}")
    put("net.above.tol%d" % tol, int((t["obs"] > t["ker_hi"]).sum()), lambda n: _w[n])
_t25 = NET_T[25.0]
put("net.disc.km", NET_DISC / 1e3, "{:.0f}")
put("net.n.disc", len(NET_P), "{:d}")
put("net.n.all", len(_S), "{:d}")
put("net.share.disc", 100.0 * len(NET_P) / len(_S), "{:.0f}")
put("net.len.km", NET_LEN / 1e3, "{:.0f}")
put("net.nodes", len(NET.xy), "{:d}")
put("net.edges", len(NET.e), "{:d}")
put("net.retrieved", _net["retrieved"][:10])
put("net.moved.median", float(np.median(NET_MOVED)), "{:.1f}")
put("net.moved.p90", float(np.percentile(NET_MOVED, 90)), "{:.0f}")
put("net.draws", NET_NDRAW, "{:d}")
put("net.step", NET_STEP, "{:.0f}")
put("net.csr.lo", float((_t25["uni_mean"] / NET_RADII).min()), "{:.2f}")
put("net.csr.hi", float((_t25["uni_mean"] / NET_RADII).max()), "{:.2f}")
put("net.ratio25.first", float(_t25["obs"][0] / _t25["ker_mean"][0]), "{:.1f}")
put("net.ratio25.last", float(_t25["obs"][-1] / _t25["ker_mean"][-1]), "{:.1f}")
put("net.ratio25.uni.first", float(_t25["obs"][0] / _t25["uni_mean"][0]), "{:.1f}")
put("net.ratio25.uni.last", float(_t25["obs"][-1] / _t25["uni_mean"][-1]), "{:.1f}")
# The prose: uniform placement returns K close to r; at 25 m and 50 m the sites lie above the
# kernel null at all five radii; at 5 m, where fewer sites match, they do not.
assert 0.85 < get("net.csr.lo") and get("net.csr.hi") < 1.15, "the estimator is not calibrated on uniform placement"
assert all((NET_T[tol]["obs"] > NET_T[tol]["ker_hi"]).all() for tol in NET_TOLS[2:]), "a tolerance of 25 m or more falls inside the kernel null"
assert not (NET_T[NET_TOLS[0]]["obs"] > NET_T[NET_TOLS[0]]["ker_hi"]).all(), "the 5 m tolerance is above the null at every radius"
assert NET_T[NET_TOLS[0]]["n"] < 0.6 * len(NET_P)
print({tol: (t["n"], np.round(t["obs"]).tolist(), np.round(t["ker_hi"]).tolist()) for tol, t in NET_T.items()})

# Independent check: the same K from spatstat.linnet on a small part of the graph.
NET_XCK_RADIUS = 600.0         # metres about the core centre
NET_XCK_N = 60
NET_XCK_RADII = np.array([50.0, 100.0, 150.0, 200.0, 300.0])
NET_XCK_R = r'''
suppressPackageStartupMessages(library(spatstat.linnet))
d <- commandArgs(trailingOnly = TRUE)[1]
v <- read.csv(file.path(d, "v.csv")); e <- read.csv(file.path(d, "e.csv")); p <- read.csv(file.path(d, "p.csv"))
W <- owin(range(v$x) + c(-1, 1), range(v$y) + c(-1, 1))
L <- linnet(ppp(v$x, v$y, window = W, check = FALSE), edges = as.matrix(e), sparse = FALSE)
X <- lpp(data.frame(seg = as.integer(p$seg), tp = p$tp), L)
r <- as.numeric(strsplit(commandArgs(trailingOnly = TRUE)[2], ",")[[1]])
K <- linearK(X, r = c(0, r), correction = "Ang")
write.csv(as.data.frame(K), file.path(d, "K.csv"), row.names = FALSE)
writeLines(as.character(packageVersion("spatstat.linnet")), file.path(d, "version.txt"))
'''


def net_xck():
    """K of uniform points on the largest connected part of the graph near the centre, here and in spatstat.linnet."""
    import shutil, tempfile
    from scipy.sparse.csgraph import connected_components
    keep = np.hypot(*(NET.xy - EDG_CENTRE).T) <= NET_XCK_RADIUS
    e = NET.e[keep[NET.e[:, 0]] & keep[NET.e[:, 1]]]
    lab = connected_components(_sp.coo_matrix((np.ones(len(e)), (e[:, 0], e[:, 1])), shape=(len(NET.xy),) * 2),
                               directed=False)[1]
    e = e[lab[e[:, 0]] == np.bincount(lab[np.unique(e)]).argmax()]
    used = np.unique(e)
    remap = -np.ones(len(NET.xy), int)
    remap[used] = np.arange(len(used))
    sub = Net(NET.xy[used], remap[e])
    g = rng("net:xck")
    pe = g.choice(len(sub.e), NET_XCK_N, p=sub.len / sub.len.sum())
    off = g.uniform(0, 1, NET_XCK_N) * sub.len[pe]
    out = {"km": float(sub.len.sum() / 1e3), "nodes": len(sub.xy),
           "K": net_k(sub, pe, off, float(sub.len.sum()), NET_XCK_RADII), "R": None}
    if shutil.which("Rscript") is None:
        return out
    d = pathlib.Path(tempfile.mkdtemp(prefix="net_r_"))
    try:
        np.savetxt(d / "v.csv", sub.xy, delimiter=",", header="x,y", comments="", fmt="%.4f")
        np.savetxt(d / "e.csv", sub.e + 1, delimiter=",", header="a,b", comments="", fmt="%d")
        np.savetxt(d / "p.csv", np.column_stack([pe + 1, off / sub.len[pe]]), delimiter=",", header="seg,tp",
                   comments="", fmt="%.10f")
        (d / "run.R").write_text(NET_XCK_R)
        p = subprocess.run(["Rscript", str(d / "run.R"), str(d), ",".join("%g" % r for r in NET_XCK_RADII)],
                           capture_output=True, text=True)
        if p.returncode != 0:
            print("WARNING: spatstat.linnet failed, network cross-check skipped:", p.stderr[-300:])
            return out
        out["R"] = np.genfromtxt(d / "K.csv", delimiter=",", names=True)["est"][1:]
        out["version"] = (d / "version.txt").read_text().strip()
        return out
    finally:
        shutil.rmtree(d, ignore_errors=True)


NET_XCK = memo("net:xck", net_xck)
put("net.xck.km", NET_XCK["km"], "{:.0f}")
put("net.xck.n", NET_XCK_N, "{:d}")
put("net.xck.radius", NET_XCK_RADIUS, "{:.0f}")
if NET_XCK["R"] is None:
    put("net.xck.maxdiff", "n/a"); put("net.xck.version", "not available")
else:
    _diff = float(np.max(np.abs(NET_XCK["K"] / NET_XCK["R"] - 1.0)))
    put("net.xck.maxdiff", 100.0 * _diff, "{:.4f}")
    put("net.xck.version", NET_XCK["version"])
    assert _diff < 1e-5, "the network K differs from spatstat.linnet: %s vs %s" % (NET_XCK["K"], NET_XCK["R"])
    print("network K against spatstat.linnet %s: largest relative difference %.2e" % (NET_XCK["version"], _diff))

# The count of core-disc sites left unmatched at 25 m and of street edges in the disc.
_k25 = NET_MOVED <= 25.0
put("net.n.unmatched25", int((~_k25).sum()), "{:d}")
put("net.edges.disc", int(_ein.sum()), "{:d}")
assert int(_k25.sum()) == NET_T[25.0]["n"]

/tmp/ipykernel_16/2227444663.py:14: DeprecationWarning: This process (pid=16) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


{5.0: (93, [831.0, 1305.0, 1699.0, 2052.0, 2886.0], [856.0, 1376.0, 1919.0, 2396.0, 3435.0]), 10.0: (129, [1209.0, 1719.0, 2102.0, 2463.0, 3181.0], [721.0, 1224.0, 1740.0, 2262.0, 3202.0]), 25.0: (161, [1359.0, 1825.0, 2215.0, 2541.0, 3196.0], [667.0, 1115.0, 1571.0, 2019.0, 2943.0]), 50.0: (170, [1284.0, 1751.0, 2138.0, 2458.0, 3106.0], [639.0, 1115.0, 1547.0, 2003.0, 2877.0])}
network K against spatstat.linnet 3.5.4: largest relative difference 2.44e-15


### §3.6/§4.3/S19 · The network K in every year of the record

The cell above tests one year. This one repeats the test for each of the
fourteen years inside the same core disc and on the same street graph: the
year's merged sites are matched within 25 m, the geometrically corrected
network K is computed, and it is set against draws from that year's own
kernel intensity placed on the streets. The 2026 row is the 25 m row of the
table above. The cell fills Table `si:linnet:all` and the site counts printed
under the per-year maps.

In [97]:
NET_ALL_TOL = 25.0
WORDS_NET = ("no", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten",
             "eleven", "twelve", "thirteen", "fourteen")


def net_year(y):
    """Network K of one year's merged sites in the core disc and its kernel null on the streets."""
    S = SITES25[y]
    P = S[np.hypot(*(S - EDG_CENTRE).T) <= NET_DISC]
    e, off, moved = NET.snap(P)
    keep = moved <= NET_ALL_TOL
    n = int(keep.sum())
    if y == NET_YEAR:
        t = NET_T[NET_ALL_TOL]
        assert t["n"] == n
        return {"disc": len(P), "n": n, "obs": t["obs"], "ker_mean": t["ker_mean"], "ker_hi": t["ker_hi"]}
    e, off = e[keep], off[keep]
    dd = cKDTree(NET.xy_of(e, off)).query(NET.s_xy[NET_POS], k=min(n, 100))[0]
    piece = NET.s_piece[NET_POS]
    pk = np.exp(-0.5 * (dd / NET_H) ** 2).sum(axis=1) * piece
    pk = pk / pk.sum()

    def draw(g):
        i = NET_POS[g.choice(len(NET_POS), n, replace=False, p=pk)]
        ed = NET.s_edge[i]
        return ed, np.clip(NET.s_off[i] + g.uniform(-0.5, 0.5, n) * NET.s_piece[i], 0.0, NET.len[ed])
    ker = np.array(pmap(lambda j: net_k(NET, *draw(rng("net:year:%d:%d" % (y, j))), NET_LEN), range(NET_NDRAW)))
    return {"disc": len(P), "n": n, "obs": net_k(NET, e, off, NET_LEN), "ker_mean": ker.mean(axis=0),
            "ker_hi": np.percentile(ker, 97.5, axis=0)}


NET_Y = memo("net:years", lambda: {y: net_year(y) for y in YEARS})
NET_ABOVE = {y: int((t["obs"] > t["ker_hi"]).sum()) for y, t in NET_Y.items()}
table("si-linnet-all", [
    "%d & %d & %d & " % (y, t["disc"], t["n"])
    + " & ".join("$%.0f$ / $%.0f$" % (t["obs"][k], t["ker_hi"][k]) for k in range(len(NET_RADII)))
    + " & %d/%d" % (NET_ABOVE[y], len(NET_RADII))
    for y, t in NET_Y.items()])
for y, t in NET_Y.items():
    put("net.yr.%d.disc" % y, t["disc"], "{:d}")
    put("net.yr.%d.n" % y, t["n"], "{:d}")
    put("net.yr.%d.out" % y, t["disc"] - t["n"], "{:d}")
_full = [y for y in YEARS if NET_ABOVE[y] == len(NET_RADII)]
_none = [y for y in YEARS if NET_ABOVE[y] == 0]
_part = [y for y in YEARS if 0 < NET_ABOVE[y] < len(NET_RADII)]
put("net.all.tol", NET_ALL_TOL, "{:.0f}")
put("net.all.years", len(YEARS), lambda n: WORDS_NET[n])
put("net.all.full", len(_full), lambda n: WORDS_NET[n])
put("net.all.full.first", _full[0], "{:d}")
put("net.all.full.last", _full[-1], "{:d}")
put("net.all.small.n", max(NET_Y[y]["n"] for y in _none) if _none else 0, "{:d}")
put("net.all.ratio.lo", min(float(NET_Y[y]["obs"][0] / NET_Y[y]["ker_mean"][0]) for y in _full), "{:.1f}")
put("net.all.ratio.hi", max(float(NET_Y[y]["obs"][0] / NET_Y[y]["ker_mean"][0]) for y in _full), "{:.1f}")
print({y: (t["disc"], t["n"], NET_ABOVE[y], np.round(t["obs"] / t["ker_mean"], 2).tolist()) for y, t in NET_Y.items()})
# The prose: above the null at every radius from 2015 on, at none in the two smallest years.
assert _full == list(range(2015, 2027)) and _none == [2013, 2014] and not _part, (_full, _part, _none)
assert get("net.all.small.n") < 50 and get("net.all.ratio.lo") > 2.0
print("full", _full, "part", _part, "none", _none)

/tmp/ipykernel_16/2227444663.py:14: DeprecationWarning: This process (pid=16) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


{2013: (21, 20, 0, [1.41, 1.89, 2.06, 1.92, 1.67]), 2014: (38, 35, 0, [2.39, 1.6, 1.62, 1.47, 1.27]), 2015: (131, 121, 5, [4.55, 3.59, 2.81, 2.28, 1.76]), 2016: (216, 201, 5, [4.62, 3.29, 2.5, 2.05, 1.71]), 2017: (315, 292, 5, [4.3, 3.16, 2.53, 2.15, 1.81]), 2018: (339, 320, 5, [3.64, 2.69, 2.17, 1.86, 1.57]), 2019: (310, 294, 5, [3.94, 2.88, 2.3, 1.96, 1.62]), 2020: (321, 303, 5, [3.62, 2.75, 2.23, 1.92, 1.6]), 2021: (279, 268, 5, [3.45, 2.45, 2.01, 1.73, 1.44]), 2022: (222, 210, 5, [3.39, 2.4, 1.98, 1.72, 1.45]), 2023: (171, 156, 5, [4.62, 3.07, 2.38, 1.99, 1.6]), 2024: (193, 181, 5, [2.55, 1.87, 1.57, 1.39, 1.2]), 2025: (186, 173, 5, [2.85, 1.99, 1.63, 1.44, 1.24]), 2026: (174, 161, 5, [3.27, 2.19, 1.77, 1.53, 1.3])}
full [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026] part [] none [2013, 2014]


### S19 · The orientation test with the geometry of each group kept

The conditional null gives every pair its own orientation, which no set of
points can realise once pairs share a vertex. Two checks keep the geometry.
The first collapses the groups at a coarser radius and repeats the test.
The second leaves the pattern as it is, splits it into single-linkage blocks
and turns each block rigidly by an independent angle, so every triangle
inside a block is preserved and only the orientation of one block against
another is randomised.

In [98]:
DIR_NBLOCK = 1999                       # block-rotation draws
DIR_COLLAPSE = (100.0, 250.0)           # collapse radii, metres
DIR_BLOCKS = (("loc", "Exact locations", 250.0), ("loc", "Exact locations", 500.0),
              ("loc", "Exact locations", DIR_R), ("site25", "Merged \\SI{25}{\\meter}", DIR_R))

for _r in DIR_COLLAPSE:
    _S = merge_centroid(DIR_P, _r)
    _pr, _ang, _ni, _ins = dir_pairs(_S)
    _t = dir_test(_ang, DIR_NMAIN, rng("dir:collapse:%d" % _r))
    _key = "dir.col%d" % _r
    put(_key + ".sites", len(_S), "{:d}")
    put(_key + ".pairs", len(_ang), "{:d}")
    put(_key + ".ratio", _t["ratio"], "{:.2f}")
    put(_key + ".mean", _t["mean"], "{:.2f}")
    put(_key + ".p", _t["p"], "{:.3f}")
    put(_key + ".tri", dir_sharing(_pr, _ins)[1], pct)
    print("collapsed at %g m: %d sites, %d pairs, ratio %.2f, p %.3f" % (_r, len(_S), len(_ang), _t["ratio"], _t["p"]))
assert get("dir.col250.tri") > 0.5, "collapsing at 250 m removes most triangles"   # the prose says it does not


def dir_block_test(P, block_r, tag):
    """Sector ratio of the pairs inside single-linkage blocks against independent rigid turns of the blocks."""
    pr, ang, ni, ins = dir_pairs(P)
    lab = single_labels(P, block_r)
    same = lab[pr[:, 0]] == lab[pr[:, 1]]
    a_in = ang[same]
    ub, inv = np.unique(lab[pr[same, 0]], return_inverse=True)
    obs = dir_ratio(dir_hist(a_in))
    g = rng(tag)
    sim = np.array([dir_ratio(dir_hist((a_in + g.uniform(0, math.pi, len(ub))[inv]) % math.pi))
                    for _ in range(DIR_NBLOCK)])
    w = np.bincount(inv)
    ge = int((sim >= obs).sum())
    return {"blocks": len(ub), "pairs": int(same.sum()), "all": len(ang), "top": float(w.max() / w.sum()),
            "ratio": obs, "mean": float(sim.mean()), "hi": float(np.percentile(sim, 97.5)),
            "ge": ge, "p": (ge + 1) / (DIR_NBLOCK + 1)}


DIR_BLOCK = [(u, label, r, dir_block_test(UNITS[u][DIR_YEAR], r, "dir:block:%s:%d" % (u, r)))
             for u, label, r in DIR_BLOCKS]
table("si-blockrot", ["%s & %.0f & %d & %d & %.1f & %.2f & %.2f & %.2f & %.3f" % (
    label, r, b["blocks"], b["pairs"], 100.0 * b["top"], b["ratio"], b["mean"], b["hi"], b["p"])
    for u, label, r, b in DIR_BLOCK])
_b1k = next(b for u, _, r, b in DIR_BLOCK if u == "loc" and r == DIR_R)
assert _b1k["pairs"] == _b1k["all"] == get("dir.n.pairs"), "a counted pair joins two blocks at the pair radius"
put("dir.block.draws", DIR_NBLOCK, "{:d}")
put("dir.block.p1k", _b1k["p"], "{:.2f}")
put("dir.block.blocks1k", _b1k["blocks"], "{:d}")
put("dir.block.top1k", 100.0 * _b1k["top"], "{:.0f}")
put("dir.block.mean1k", _b1k["mean"], "{:.2f}")
put("dir.block.hi1k", _b1k["hi"], "{:.2f}")
put("dir.block.pmin", min(b["p"] for _, _, _, b in DIR_BLOCK), "{:.2f}")
put("dir.block.pmax", max(b["p"] for _, _, _, b in DIR_BLOCK), "{:.2f}")
# The prose: the imbalance is not unusual once each block keeps its geometry, at any block radius.
assert min(b["p"] for _, _, _, b in DIR_BLOCK) > 0.05, "a block-rotation test rejects"
print("block rotation:", [(u, r, round(b["p"], 3), round(b["top"], 3)) for u, _, r, b in DIR_BLOCK])

collapsed at 100 m: 338 sites, 861 pairs, ratio 1.57, p 0.024
collapsed at 250 m: 272 sites, 341 pairs, ratio 1.48, p 0.612
block rotation: [('loc', 250.0, 0.278, 0.224), ('loc', 500.0, 0.15, 0.325), ('loc', 1000.0, 0.248, 0.722), ('site25', 1000.0, 0.795, 0.766)]


### S19 · The two theme subsets on their own hulls, against their own kernel nulls

The comparison above uses the fixed hull and a homogeneous control. Here each
subset is measured on its own convex hull against draws from its own
Gaussian-kernel intensity, kept inside that hull, which is the null the
headline multiscale test uses.

In [99]:
ROAD_NOWN = 199      # kernel-null draws per subset
ROAD_H = 750.0       # kernel bandwidth, metres


def road_own(name, P):
    poly, area = hull_of(P)
    draw = msn_kernel_sampler(P, ROAD_H)

    def inside_draw(g):
        out = np.empty((0, 2))
        while len(out) < len(P):
            Q = draw(g)
            out = np.vstack([out, Q[in_hull(Q, poly)]])
        return out[:len(P)]
    sim = np.array(pmap(lambda k: L_border(inside_draw(rng("dir:road-own:%s:%d" % (name, k))), ROAD_RADII, poly, area),
                        range(ROAD_NOWN)))
    return {"n": len(P), "area": area, "obs": L_border(P, ROAD_RADII, poly, area),
            "hi": np.nanpercentile(sim, 97.5, axis=0), "mean": np.nanmean(sim, axis=0)}


ROAD_OWN = {"veh": road_own("veh", ROAD_VEH), "non": road_own("non", ROAD_NON)}
table("si-roadown", [
    "%s & %d & %.0f & " % (label, ROAD_OWN[k]["n"], ROAD_OWN[k]["area"] / 1e6)
    + " & ".join("$%.0f$ / $%.0f$" % (ROAD_OWN[k]["obs"][j], ROAD_OWN[k]["hi"][j]) for j in range(len(ROAD_RADII)))
    for k, label in (("veh", "Vehicles"), ("non", "Other themes"))])
for k in ROAD_OWN:
    put("dir.road.own.%s.km2" % k, ROAD_OWN[k]["area"] / 1e6, "{:.0f}")
    put("dir.road.own.%s.share" % k, ROAD_OWN[k]["area"] / A_HULL, "{:.2f}")
    for j in range(len(ROAD_RADII)):
        put("dir.road.own.%s.ex%d" % (k, j + 1), float(ROAD_OWN[k]["obs"][j] - ROAD_OWN[k]["mean"][j]), "{:.0f}")
put("dir.road.own.draws", ROAD_NOWN, "{:d}")
assert all((ROAD_OWN[k]["obs"] > ROAD_OWN[k]["hi"]).all() for k in ROAD_OWN), "a subset is inside its own kernel null"
assert ROAD_OWN["non"]["area"] > 2 * ROAD_OWN["veh"]["area"], "the two subsets occupy similar ground"
# The prose: on their own hulls the other themes keep the larger excess in metres at every radius.
assert ((ROAD_OWN["non"]["obs"] - ROAD_OWN["non"]["mean"]) > (ROAD_OWN["veh"]["obs"] - ROAD_OWN["veh"]["mean"])).all()
# Which locations the two named themes hold: none is shared with a vehicle identifier.
_vehset = {tuple(p) for p in ROAD_VEH}
_peo = locs_xy(DIR_YEAR, lambda r: r["theme"] == "People")
_aq = locs_xy(DIR_YEAR, lambda r: r["theme"] == "Air Quality")
put("dir.road.shared", len(_vehset & {tuple(p) for p in ROAD_NON}), "{:d}")
assert len(ROAD_VEH) + len(ROAD_NON) == len(LOCS[DIR_YEAR]) and get("dir.road.shared") == 0
assert not (_vehset & {tuple(p) for p in _peo}) and not (_vehset & {tuple(p) for p in _aq})

/tmp/ipykernel_16/2227444663.py:14: DeprecationWarning: This process (pid=16) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


## §2.1/§2.2/§3.1/§3.2/§4/§5 · The census, the headline table and the figures

This section sets the numbers the main text states about the archive and the
roster: how many files and readings the archive holds, how many identifiers
and locations the census keeps, how the roster turns over, and the windows
the indices are measured in. It then builds the headline table and draws the
five figures of the main text.

In [100]:
WORDS = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten",
         "eleven", "twelve", "thirteen", "fourteen", "fifteen", "sixteen", "seventeen",
         "eighteen", "nineteen", "twenty"]


def word(n):
    """A small count as a word, as running text prints it."""
    return WORDS[int(n)]


def whole(x):
    return "{:.0f}".format(x)

### §2.1 · The archive (files, readings, size)

The file count, the size and the number of readings come from the census of
section 1. A file the coordinate census cannot read is one that holds no
identifier.

In [101]:
FILES = CENSUS.get("files") or []
if FILES:
    UNREAD = [f for f in FILES if f["n_ids"] == 0]
else:
    # The development census carries no file list; read the zip directories instead.
    UNREAD = []
    for _z in sorted(ARCHIVE.rglob("*.csv.zip")):
        with zipfile.ZipFile(_z) as _zf:
            if sum(m.file_size for m in _zf.infolist() if m.filename.endswith(".csv")) == 0:
                UNREAD.append({"path": str(_z.relative_to(ARCHIVE)), "bytes": _z.stat().st_size,
                               "theme": _z.relative_to(ARCHIVE).parts[1]})
assert len(UNREAD) == 1 and UNREAD[0]["theme"] == "Vehicles", UNREAD  # the text names one empty vehicle file

put("cen.files.total", CENSUS["total_files"], "{:d}", macro="nArchiveFiles")
put("cen.files.read", CENSUS["total_files"] - len(UNREAD), "{:d}")
put("cen.files.emptybytes", UNREAD[0]["bytes"], "{:d}")
put("cen.archive.gb", CENSUS["total_bytes"] / 1e9, "{:.1f}", macro="nArchiveGB")
put("cen.archive.readings", CENSUS["total_rows"], "{:d}", macro="nArchiveReadings")
put("cen.grant", "26UQU4350624GSSR07", macro="uquGrant")

'26UQU4350624GSSR07'

### §2.2 · Identifiers, locations and co-location

An identifier is in range when it lies within 60 km of the projection origin
on both axes. The shared and surplus counts are stated on one basis: every
coordinate an identifier holds in any year, rounded to 0.1 m. An identifier
is *shared* when one of its coordinates is also held by another identifier,
and the *surplus* is identifiers minus distinct coordinates.

In [102]:
IN_RANGE = set().union(*[set(REC[y]) for y in YEARS])
assert not N_NOCOORD, len(N_NOCOORD)          # the text says every identifier has a coordinate
assert N_RAW_IDS - IN_RANGE == N_OUT - IN_RANGE

COORDS_OF = defaultdict(set)                  # identifier -> coordinates held in any year
for _y in YEARS:
    for _i, _r in REC[_y].items():
        COORDS_OF[_i].add((round(_r["x"], 1), round(_r["y"], 1)))
_occ = Counter(c for s in COORDS_OF.values() for c in s)
N_COORDS = len(_occ)
N_SHARED = sum(1 for s in COORDS_OF.values() if any(_occ[c] > 1 for c in s))
N_SURPLUS = len(IN_RANGE) - N_COORDS

put("cen.ids.raw", len(N_RAW_IDS), "{:d}")
put("cen.ids.out", len(N_RAW_IDS - IN_RANGE), word)
put("cen.ids.inrange", len(IN_RANGE), "{:d}", macro="nAnalysed")
put("cen.locs.record", N_COORDS, "{:d}", macro="nLocations")
put("cen.shared.n", N_SHARED, "{:d}", macro="nShared")
put("cen.shared.pct", N_SHARED / len(IN_RANGE), pct, macro="sharePct")
put("cen.surplus.n", N_SURPLUS, "{:d}", macro="nSurplus")
put("cen.surplus.pct", N_SURPLUS / len(IN_RANGE), pct, macro="surplusPct")

put("cen.n.ids2013", int(N_IDS[0]), "{:d}", macro="nFirst")
put("cen.n.ids2026", int(N_IDS[-1]), "{:d}", macro="nLast")
put("cen.n.loc2013", int(N_LOCS[0]), "{:d}")
put("cen.n.loc2026", int(N_LOCS[-1]), "{:d}")
put("cen.growth.ids", N_IDS[-1] / N_IDS[0], "{:.1f}", macro="growth")
put("cen.growth.loc", N_LOCS[-1] / N_LOCS[0], "{:.1f}")

13.116279069767442

### §2.2 · Identifiers that change coordinate

The record-wide count of distinct coordinates takes every coordinate an
identifier holds in any year, so a re-sited identifier contributes more
than one. Those identifiers are counted, with the distance they moved, and
so are the identifier-years in which the archive records more than one
coordinate, where the census keeps the first in file order.

In [103]:
_moved = {i: s for i, s in COORDS_OF.items() if len(s) > 1}
_dist = sorted(max(math.dist(a, b) for a in s for b in s) for s in _moved.values())
_first = {}
for _y in YEARS:
    for _i, _r in REC[_y].items():
        _first.setdefault(_i, (round(_r["x"], 1), round(_r["y"], 1)))
put("cen.resited.n", len(_moved), word)
put("cen.resited.dmin", _dist[0], "{:.1f}")
put("cen.resited.dmed", float(np.median(_dist)) / 1e3, "{:.1f}")
put("cen.resited.dmax", _dist[-1] / 1e3, "{:.1f}")
put("cen.resited.extra", sum(len(s) for s in COORDS_OF.values()) - len(COORDS_OF), "{:d}")
put("cen.coords.first", len(set(_first.values())), "{:d}")
put("cen.idyears", sum(len(REC[y]) for y in YEARS), "{:d}")
put("cen.multicoord.idyears", sum(1 for y in YEARS for r in REC[y].values() if len(r["coords"]) > 1), "{:d}")
put("cen.multicoord.ids", len({i for y in YEARS for i, r in REC[y].items() if len(r["coords"]) > 1}), "{:d}")
assert 0 < len(_moved) < 0.02 * len(IN_RANGE), "re-sited identifiers are no longer a small minority"
assert len(dedup(np.vstack([LOCS[y] for y in YEARS]))) == N_COORDS, "the pooled locations differ from the record-wide coordinate count"
print("re-sited identifiers %d, displacement %.1f m to %.1f km" % (len(_moved), _dist[0], _dist[-1] / 1e3))

re-sited identifiers 14, displacement 1.2 m to 10.8 km


### §3.2 · The roster: arrivals, departures, replacement rate

An identifier is present in a year when it appears in any file of that year.
For each of the thirteen transitions, the entries are the identifiers present
in a year and absent the year before, and the exits are those present the
year before and absent now. The replacement rate is twice the smaller of the
two over their sum. The same count is made on the location roster.

In [104]:
PRESENT = {i: [y for y in YEARS if i in REC[y]] for i in IN_RANGE}
put("cen.roster.gone", sum(1 for v in PRESENT.values() if v[-1] < YEARS[-1]), "{:d}")
put("cen.roster.active", sum(1 for v in PRESENT.values() if v[-1] == YEARS[-1]), "{:d}")
put("cen.roster.gap", sum(1 for v in PRESENT.values() if v[-1] - v[0] + 1 > len(v)), "{:d}")
put("cen.roster.single", sum(1 for v in PRESENT.values() if len(v) == 1), "{:d}")
put("cen.roster.multi", sum(1 for v in PRESENT.values() if len(v) > 1), "{:d}")
assert get("cen.roster.single") + get("cen.roster.multi") == len(IN_RANGE)
assert get("cen.roster.active") == N_IDS[-1]


def transitions(sets):
    """Entries, exits and the replacement rate for each year after the first."""
    rows = []
    for a, b in zip(YEARS[:-1], YEARS[1:]):
        n_in, n_out = len(sets[b] - sets[a]), len(sets[a] - sets[b])
        rows.append({"year": b, "in": n_in, "out": n_out,
                     "rate": 2.0 * min(n_in, n_out) / (n_in + n_out)})
    return rows


TURN_IDS = transitions({y: set(REC[y]) for y in YEARS})
TURN_LOC = transitions({y: set(map(tuple, LOCS[y])) for y in YEARS})
_lo = min(TURN_IDS, key=lambda r: r["rate"])
_hi = max(TURN_IDS, key=lambda r: r["rate"])
put("cen.repl.n", len(TURN_IDS), word)
put("cen.repl.median", float(np.median([r["rate"] for r in TURN_IDS])), "{:.2f}", macro="replMedian")
put("cen.repl.minyear", _lo["year"], "{:d}")
put("cen.repl.min", _lo["rate"], "{:.2f}")
put("cen.repl.maxyear", _hi["year"], "{:d}")
put("cen.repl.max", _hi["rate"], "{:.2f}")
put("cen.repl.maxin", _hi["in"], "{:d}")
put("cen.repl.maxout", _hi["out"], "{:d}")
put("cen.repl.locmedian", float(np.median([r["rate"] for r in TURN_LOC])), "{:.2f}")

0.5128205128205128

### §3.2/Abstract · The shape of the population series

The population is not a ramp. This cell records the peak, the change since
the peak, how long an identifier stays on the roster, and the transitions in
which exits exceed entries. The assertions state what the text says about
them, so the run stops if the data stop supporting a sentence.

In [105]:
_k = int(np.argmax(N_IDS))
put("cen.peak.year", YEARS[_k], "{:d}")
put("cen.peak.ids", int(N_IDS[_k]), "{:d}")
put("cen.peak.factor", N_IDS[_k] / N_IDS[0], "{:.1f}")
put("cen.peak.decline", 100.0 * (1.0 - N_IDS[-1] / N_IDS[_k]), whole)
put("cen.peak.loc", int(N_LOCS.max()), "{:d}")
assert YEARS[int(np.argmax(N_LOCS))] == YEARS[_k]                  # both units peak in the same year
assert 0 < _k < len(YEARS) - 1 and N_IDS[-1] < N_IDS[_k]           # the peak is inside the record
assert all(np.diff(N_IDS[:_k + 1]) > 0)                            # the count rises every year up to the peak

_len = [len(v) for v in PRESENT.values()]
put("cen.roster.allyears", sum(1 for n in _len if n == len(YEARS)), word)
put("cen.roster.medyears", int(np.median(_len)), word)
assert float(np.median(_len)) == int(np.median(_len))
put("cen.roster.gonepct", 100.0 * get("cen.roster.gone") / len(IN_RANGE), whole)

_loss = [r["year"] for r in TURN_IDS if r["out"] > r["in"]]
put("cen.net.loss.n", len(_loss), word)
assert _loss and min(_loss) > YEARS[_k]                            # every net loss comes after the peak
assert _lo["in"] > 5 * _lo["out"]                                  # the lowest rate is a growth year
assert abs(_hi["in"] - _hi["out"]) / N_IDS[YEARS.index(_hi["year"])] < 0.05   # the highest is a flat year
assert get("cen.repl.locmedian") > get("cen.repl.median")          # the location roster turns over more evenly

# A transition in which the count rises while the footprint contracts.
_cand = [(A_OWN[k + 1] / A_OWN[k] - 1.0, k) for k in range(len(YEARS) - 1) if N_IDS[k + 1] > N_IDS[k]]
_shrink, _kk = min(_cand)
assert _shrink < -0.10
put("cen.foot.from", YEARS[_kk], "{:d}")
put("cen.foot.to", YEARS[_kk + 1], "{:d}")
put("cen.foot.idsup", 100.0 * (N_IDS[_kk + 1] / N_IDS[_kk] - 1.0), whole)
put("cen.foot.hulldown", -100.0 * _shrink, whole)

# Claims the abstract, the highlights and the introduction make outside their own sections.
assert ols(Y, R_SITE25)["p"] > 0.05                                # no fitted trend on the merged unit
assert all(ols(Y, np.array([eg_index(LOCS[y], k) for y in YEARS]))["slope"] < 0 for k in (20, 30, 40, 50))
assert ols(Y, rarefied("loc").mean(axis=1))["p"] > 0.05            # rarefaction removes the fitted trend

### §2.2 · The projection and the fixed-hull window

The extent is that of all in-range identifiers. The scale deviation is the
relative change of the east-west scale between the central parallel and the
northernmost and southernmost identifiers. The bounding-box values of the
index are those of the identifiers and of the locations on each year's own box.

In [106]:
_ext = ALL_XY.max(axis=0) - ALL_XY.min(axis=0)
put("cen.win.ew", _ext[0] / 1e3, whole, macro="winEW")
put("cen.win.ns", _ext[1] / 1e3, whole, macro="winNS")
_lats = [float(r["lat"]) for y in YEARS for r in REC[y].values()]
_dev = max(abs(math.cos(math.radians(l)) / math.cos(math.radians(LAT0)) - 1.0) for l in (min(_lats), max(_lats)))
put("cen.proj.dev", _dev, pct)

put("cen.r.idsbox2013", R_IDS_BOX[0], "{:.3f}")
put("cen.r.idsbox2026", R_IDS_BOX[-1], "{:.3f}")
put("cen.r.idsboxfall", 100.0 * fall(R_IDS_BOX), whole, macro="rDrop")
put("cen.r.locbox2013", R_LOC_BOX[0], "{:.3f}", macro="locFirst")
put("cen.r.locbox2026", R_LOC_BOX[-1], "{:.3f}")

0.3141068441366101

### §3.1/Tab. 2 · The headline table

Each row states one quantity on its frame and unit: the first and last year,
the endpoint change in whole per cent, and the nominal OLS p-value of the
fitted trend. The last row is the exponent of the mean nearest-neighbour
distance on the location count, tested against minus one half with the
Newey-West standard error.

In [107]:
def _dbar(unit):
    return np.array([float(np.mean(nn_dist(UNITS[unit][y]))) for y in YEARS])


def scaling(unit, areas=None):
    """Exponent of the mean nearest-neighbour distance on the count, its se, and t against -1/2."""
    n = np.array([len(UNITS[unit][y]) for y in YEARS], float)
    d = _dbar(unit) if areas is None else _dbar(unit) / np.sqrt(areas)
    b, se, _, _ = hac(np.log(n), np.log(d))
    return b, se, (b + 0.5) / se


def _r_row(unit_name, v, frame="fixed hull"):
    return (r"Clustering $R$ & %s & %s & %.3f & %.3f & $-%s\%%$ & OLS $p%s$"
            % (frame, unit_name, v[0], v[-1], whole(100.0 * fall(v)),
               "{<}0.001" if ols(Y, v)["p"] < 0.001 else "{=}%.3f" % ols(Y, v)["p"]))


_b, _se, _t = scaling("loc")
table("tab-summary", [
    r"Population & -- & identifiers & %d & %d & $%.1f\times$ & census" % (N_IDS[0], N_IDS[-1], N_IDS[-1] / N_IDS[0]),
    r"Population & -- & locations & %d & %d & $%.1f\times$ & census" % (N_LOCS[0], N_LOCS[-1], N_LOCS[-1] / N_LOCS[0]),
    _r_row("locations", R_LOC),
    _r_row("identifiers", R_IDS),
    _r_row("identifiers", R_IDS_BOX, "own box"),
    _r_row("merged sites", R_SITE25),
    r"$\ln\bar{d}$ vs $\ln N$ & fixed hull & locations & \multicolumn{3}{c}{exponent $%.3f$, vs $-\tfrac12$} & $t{=}%.2f$"
    % (_b, _t),
])
put("cen.hl.fall", 100.0 * fall(R_LOC), whole)
put("cen.hl.merged", 100.0 * fall(R_SITE25), whole)

15.801385172722787

### §4.1/§4.2/§5 · Numbers the main text repeats outside the results

Two quantities are quoted in the introduction, the limitations and the
conclusions. The exponent with each year's own hull area divided out of the
distance, and the nearest-neighbour distance the rarefied sample can resolve,
which is the complete-spatial-randomness expectation at the rarefied size on
the fixed hull.

In [108]:
put("cen.scale.own", scaling("loc", A_OWN)[0], "{:.3f}")
put("cen.rare.scalekm", 0.5 * math.sqrt(A_HULL / N_LOCS.min()) / 1e3, whole)

5.083972523730565

### Figs. 1–5 · The five main-text figures

The five figures of the main text are drawn from the series above and saved
under the names the document includes. The rarefied concentration series of
the fourth figure is stored under the tag `gini:rare:20` as draws by year, so
a section that needs the same draws reads the same array.

In [109]:
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon

FIG_RC = {"font.size": 9, "axes.grid": True, "grid.alpha": 0.3, "figure.dpi": 300,
          "savefig.bbox": "tight", "axes.axisbelow": True}
GINI_K = 20
N_GINI_DRAWS = 1000
TOL_SWEEP = [1.0, 2.0, 5.0, 10.0, 15.0, 25.0, 50.0]
TOL_USED = 25.0
NN_YEARS = [(2020, "#dd8452"), (2026, "#4c72b0")]


def _finish(fig, name):
    fig.savefig(FIGDIR / name)
    plt.close(fig)
    print("wrote figures/%s" % name)


# Figure: the study area and each year's bounding box as a share of the hull.
fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.6))
ax = axes[0]
ax.add_patch(MplPolygon(HULL_POLY / 1e3, closed=True, facecolor="#f2f2f2", edgecolor="#888888", lw=1.0,
                        zorder=1, label="fixed reference hull (%.0f km$^2$)" % (A_HULL / 1e6)))
ax.scatter(LOCS[2026][:, 0] / 1e3, LOCS[2026][:, 1] / 1e3, s=9, c="#1f77b4", alpha=0.55, lw=0,
           label="2026 locations (%d)" % len(LOCS[2026]), zorder=2)
ax.scatter(LOCS[2013][:, 0] / 1e3, LOCS[2013][:, 1] / 1e3, s=46, c="#d62728", marker="^", alpha=0.95,
           edgecolor="white", lw=0.6, label="2013 locations (%d)" % len(LOCS[2013]), zorder=3)
ax.set_aspect("equal")
ax.set_xlabel("east of projection origin (km)")
ax.set_ylabel("north of projection origin (km)")
ax.set_title("(a)", fontsize=8.4, loc="left")
ax.legend(loc="upper left", fontsize=7, framealpha=0.9)
ax = axes[1]
_frac = 100.0 * A_BOX / A_HULL
ax.plot(YEARS, _frac, "o-", color="#2ca02c", lw=1.4, ms=4)
ax.axhline(100.0, color="#888888", ls="--", lw=1.0)
ax.text(YEARS[0], 103, "the hull itself", fontsize=7, color="#666666")
ax.set_xlabel("year")
ax.set_ylabel("per-year bounding box, % of hull area")
ax.set_title("(b)", fontsize=8.4, loc="left")
ax.set_ylim(0, max(120, 1.08 * _frac.max()))
ax.grid(alpha=0.25, lw=0.5)
for _k in (0, int(np.argmax(_frac))):
    ax.annotate("%.0f%%" % _frac[_k], (YEARS[_k], _frac[_k]), textcoords="offset points",
                xytext=(0, 7), fontsize=7, ha="center")
fig.tight_layout(rect=[0, 0, 1, 0.98])
fig.savefig(FIGDIR / "fig2_study_area.pdf", bbox_inches="tight")
plt.close(fig)
print("wrote figures/fig2_study_area.pdf")

# Figure: the population with yearly entries and exits.
_ins = [0] + [r["in"] for r in TURN_IDS]
_outs = [0] + [-r["out"] for r in TURN_IDS]
fig, ax = plt.subplots(figsize=(6.6, 3.4))
ax.plot(YEARS, N_IDS, "o-", color="#4c72b0", lw=1.7, ms=4.5, label="instruments reporting")
ax.bar([y - 0.16 for y in YEARS], _ins, width=0.32, color="#8fbc8f", alpha=0.75, label="entries (incl. re-entries)")
ax.bar([y + 0.16 for y in YEARS], _outs, width=0.32, color="#d98b8b", alpha=0.75, label="exits")
ax.axhline(0, color="k", lw=0.7)
ax.set_xlabel("year")
ax.set_ylabel("instruments")
ax.set_ylim(min(_outs) * 1.12, max(N_IDS.max(), max(_ins)) * 1.10)
ax.grid(alpha=0.25, lw=0.5, axis="y")
ax.legend(fontsize=7, loc="upper left", frameon=False)
fig.tight_layout()
fig.savefig(FIGDIR / "fig1_population.pdf", bbox_inches="tight")
plt.close(fig)
print("wrote figures/fig1_population.pdf")

# Figure: the clustering index on the fixed hull, both units, with the rarefied means.
with plt.rc_context(FIG_RC):
    fig, ax = plt.subplots(figsize=(5.6, 3.3))
    _rids, _rloc = rarefied("ids"), rarefied("loc")
    ax.plot(Y, _rids.mean(axis=1), color="0.7", lw=1.0, ls=":", marker="o", ms=2.6, zorder=2,
            label="rarefied to $N=%d$ (ids)" % N_IDS.min())
    ax.plot(Y, _rloc.mean(axis=1), color="#55a868", lw=1.2, ls=":", marker="^", ms=2.8, zorder=2,
            label="rarefied to $N=%d$ (locs, headline)" % N_LOCS.min())
    ax.plot(Y, R_IDS, color="#c44e52", lw=1.6, marker="o", ms=3.2, zorder=3, label="identifiers (hull)")
    ax.plot(Y, R_LOC, color="#4c72b0", lw=2.0, marker="s", ms=3.2, zorder=4, label="locations (hull, headline)")
    for _v, _c in ((R_IDS, "#c44e52"), (R_LOC, "#4c72b0")):
        ax.plot(Y, _v[0] + hac(Y, _v)[0] * (Y - Y[0]), color=_c, lw=0.9, ls="--", alpha=0.7, zorder=3)
    ax.axhline(1.0, color="k", lw=0.8)
    ax.annotate("randomness", xy=(2013.2, 1.02), fontsize=7, color="0.3")
    ax.set_xlabel("year")
    ax.set_ylabel("Clark–Evans $R$")
    ax.set_ylim(0, 1.15)
    ax.legend(fontsize=7, loc="upper right")
    fig.tight_layout()
    _finish(fig, "fig3_clustering.pdf")


# Figure: the concentration index, uncorrected and rarefied to the smallest year.
def _gini_draws(i):
    P, n = LOCS[YEARS[i]], int(N_LOCS.min())
    if len(P) == n:
        return [gini_occupied(P, GINI_K)] * N_GINI_DRAWS
    g = rng("gini:rare:%d:%d" % (GINI_K, i))
    return [gini_occupied(P[g.choice(len(P), n, replace=False)], GINI_K) for _ in range(N_GINI_DRAWS)]


GINI_RARE = np.asarray(memo("gini:rare:%d" % GINI_K, lambda: np.array(pmap(_gini_draws, range(len(YEARS))))))
assert GINI_RARE.shape[0] == len(YEARS)
with plt.rc_context(FIG_RC):
    fig, ax = plt.subplots(figsize=(5.4, 3.2))
    ax.plot(YEARS, [gini_occupied(LOCS[y], GINI_K) for y in YEARS], "s--", color="0.65", lw=1.2, ms=3,
            label="uncorrected")
    ax.fill_between(YEARS, np.percentile(GINI_RARE, 2.5, axis=1), np.percentile(GINI_RARE, 97.5, axis=1),
                    color="#4c72b0", alpha=0.20, label="rarefied interval")
    ax.plot(YEARS, GINI_RARE.mean(axis=1), "o-", color="#4c72b0", lw=1.8, ms=4,
            label="rarefied to %d locations" % N_LOCS.min())
    ax.set_xlabel("year")
    ax.set_ylabel("Gini of occupied cells (%d x %d, fixed grid)" % (GINI_K, GINI_K))
    ax.set_ylim(0, 1)
    ax.set_xlim(YEARS[0] - 0.5, YEARS[-1] + 0.5)
    ax.set_xticks(YEARS)
    ax.set_xticklabels([str(y) for y in YEARS], fontsize=7, rotation=45)
    ax.legend(fontsize=6.6, loc="upper left")
    fig.tight_layout()
    _finish(fig, "fig5_gini_ci.pdf")

# Figure: the fall and its p-value by merge radius, and nearest-neighbour distances.
_sweep = [np.array([ce_r(merge_single(IDS[y], r), A_HULL) for y in YEARS]) for r in TOL_SWEEP]
_falls = [100.0 * fall(v) for v in _sweep]
_ps = [ols(Y, v)["p"] for v in _sweep]
with plt.rc_context(FIG_RC):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.4, 3.1))
    ax1.plot(TOL_SWEEP, _falls, "o-", color="#4c72b0", lw=1.8, ms=4.5)
    ax1.axhline(0, color="k", lw=0.9)
    ax1.axvline(TOL_USED, color="#c44e52", lw=1.2, ls="--")
    ax1.annotate("%.0f m, used in the text" % TOL_USED, xy=(1.15, -11), fontsize=6.6, color="#c44e52",
                 ha="left", va="bottom")
    ax1.set_xscale("log")
    ax1.set_xlabel("merge tolerance (m)")
    ax1.set_ylabel("fall in $R$ (%)")
    ax1.set_ylim(min(-14, min(_falls) - 4), max(52, max(_falls) + 4))
    ax1b = ax1.twinx()
    ax1b.plot(TOL_SWEEP, _ps, "s--", color="#55a868", lw=1.3, ms=3.5, alpha=0.9)
    ax1b.axhline(0.05, color="#55a868", lw=0.8, ls=":", alpha=0.7)
    ax1b.set_ylabel("$p$ (dashed, right)", color="#55a868")
    ax1b.tick_params(axis="y", colors="#55a868")
    ax1b.set_yscale("log")
    ax1b.set_ylim(min(0.02, 0.8 * min(_ps)), 1.2)
    ax1.grid(False)
    ax1b.grid(False)
    ax1.set_title("(a)", fontsize=8.4)
    _edges = np.arange(0.5, 300.5, 1.0)
    for _y, _c in NN_YEARS:
        _h, _ = np.histogram(nn_dist(IDS[_y]), bins=_edges)
        ax2.step(_edges[:-1], _h, where="post", color=_c, lw=1.6, label=str(_y))
    ax2.set_xscale("log")
    ax2.axvline(TOL_USED, color="#c44e52", lw=1.2, ls="--")
    ax2.set_xlabel("nearest-neighbour distance (m)")
    ax2.set_ylabel("identifiers")
    ax2.legend(fontsize=7)
    ax2.set_title("(b)", fontsize=8.4)
    fig.tight_layout()
    _finish(fig, "fig4_tolerance.pdf")

wrote figures/fig2_study_area.pdf
wrote figures/fig1_population.pdf
wrote figures/fig3_clustering.pdf


/tmp/ipykernel_16/2227444663.py:14: DeprecationWarning: This process (pid=16) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


wrote figures/fig5_gini_ci.pdf
wrote figures/fig4_tolerance.pdf


### Graphical abstract (51% → 16%)

Three panels: population bars, clustering series with the measurement artefact
shaded, and the headline numbers.

In [110]:
_fig = plt.figure(figsize=(16, 5.2), dpi=300)
_gs = _fig.add_gridspec(1, 3, width_ratios=[1.1, 1.8, 0.7], wspace=0.35)

_ax1 = _fig.add_subplot(_gs[0])
_ax1.bar(Y, N_IDS, color="#999", edgecolor="white", lw=0.5, zorder=3)
_ax1.set_xlabel("Year"); _ax1.set_ylabel("Identifiers")
_ax1.set_title("Sensor population", fontweight="bold")
_pk = int(np.argmax(N_IDS))
for _ix, _lab in [(0, str(int(N_IDS[0]))), (_pk, str(int(N_IDS[_pk]))), (-1, str(int(N_IDS[-1])))]:
    _ax1.annotate(_lab, (Y[_ix], N_IDS[_ix]), textcoords="offset points",
                  xytext=(0, 5), ha="center", fontsize=8, fontweight="bold")
_ax1.set_ylim(0, int(N_IDS.max() * 1.15))
_ax1.grid(axis="y", alpha=0.3); _ax1.set_axisbelow(True)

_ax2 = _fig.add_subplot(_gs[1])
_ax2.fill_between(Y, R_SITE25, R_LOC, color="#e8c0c0", alpha=0.4,
                  label="Measurement artefact", zorder=1)
_ax2.plot(Y, R_LOC, "o-", color="#c44e52", lw=2, ms=5,
          label="Published coordinates", zorder=3)
_ax2.plot(Y, R_SITE25, "s-", color="#4c72b0", lw=2, ms=5,
          label="After 25 m merge", zorder=3)
_z1 = np.polyfit(Y, R_LOC, 1)
_ax2.plot(Y, np.polyval(_z1, Y), "--", color="#c44e52", alpha=0.5, lw=1)
_z2 = np.polyfit(Y, R_SITE25, 1)
_ax2.plot(Y, np.polyval(_z2, Y), "--", color="#4c72b0", alpha=0.5, lw=1)
_ax2.set_xlabel("Year"); _ax2.set_ylabel("Clark–Evans $R$")
_ax2.set_title("Clustering index over 14 years", fontweight="bold")
_ax2.set_ylim(0.1, 0.6)
_ax2.legend(fontsize=8, loc="upper left")
_ax2.grid(alpha=0.3); _ax2.set_axisbelow(True)

_fa = round(100 * fall(R_LOC))
_fc = round(100 * fall(R_SITE25))

_ax3 = _fig.add_subplot(_gs[2]); _ax3.axis("off")
_ax3.text(0.5, 0.82, f"{_fa}%", fontsize=42, fontweight="bold", color="#c44e52",
          ha="center", va="center", transform=_ax3.transAxes)
_ax3.text(0.5, 0.68, "apparent decline", fontsize=11, color="#c44e52",
          ha="center", va="center", transform=_ax3.transAxes)
_ax3.text(0.5, 0.57, "↓", fontsize=24, ha="center", va="center",
          transform=_ax3.transAxes)
_ax3.text(0.5, 0.42, f"{_fc}%", fontsize=42, fontweight="bold", color="#4c72b0",
          ha="center", va="center", transform=_ax3.transAxes)
_ax3.text(0.5, 0.28, "after correction", fontsize=11, color="#4c72b0",
          ha="center", va="center", transform=_ax3.transAxes)
_ax3.text(0.5, 0.10, "Most change is\nmeasurement,\nnot deployment",
          fontsize=9, fontstyle="italic", ha="center", va="center",
          transform=_ax3.transAxes,
          bbox=dict(boxstyle="round,pad=0.4", facecolor="#ffffcc",
                    edgecolor="#cccc00", alpha=0.8))
_fig.suptitle("Measurement-Dependent Clustering in an Urban Observatory Sensor Network",
              fontsize=13, fontweight="bold", y=0.98)
_fig.savefig(ROOT / "graphical_abstract.png", bbox_inches="tight", dpi=300)
_fig.savefig(ROOT / "graphical_abstract.tiff", bbox_inches="tight", dpi=300)
plt.close(_fig)
print(f"wrote graphical_abstract.png: {_fa}% -> {_fc}%")

wrote graphical_abstract.png: 51% -> 16%


### Dataset card

The image shown for the public dataset. One upload is cropped into a header of
564 x 284 and a thumbnail of 240 x 235, so the card is drawn at the geometric
mean of the two shapes and every mark is kept inside the box both crops
preserve. The cell stops if a crop would cut ink.

In [111]:
_HEAD, _THUMB = 564 / 284, 240 / 235
_DESIGN = (_HEAD * _THUMB) ** 0.5
_KEEP_LO = (1 - _THUMB / _DESIGN) / 2
_RED, _GREY, _PALE, _INK = "#b3261e", "#9e9e9e", "#d2d2d2", "#1a1a1a"
_falls = [(lab, get("unt.tol." + k)) for k, lab in (("fall1", "as published"), ("fall5", "5 m"), ("fall10", "10 m"),
                                                    ("fall15", "15 m"), ("fall25", "25 m"), ("fall50", "50 m"))]
_hl, _mg = int(round(get("cen.hl.fall"))), int(round(get("cen.hl.merged")))
# the headline sentence and the bars must be the same two numbers
assert _hl == round(_falls[0][1]) and _mg == round(_falls[4][1]), (_hl, _mg, _falls)

_fig = plt.figure(figsize=(11.40, 8.00), dpi=300)
_fig.patch.set_facecolor("white")
_x0 = 0.155
_fig.text(_x0, 0.835, "The clustering trend depends\non how a sensor is counted.",
          fontsize=25, fontweight="bold", color=_INK, va="top", linespacing=1.25)
_fig.text(_x0, 0.705, "The Clark\u2013Evans index falls %d%% on published coordinates\n"
          "and %d%% once registrations within 25 m are merged." % (_hl, _mg),
          fontsize=14.5, color=_RED, va="top", linespacing=1.4)
_ax = _fig.add_axes([_x0 + 0.115, 0.215, 0.225, 0.320])
_labs = [lab for lab, _ in _falls][::-1]
_vals = [val for _, val in _falls][::-1]
_cols = [_RED if lab == "as published" else (_INK if lab == "25 m" else _GREY) for lab in _labs]
_ax.barh(range(len(_vals)), _vals, color=_cols, height=0.62)
for _i, (_v, _c) in enumerate(zip(_vals, _cols)):
    _ax.text(_v + 1.5, _i, "%.0f%%" % _v, va="center", fontsize=13,
             fontweight="bold" if _c != _GREY else "normal", color=_c if _c != _GREY else _INK)
_ax.set_yticks(range(len(_labs)))
_ax.set_yticklabels(_labs, fontsize=11.5)
_ax.set_xlim(0, 66)
_ax.set_xticks([])
_ax.tick_params(length=0)
for _s in _ax.spines.values():
    _s.set_visible(False)
_fig.text(_x0, 0.598, "fall of the index, 2013\u20132026,\nby merge radius",
          fontsize=11.5, color="#555555", va="top", linespacing=1.3)
_bx = _fig.add_axes([_x0 + 0.435, 0.215, 0.225, 0.300])
_ns = [int(get("cen.n.ids2013")), int(get("cen.peak.ids")), int(get("cen.n.ids2026"))]
_bx.bar(range(3), _ns, color=[_PALE, _GREY, _PALE], width=0.62)
for _i, _n in enumerate(_ns):
    _bx.text(_i, _n + 0.03 * _ns[1], "{:,}".format(_n), ha="center", fontsize=13, fontweight="bold", color=_INK)
_bx.set_xticks(range(3))
_bx.set_xticklabels(["2013", str(int(get("cen.peak.year"))), "2026"], fontsize=11.5)
_bx.set_yticks([])
_bx.set_ylim(0, _ns[1] * 1.18)
_bx.tick_params(length=0)
for _s in ("top", "left", "right"):
    _bx.spines[_s].set_visible(False)
_bx.spines["bottom"].set_color(_PALE)
_fig.text(_x0 + 0.435, 0.598, "identifiers reporting in the year", fontsize=11.5, color="#555555", va="top")
_fig.text(_x0, 0.168, "Newcastle Urban Observatory, 2013\u20132026: {:,} identifiers, census of every archive file"
          .format(int(get("cen.ids.inrange"))), fontsize=10.5, color="#555555", va="bottom")
_fig.canvas.draw()
_ink = np.asarray(_fig.canvas.buffer_rgba())[..., :3].min(axis=2) < 245
with plt.rc_context({"savefig.bbox": None}):     # the full canvas, so the crop geometry holds
    _fig.savefig(ROOT / "graphical_abstract_card.png", dpi=300, facecolor="white")
plt.close(_fig)
# No ink may fall outside the window that both crops keep.
_cx, _ry = np.where(_ink.any(axis=0))[0], np.where(_ink.any(axis=1))[0]
_box = (_cx[0] / _ink.shape[1], _cx[-1] / _ink.shape[1], _ry[0] / _ink.shape[0], _ry[-1] / _ink.shape[0])
assert abs(_ink.shape[1] / _ink.shape[0] - _DESIGN) < 0.01, _ink.shape
assert _KEEP_LO < min(_box) and max(_box) < 1 - _KEEP_LO, (_box, _KEEP_LO)
print("wrote graphical_abstract_card.png, ink box %s, crops keep %.3f-%.3f" % (np.round(_box, 3), _KEEP_LO, 1 - _KEEP_LO))

wrote graphical_abstract_card.png, ink box [0.155 0.815 0.165 0.832], crops keep 0.141-0.859


## Interactive maps — one per year (2013–2026)

Each map shows the OpenStreetMap street network within the core disc and
the sensor locations for that year, with popups carrying the identifier,
theme and row count. The maps are saved as standalone HTML files in the
figures directory and uploaded alongside the static figures.


In [112]:
import folium
from folium.plugins import MarkerCluster

_sj = json.loads(NET_FILE.read_text())
_proj = _sj["projection"]
_lat0, _lon0, _Re = _proj["lat0"], _proj["lon0"], _proj["earth_radius_m"]

def _dm_to_ll(dx, dy):
    xm, ym = dx / 10.0, dy / 10.0
    lat = _lat0 + math.degrees(ym / _Re)
    lon = _lon0 + math.degrees(xm / (_Re * math.cos(math.radians(_lat0))))
    return lat, lon

_sll = [_dm_to_ll(dx, dy) for dx, dy in _sj["nodes"]]
_epairs = [[_sll[i], _sll[j]] for i, j in _sj["edges"]]
print("Street nodes: %d, edges: %d" % (len(_sll), len(_epairs)))

for _yr in YEARS:
    _ids = REC[_yr]                 # the in-range identifiers the manuscript counts
    _sites = []
    for _sid, _info in _ids.items():
        try:
            _la, _lo = float(_info["lat"]), float(_info["lon"])
            _sites.append((_sid, _la, _lo, _info.get("rows", 0),
                           _info.get("theme", ""), _info.get("variable", "")))
        except (ValueError, KeyError):
            continue
    assert len(_sites) == N_IDS[YEARS.index(_yr)], (_yr, len(_sites))
    _clat = sum(s[1] for s in _sites) / len(_sites) if _sites else _lat0
    _clon = sum(s[2] for s in _sites) / len(_sites) if _sites else _lon0
    _m = folium.Map(location=[_clat, _clon], zoom_start=13,
                    tiles="OpenStreetMap", control_scale=True)
    _fg = folium.FeatureGroup(name="Street network", show=True)
    _CHUNK = 5000
    for _c in range(0, len(_epairs), _CHUNK):
        folium.PolyLine(locations=_epairs[_c:_c+_CHUNK],
                        color="#888888", weight=1, opacity=0.5,
                        smooth_factor=0, no_clip=True).add_to(_fg)
    _fg.add_to(_m)
    _mc = MarkerCluster(name="Sensors (%d)" % len(_sites))
    for _sid, _la, _lo, _rows, _th, _var in _sites:
        folium.CircleMarker(
            location=[_la, _lo], radius=6,
            color="#1f5fa8", fill=True, fill_color="#1f5fa8", fill_opacity=0.7,
            popup="<b>ID %s</b><br>Rows: %s<br>Theme: %s" % (_sid, "{:,}".format(_rows), _th)
        ).add_to(_mc)
    _mc.add_to(_m)
    folium.LayerControl().add_to(_m)
    _m.get_root().html.add_child(folium.Element(
        "<div style='position:fixed;top:10px;left:60px;z-index:9999;"
        "background:white;padding:8px 16px;border-radius:6px;"
        "box-shadow:0 2px 6px rgba(0,0,0,0.3);font:bold 16px sans-serif'>" +
        "Newcastle upon Tyne — %d (%d sensors)</div>" % (_yr, len(_sites))))
    _out = FIGDIR / ("map_%d.html" % _yr)
    _m.save(str(_out))
    print("%d: %d sensors, %.1f MB" % (_yr, len(_sites), _out.stat().st_size / 1e6))

print("Wrote %d interactive maps to %s" % (len(YEARS), FIGDIR))


Street nodes: 157303, edges: 176388
2013: 45 sensors, 15.2 MB
2014: 111 sensors, 15.3 MB
2015: 380 sensors, 15.6 MB
2016: 694 sensors, 15.9 MB
2017: 893 sensors, 16.1 MB
2018: 1095 sensors, 16.4 MB
2019: 947 sensors, 16.2 MB
2020: 975 sensors, 16.2 MB
2021: 946 sensors, 16.2 MB
2022: 794 sensors, 16.0 MB
2023: 762 sensors, 16.0 MB
2024: 875 sensors, 16.1 MB
2025: 786 sensors, 16.0 MB
2026: 756 sensors, 16.0 MB
Wrote 14 interactive maps to /kaggle/working/figures


### Per-year static street maps

One figure per year: merged sites (25 m) on the street network,
coloured by whether they matched a street within 25 m.


In [113]:
from matplotlib.collections import LineCollection as _LC

_ein_seg = np.stack([NET.xy[NET.e[_ein, 0]], NET.xy[NET.e[_ein, 1]]], axis=1) / 1e3
_t = np.linspace(0.0, 2.0 * np.pi, 361)
_disc_x = (EDG_CENTRE[0] + NET_DISC * np.cos(_t)) / 1e3
_disc_y = (EDG_CENTRE[1] + NET_DISC * np.sin(_t)) / 1e3
_xlim = ((EDG_CENTRE[0] - NET_DISC) / 1e3 - 0.2, (EDG_CENTRE[0] + NET_DISC) / 1e3 + 0.2)
_ylim = ((EDG_CENTRE[1] - NET_DISC) / 1e3 - 0.2, (EDG_CENTRE[1] + NET_DISC) / 1e3 + 0.2)
_x0, _y0 = (EDG_CENTRE[0] - NET_DISC) / 1e3 + 0.4, (EDG_CENTRE[1] - NET_DISC) / 1e3 + 0.3
plt.rcParams.update({"font.size": 8.5, "axes.grid": False, "figure.dpi": 300, "savefig.bbox": "tight"})

for _yr in YEARS:
    _S = SITES25[_yr]
    _in_disc = np.hypot(*(_S - EDG_CENTRE).T) <= NET_DISC
    _P = _S[_in_disc]
    _e, _off, _moved = NET.snap(_P)
    _k25 = _moved <= 25.0
    _n_match = int(_k25.sum())
    _n_unmatch = int((~_k25).sum())
    fig, ax = plt.subplots(figsize=(6.4, 6.4))
    ax.add_collection(_LC(_ein_seg, colors="0.62", linewidths=0.25, rasterized=True, zorder=1))
    ax.plot(_disc_x, _disc_y, color="0.25", lw=0.8, ls="--", zorder=2,
            label="core disc, radius %.0f km" % (NET_DISC / 1e3))
    if _n_match > 0:
        ax.scatter(*(_P[_k25] / 1e3).T, s=9, c="#1f5fa8", edgecolors="white",
                   linewidths=0.3, zorder=3, label="site within 25 m of a way (N=%d)" % _n_match)
    if _n_unmatch > 0:
        ax.scatter(*(_P[~_k25] / 1e3).T, s=22, c="#c44e52", marker="x",
                   linewidths=1.0, zorder=4, label="site further than 25 m, left out (N=%d)" % _n_unmatch)
    ax.plot([_x0, _x0 + 1.0], [_y0, _y0], color="k", lw=2.0, zorder=5)
    ax.text(_x0 + 0.5, _y0 + 0.12, "1 km", ha="center", va="bottom", fontsize=8)
    ax.set_aspect("equal")
    ax.set_xlim(*_xlim)
    ax.set_ylim(*_ylim)
    ax.set_xlabel("East of the projection origin (km)")
    ax.set_ylabel("North of the projection origin (km)")
    ax.legend(fontsize=7.5, loc="upper center", bbox_to_anchor=(0.5, -0.09), ncol=1, frameon=False)
    fig.savefig(FIGDIR / ("figS%d_street_%d.pdf" % (_yr - YEARS[0] + 2, _yr)), dpi=300)
    plt.close(fig)
    print("%d: %d sites, %d matched, %d unmatched" % (_yr, len(_P), _n_match, _n_unmatch))

print("Wrote %d per-year street maps to %s" % (len(YEARS), FIGDIR))

2013: 21 sites, 20 matched, 1 unmatched
2014: 38 sites, 35 matched, 3 unmatched
2015: 131 sites, 121 matched, 10 unmatched
2016: 216 sites, 201 matched, 15 unmatched
2017: 315 sites, 292 matched, 23 unmatched
2018: 339 sites, 320 matched, 19 unmatched
2019: 310 sites, 294 matched, 16 unmatched
2020: 321 sites, 303 matched, 18 unmatched
2021: 279 sites, 268 matched, 11 unmatched
2022: 222 sites, 210 matched, 12 unmatched
2023: 171 sites, 156 matched, 15 unmatched
2024: 193 sites, 181 matched, 12 unmatched
2025: 186 sites, 173 matched, 13 unmatched
2026: 174 sites, 161 matched, 13 unmatched
Wrote 14 per-year street maps to /kaggle/working/figures


## Build · Writing values and tables into main.tex and supplement.tex

The values and table bodies recorded above are written into `main.tex` and
`supplement.tex` between `% BEGIN GENERATED:name` and `% END GENERATED:name`
markers. The step fails if a document reads a key that was never set, if a
marker has no table, or if a table has no marker.

### Highlights

The five highlights as the journal takes them: a separate file, one line each,
no line over 85 characters. The two percentages are the registry values the
manuscript prints. Where `main.tex` is present its highlights are checked
against these lines word for word.

In [114]:
HIGHLIGHTS = [
    "First full census of a smart-city sensor population over its lifetime",
    "A %s%% clustering decline reduces to %s%% once co-located devices merge"
    % (VAL["cen.hl.fall"]["text"], VAL["cen.hl.merged"]["text"]),
    "Measurement controls interact, reshaping downstream coverage analyses",
    "Concentration declines once the Ellison-Glaeser index removes count bias",
    "A three-item reporting rule for any city's longitudinal sensor platform",
]
assert 3 <= len(HIGHLIGHTS) <= 5 and all(len(h) <= 85 for h in HIGHLIGHTS), [len(h) for h in HIGHLIGHTS]
(ROOT / "highlights.txt").write_text("Highlights\n\n" + "".join("- %s\n" % h for h in HIGHLIGHTS))
if (ROOT / "main.tex").exists():
    _t = (ROOT / "main.tex").read_text()
    _b = _t[_t.index("\\begin{highlights}"):_t.index("\\end{highlights}")]
    _tex = [re.sub(r"\\V\{([A-Za-z0-9.\-]+)\}", lambda m: VAL[m.group(1)]["text"], l[len("\\item "):])
            .replace("\\%", "%").replace("--", "-") for l in _b.splitlines() if l.startswith("\\item ")]
    assert _tex == HIGHLIGHTS, (_tex, HIGHLIGHTS)
print("wrote highlights.txt:", [len(h) for h in HIGHLIGHTS])

wrote highlights.txt: [69, 69, 69, 72, 71]


In [115]:
def _replace_block(text, name, body):
    a, b = "%% BEGIN GENERATED:%s\n" % name, "%% END GENERATED:%s\n" % name
    assert text.count(a) == 1 and text.count(b) == 1, "marker %s missing or repeated" % name
    i, j = text.index(a) + len(a), text.index(b)
    return text[:i] + body.rstrip("\n") + "\n" + text[j:]


def _values_block(with_macros):
    lines = [r"\newcommand{\V}[1]{\ifcsname v@#1\endcsname\csname v@#1\endcsname"
             r"\else\errmessage{No value for #1}\fi}"]
    for k in sorted(VAL):
        lines.append(r"\expandafter\def\csname v@%s\endcsname{%s}" % (k, VAL[k]["text"]))
    if with_macros:
        for m in sorted(MACROS):
            lines.append(r"\newcommand{\%s}{%s}" % (m, VAL[MACROS[m]]["text"]))
    return "\n".join(lines)


def emit(write=True):
    """Write values and tables into both documents; return a report."""
    report = {"missing_keys": {}, "unused_keys": [], "tables_without_marker": [], "markers_without_table": {}}
    used = set()
    placed = set()
    for name in ("main.tex", "supplement.tex"):
        path = ROOT / name
        if not path.exists():
            print("skip %s (not found at %s)" % (name, path))
            continue
        text = path.read_text()
        keys = set(re.findall(r"\\V\{([A-Za-z0-9.\-]+)\}", text))
        used |= keys
        missing = sorted(keys - set(VAL))
        if missing:
            report["missing_keys"][name] = missing
        text = _replace_block(text, "values", _values_block(with_macros=(name == "main.tex")))
        markers = set(re.findall(r"% BEGIN GENERATED:([A-Za-z0-9.\-]+)\n", text)) - {"values"}
        for m in sorted(markers):
            if m in TABLES:
                text = _replace_block(text, m, TABLES[m])
                placed.add(m)
            else:
                report["markers_without_table"].setdefault(name, []).append(m)
        if write:
            path.write_text(text)
    report["tables_without_marker"] = sorted(set(TABLES) - placed)
    report["unused_keys"] = sorted(set(VAL) - used - set(MACROS.values()))
    _rout = ROOT / "data"
    _rout.mkdir(parents=True, exist_ok=True)
    (_rout / "results.json").write_text(json.dumps(
        {"values": VAL, "tables": TABLES, "macros": MACROS}, indent=1, sort_keys=True))
    return report


REPORT = emit()
print("values %d, tables %d, macros %d" % (len(VAL), len(TABLES), len(MACROS)))
for k, v in REPORT.items():
    print(k, len(v) if not isinstance(v, dict) else {a: len(b) for a, b in v.items()})
assert not REPORT["missing_keys"], REPORT["missing_keys"]
assert not REPORT["markers_without_table"], REPORT["markers_without_table"]
if REPORT["tables_without_marker"]:
    print("WARNING: tables without marker:", REPORT["tables_without_marker"])

skip main.tex (not found at /kaggle/working/main.tex)
skip supplement.tex (not found at /kaggle/working/supplement.tex)
values 1046, tables 64, macros 86
missing_keys {}
unused_keys 960
tables_without_marker 64
markers_without_table {}
